# CASMI 26: The Data Behind Spectra to Structures

The task, data and scoring are described in [CASMI 26: From Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-from-spectra-to-structures).

This notebook shows the data sources behind a public structure-identification pipeline and how each source affects candidate coverage, scoring or evaluation.

| Dataset | What it adds | Section |
|---|---|---|
| CASMI26 Public Models and Candidate Sets | Candidate structures, fingerprint models, ranking models and spectrum simulators | 1 |
| PubChem Popularity Prior | A broad structure catalogue and public documentation for ranking candidates | 2 |
| Natural-Product Panel Benchmark | Cached candidates and scores for labelled natural products | 3 |
| CASMI26 Fold-Safe FPNet | Five fingerprint models, each excluding one identity fold and all panel molecules from training | 3 |
| Natural Product Knowledge Table | Natural-product classes, organisms, literature and public reference spectra | 4 |

Sections 1–4 describe the datasets; Section 5 follows the pipeline and reports public-score comparisons. Section 4 also breaks down which atoms, fragment ions and neutral losses occur in the training spectra, how many fragments are catalogued and which physical rules hold: [Chemistry inside the spectra](#chemistry-inside-the-spectra).


In [ ]:
import os, sys, subprocess, importlib.metadata, time
os.environ['PYTHONHASHSEED'] = '0'
NOTEBOOK_STARTED = time.monotonic()   # record total notebook time, including the retained analysis
from pathlib import Path
if not Path('/kaggle/input').exists() and Path('.runtime/rdkit2026033').exists():
    sys.path.insert(0, str(Path('.runtime/rdkit2026033').resolve()))
try:
    installed = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    installed = ''
if installed.replace('.03.', '.3.') != '2026.3.3':
    tag=f'cp{sys.version_info.major}{sys.version_info.minor}'
    wheels=sorted(Path('/kaggle/input').rglob(f'rdkit-2026.3.3-{tag}-*.whl'))
    assert wheels, f'Attach the RDKit 2026.3.3 offline wheel for {tag}.'
    subprocess.check_call([sys.executable,'-m','pip','install','--no-index','--no-deps','-q',str(wheels[0])], env={**os.environ, 'PYTHONHASHSEED': '0'})
from rdkit import rdBase
assert rdBase.rdkitVersion == '2026.03.3', rdBase.rdkitVersion
print('Official-metric RDKit:', rdBase.rdkitVersion)


In [ ]:
from pathlib import Path
import os
import hashlib
import platform
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, Draw, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

SEED = 20260915
rng = np.random.default_rng(SEED)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (11, 5.5), "axes.titleweight": "bold", "figure.dpi": 120})

# Sharp inline figures, with PNG retained for readers that do not support SVG.
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg', 'png')
plt.rcParams.update({
    'figure.dpi': 160, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.size': 12, 'axes.labelsize': 12, 'axes.titlesize': 13,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'svg.fonttype': 'none', 'figure.facecolor': 'white',
})


In [ ]:
def locate_data_dir():
    candidates = [
        Path("/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"),
        Path("/kaggle/input/enveda-casmi26-molecule-id-mass-spectra"),
        Path("/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra"),
        Path("data/raw"),
        Path("../data/raw"),
    ]
    for path in candidates:
        if all((path / name).exists() for name in ("train.parquet", "test.parquet", "sample_submission.csv")):
            return path.resolve()
    raise FileNotFoundError("Could not find train.parquet, test.parquet, and sample_submission.csv")


DATA = locate_data_dir()
TRAIN_PATH = DATA / "train.parquet"
TEST_PATH = Path(os.environ.get("CASMI_TEST_PATH", DATA / "test.parquet"))
SAMPLE_PATH = Path(os.environ.get("CASMI_SAMPLE_PATH", DATA / "sample_submission.csv"))

In [ ]:
# Locate each mounted dataset by a file it contains (mount paths differ between older and newer notebooks).
import glob, json as _json
from pathlib import Path


def dataset_dir(slug, marker):
    hits = sorted(glob.glob(f'/kaggle/input/**/{slug}/{marker}', recursive=True), key=len)
    return Path(hits[0]).parent if hits else None


V4G = dataset_dir('casmi26-attributed-v4g-reproduction-assets', 'ATTRIBUTION.md')
POP = dataset_dir('casmi26-pubchem-popularity-prior', 'pc_lsid.npy')
PANEL = dataset_dir('casmi26-natural-product-panel-benchmark', 'panel.parquet')
TABLE = dataset_dir('casmi26-natural-product-knowledge-table', 'natural_products.parquet')
FOLD = dataset_dir('casmi26-fold-safe-fpnet', 'MODEL_INPUTS.json')
assert FOLD is not None, 'Attach the updated casmi26-fold-safe-fpnet dataset'

In [ ]:
try:
    rows = []
    for name, path, unit in (('train.parquet', TRAIN_PATH, 'a labelled spectrum'),
                             ('test.parquet', TEST_PATH, 'an unlabelled spectrum')):
        meta = pq.ParquetFile(path).metadata
        rows.append((name, unit, meta.num_rows, meta.num_columns))
    sub = pd.read_csv(SAMPLE_PATH, usecols=['molecule_id'])
    rows.append(('sample_submission.csv', 'a molecule to answer', len(sub), 2))
    display(pd.DataFrame(rows, columns=['file', 'one row is', 'rows', 'columns']).style.hide(axis='index')
            .format({'rows': '{:,}'}))
    n_train = pq.read_table(TRAIN_PATH, columns=['inchikey14']).column('inchikey14').to_pandas().nunique()
    n_test = pq.read_table(TEST_PATH, columns=['molecule_id']).column('molecule_id').to_pandas().nunique()
    print(f'training structures (distinct inchikey14): {n_train:,}; test molecules: {n_test:,}')
    del sub
except Exception as exc:
    print('competition file table unavailable:', type(exc).__name__, exc)

*Competition files: row units, dimensions and counts.*


## 1. CASMI26 Public Models and Candidate Sets

The pipeline combines public components by several authors. [CASMI26 Public Models and Candidate Sets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets) collects unchanged copies of those resources. Each source folder keeps its author and licence in `ATTRIBUTION.md`; the original licences and notices apply to every copied folder.

The main search starts from training structures and COCONUT, an open natural-product collection. A second search adds standardised ChEBI and LIPID MAPS structures and PubChemLite candidates. The searches use different models to combine reference spectra, predicted fingerprints, related structures and fragment matches. Table 1 lists the resources and their roles.


In [ ]:
ROLE = {
    'casmi26-v4b-models': 'main engine: code, four LightGBM boosters, generator',
    'casmi26-fpnet-full1': 'main engine: full1 fingerprint model',
    'casmi26-v2-pool': 'main engine: 710,701-structure candidate pool with fragments and fingerprints',
    'casmi26-v3-models': 'PubChem channel: FPNet A+B models and code; ensemble adds fold-safe FPNet from the fifth dataset',
    'casmi26-fp-models-v2': 'second engine: single and merged fingerprint models',
    'casmi26-ranker-features': 'second engine: ranker training features',
    'casmi26-simulated-ranker-rows': 'second engine: simulated ranker training rows',
    'coconut-casmi26-candidates': 'second engine: COCONUT candidates with fingerprints',
    'chebi-lipidmaps-casmi26': 'earlier ChEBI and LIPID MAPS candidate bank (kept for reproducibility)',
    'casmi-chebi-lmsd-structures': 'second engine: standardised ChEBI and LIPID MAPS candidates (sergiomalv)',
    'casmi26-pubchemlite-cc0-candidate-fingerprints': 'second engine: PubChemLite CC0 candidates (franciscoangulo)',
    'casmi26-iceberg': 'forward model ICEBERG and its inference wrapper',
    'casmi26-glacier': 'forward model GLACIER and its inference wrapper',
    'hengck23-dreams-enveda-casmi26': 'candidates for unfilled ranks (mass-compatible DreaMS structures)',
    'rdkit-2026-3-3-wheel': 'RDKit 2026.03.3 (metric identity)',
    'rdkit-wheels': 'RDKit 2025.03.6 (simulator environment)',
}
try:
    rows = []
    for d in sorted(p for p in V4G.iterdir() if p.is_dir()):
        files = [f for f in d.rglob('*') if f.is_file()]
        rows.append((d.name, ROLE.get(d.name, ''), len(files), round(sum(f.stat().st_size for f in files) / 1e9, 3)))
    flat = [f for f in V4G.iterdir() if f.is_file()]
    rows.append(('(top level)', 'original v4g inference inputs and attribution files', len(flat),
                 round(sum(f.stat().st_size for f in flat) / 1e9, 3)))
    display(pd.DataFrame(rows, columns=['folder', 'role', 'files', 'size (GB)']).style.hide(axis='index')
            .format({'files': '{:,}', 'size (GB)': '{:.3f}'}))
except Exception as exc:
    print('folder table unavailable:', type(exc).__name__)

*Table 1. Folders of the attributed assets with their role in the pipeline, file count and size (GB = 10⁹ bytes).*


## 2. Public documentation in candidate ranking

Several structures can fit the same mass and spectrum. [PubChem Popularity Prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) adds evidence about how well each structure is documented. Its basic score is the sum of ln(1 + PubChem substance records) and ln(1 + linked PubMed articles). A high value means that a structure is familiar in public records; it does not establish that the structure is present in the sample.

The main search uses this score to reorder stored candidates while leaving generated candidates in their slots. The PubChem search also adds terms for a spectral-library depositor, membership in COCONUT, LOTUS or NPAtlas, LOTUS literature references, and PubChem synonyms. The combined documentation score has weight 0.15 beside the standardised fingerprint score.

Row-aligned annotations cover the mass-sorted PubChem catalogue and two supplements. The PubChem channel searches all three stores, extending candidate supply beyond the training data and COCONUT.

Substance, PubMed and patent counts combine matched stereoisomers. Depositor, vendor, MeSH and synonym annotations describe a representative PubChem compound. A source flag records a deposited substance record, not a verified spectrum. An absent flag does not rule out evidence on another stereoisomer. Table 2 illustrates these fields with caffeine and isocaffeine.


In [ ]:
try:
    cols = ['inchikey14', 'compound_name', 'pubchem_cid', 'pop', 'n_depositors', 'n_vendors',
            'src_spectral_library', 'src_metabolite_pathway', 'src_chebi', 'log1p_mesh']
    pool = pd.read_csv(POP / 'pool_popularity.csv', usecols=cols)
    keys = ['RYYVLZVUVIJVGH', 'LPHGQDQBBGAPDZ']        # caffeine, isocaffeine (both C8H10N4O2)
    t = pool[pool.inchikey14.isin(keys)].set_index('inchikey14').loc[keys].reset_index()
    t = t[['compound_name', 'inchikey14', 'pubchem_cid', 'pop', 'n_depositors', 'n_vendors', 'log1p_mesh',
           'src_spectral_library', 'src_metabolite_pathway', 'src_chebi']]
    display(t.rename(columns={'compound_name': 'compound', 'n_depositors': 'depositors', 'n_vendors': 'vendors',
                              'log1p_mesh': 'ln(1 + MeSH terms)', 'src_spectral_library': 'spectral library',
                              'src_metabolite_pathway': 'metabolite/pathway', 'src_chebi': 'ChEBI'})
            .style.hide(axis='index').format({'pop': '{:.2f}', 'ln(1 + MeSH terms)': '{:.2f}'}))
    del pool
except Exception as exc:
    print('popularity table unavailable:', type(exc).__name__, exc)

*Table 2. Caffeine and isocaffeine in the candidate pool, with documentation counts and source annotations. The source flags describe deposited records, not verified spectra.*

The next table reports a cached documentation-only comparison on the 250 panel molecules of Section 3. Candidates have the known answer's formula. Each signal is added separately at weights of 0.10, 0.25 and 0.50. The comparison uses no spectral-model scores.


In [ ]:
try:
    b = pd.read_csv(POP / 'BENCHMARK_SOURCES.csv')
    b = b[b.channel.eq('PubChem exact formula')]
    names = {'src_spectral_library': 'spectral libraries', 'src_metabolite_pathway': 'metabolite / pathway resources',
             'src_chebi': 'ChEBI', 'src_toxicology': 'EPA toxicology', 'src_lipidmaps': 'LIPID MAPS',
             'log1p_n_depositors': 'ln(1 + distinct depositors)', 'log1p_mesh': 'ln(1 + MeSH terms)'}
    b = b[b.signal.isin(names)].assign(signal=lambda d: d.signal.map(names))
    out = pd.DataFrame({'signal': b.signal, 'answers with signal > 0 (%)': 100 * b.answer_nonzero_fraction,
                        'competing candidates with signal > 0 (%)': 100 * b.decoy_nonzero_fraction_query_mean})
    for w in ('0.10', '0.25', '0.50'):
        if f'delta_{w}_mrr25' in b.columns:
            out[f'Δ MRR@25, w = {w}'] = b[f'delta_{w}_mrr25'].values
    print(f'documentation prior alone: MRR@25 = {float(b.baseline_pop_mrr25.iloc[0]):.4f} (250 molecules)')
    fmt = {c: '{:+.4f}' for c in out.columns if c.startswith('Δ')}
    fmt.update({c: '{:.1f}' for c in out.columns if c.endswith('(%)')})
    display(out.style.hide(axis='index').format(fmt))
except Exception as exc:
    print('benchmark table unavailable:', type(exc).__name__, exc)

*Table 3. Additions to the basic documentation score on exact-formula panel candidates: seven source or count signals, their prevalence, and changes in MRR@25 at fixed weights.*

## 3. Natural-product panel benchmark

[Natural-Product Panel Benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) contains cached candidates and scores for 250 labelled natural products from the training library `enveda-np-examples`. Its main-search comparisons cover three situations: this library's reference spectra are removed, every reference spectrum of the molecule is removed, or the correct structure is also removed from the candidate pool. These comparisons separate reference availability from candidate coverage. The dataset contains no spectra.

The panel also provides identity and scaffold folds for the training file. Identity folds keep structures linked by stored or scorer keys together. Scaffold folds additionally group structures by their ring-and-linker framework. These splits support evaluation on unfamiliar molecules.

[CASMI26 Fold-Safe FPNet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet) supplies five fingerprint models for the PubChem channel. Model k excludes identity fold k and all 250 panel molecules from training. The channel adds half of their mean prediction to the FPNet A+B prediction.

That exclusion applies to these five models. The main search's full1 fingerprint model was trained on the panel too; FPNet A+B training rosters are not documented here. The panel favours familiar natural products. Rules selected on it require evaluation on separate molecules. Table 4 describes the panel, training exclusions and cached ranking comparisons.


In [ ]:
try:
    panel = pd.read_parquet(PANEL / 'panel.parquet')
    summary = {'molecules': len(panel)}
    for c in ('n_spectra', 'n_views', 'spectra'):
        if c in panel.columns and pd.api.types.is_numeric_dtype(panel[c]):
            summary['spectra'] = int(panel[c].sum()); break
    for c in ('true_mass', 'neutral_mass', 'mass'):
        if c in panel.columns:
            summary['median neutral mass (Da)'] = float(panel[c].median()); break
    display(pd.DataFrame([summary]).style.hide(axis='index')
            .format({'spectra': '{:,}', 'median neutral mass (Da)': '{:.1f}'}))
    rows = []
    for f in sorted(PANEL.glob('*.parquet')):
        names = pq.read_schema(f).names
        rows.append((f.name, pq.ParquetFile(f).metadata.num_rows, len(names),
                     ', '.join(names[:7]) + (', …' if len(names) > 7 else '')))
    for d in sorted(p for p in PANEL.iterdir() if p.is_dir()):
        rows.append((d.name + '/', None, None, ', '.join(sorted(p.name for p in d.iterdir())[:5])))
    display(pd.DataFrame(rows, columns=['file', 'rows', 'columns', 'contents']).style.hide(axis='index')
            .format({'rows': lambda v: '' if pd.isna(v) else f'{int(v):,}',
                     'columns': lambda v: '' if pd.isna(v) else f'{int(v)}'}))
except Exception as exc:
    print('panel summary unavailable:', type(exc).__name__, exc)

try:
    prior_summary = (pd.read_csv(PANEL / 'prior_rule_summary.csv')
                     .replace({'rule': {'v10 composite prior': 'composite prior (Section 2)'}})
                     .rename(columns={'queries': 'molecules', 'truth_present': 'answer among candidates',
                                      'mrr25': 'MRR@25', 'delta_documentation': 'Δ vs documentation'}))
    print('Prior-only ranking of the PubChem candidates with the answer formula; 250 molecules, ties averaged')
    display(prior_summary.style.hide(axis='index').format(
        {'MRR@25': '{:.4f}', 'Δ vs documentation': '{:+.4f}'}))
except Exception as exc:
    print('cached public prior replay unavailable:', type(exc).__name__, exc)

try:
    audit = _json.loads((FOLD / 'SPLIT_AUDIT.json').read_text())
    contract = _json.loads((FOLD / 'MODEL_INPUTS.json').read_text())
    display(pd.DataFrame([{
        'fold-safe output bits': contract['output_bits'],
        'excluded structure rows': audit['excluded_structure_rows'],
        'panel identities excluded': audit['panel_truth_keys_excluded'],
        'panel identities in its training set': audit['panel_training_overlap'],
    }]).style.hide(axis='index').format('{:,}'))
    replay = pd.DataFrame(_json.loads((FOLD / 'PANEL_REPLAY_PUBLIC.json').read_text())['notebook_summary'])
    replay = replay[replay.protocol.eq('v10_prior') &
                    (replay['fold0 weight'].eq(0) | np.isclose(replay['fold0 weight'], 1 / 3))].copy()
    replay.insert(0, 'fingerprint logits', np.where(replay['fold0 weight'].eq(0), 'A+B', 'A+B + 0.5 fold0'))
    print('PubChem-channel shortlist with the documentation prior; 250 molecules')
    display(replay.drop(columns=['protocol', 'fold0 weight']).style.hide(axis='index').format(
        {c: '{:.4f}' for c in ['MRR@25', 'top-1', 'delta MRR', 'delta MRR 95% low', 'delta MRR 95% high']}))
except Exception as exc:
    print('fold-safe public audit unavailable:', type(exc).__name__, exc)


*Table 4. Panel sizes and files; cached prior-only rankings; the fold-0 training audit; and a cached single-model PubChem-shortlist comparison.*

The cached shortlist comparison uses FPNet A+B with or without fold 0. The current PubChem channel averages five fold-safe models.

## 4. Natural product knowledge table

[Natural Product Knowledge Table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) brings together LOTUS, using the frozen release of 2026-04-13, and NPAtlas. It links natural products to producing organisms, literature, and pathway and class assignments from NPClassifier where available. It also records whether a structure appears in the training data and candidate catalogues.

The table describes gaps in reference coverage. A compound may be known in a natural-product catalogue yet have no training spectrum. Those overlaps describe public knowledge, not the composition of the hidden test. Table 5 summarises the pathways and coverage.

The PubChem search uses LOTUS literature counts from this collection. The main reference library also draws on public MassBank, MoNA and GNPS spectra for structures already represented in training. These records add reference evidence rather than new candidate structures. Per-record licences and source credits remain in `external_reference_spectra/`.


In [ ]:
try:
    nt = pd.read_parquet(TABLE / 'natural_products.parquet',
                         columns=['np_pathway', 'in_casmi_train', 'in_casmi_pool', 'pubmed_articles'])
    candidate_joins = pd.read_parquet(TABLE / 'candidate_joins.parquet',
                                     columns=['ik14', 'tier_row', 'pubchemlite_row'])
    keys = pd.read_parquet(TABLE / 'natural_products.parquet', columns=['ik14']).ik14
    assert candidate_joins.ik14.equals(keys)
    nt = nt.assign(in_pubchem_tier=candidate_joins.tier_row.ge(0).to_numpy(),
                   in_pubchemlite=candidate_joins.pubchemlite_row.ge(0).to_numpy())
    nt = nt.assign(np_pathway=nt.np_pathway.fillna('unclassified'),
                   with_pubmed=nt.pubmed_articles.fillna(0).gt(0),
                   in_casmi_train=nt.in_casmi_train.astype(bool), in_casmi_pool=nt.in_casmi_pool.astype(bool))
    def composition(g):
        return pd.Series({'natural products': len(g),
                          'with training spectrum (%)': 100 * g.in_casmi_train.mean(),
                          'in candidate pool (%)': 100 * g.in_casmi_pool.mean(),
                          'with PubMed article (%)': 100 * g.with_pubmed.mean(),
                          'in PubChem tier (%)': 100 * g.in_pubchem_tier.mean(),
                          'in PubChemLite add-on (%)': 100 * g.in_pubchemlite.mean()})
    t = pd.DataFrame({k: composition(g) for k, g in nt.groupby('np_pathway')}).T
    t = t.sort_values('natural products', ascending=False)
    t.loc['all'] = composition(nt)
    t.index.name = 'pathway'
    fmt = {c: '{:.1f}' for c in t.columns if c.endswith('(%)')}
    fmt['natural products'] = '{:,.0f}'
    display(t.style.format(fmt))
    del nt
    pcl_docs = pd.read_parquet(POP / 'pubchemlite_annotations.parquet',
                               columns=['tier_row', 'lsid', 'lpmid', 'lotus_refs'])
    display(pd.DataFrame([{
        'PubChemLite add-on candidates': len(pcl_docs),
        'matched to pinned PubChem tier (%)': 100 * pcl_docs.tier_row.ge(0).mean(),
        'with documentation prior (%)': 100 * (pcl_docs.lsid + pcl_docs.lpmid).gt(0).mean(),
        'with LOTUS references (%)': 100 * pcl_docs.lotus_refs.gt(0).mean(),
    }]).style.hide(axis='index').format({'PubChemLite add-on candidates': '{:,}', **{
        c: '{:.1f}' for c in ['matched to pinned PubChem tier (%)',
                              'with documentation prior (%)', 'with LOTUS references (%)']}}))
    del pcl_docs
except Exception as exc:
    print('knowledge table unavailable:', type(exc).__name__, exc)

*Table 5. Natural products grouped by NPClassifier pathway, with percentages calculated within each row. `unclassified` has no pathway annotation; `all` covers the whole table. The PubChem columns describe the supplied catalogue and PubChemLite add-on.*

The accompanying smaller table shows how many PubChemLite candidates have documentation counts or LOTUS references. The second search uses their structures and fingerprints; these annotation counts are not part of its ranking.


<a id="chemistry-inside-the-spectra"></a>

### Chemistry inside the spectra

Three checks work at different levels. The precursor mass narrows the formula. Fragment masses test whether that formula can explain the peaks. Fragment structures help to choose between isomers with the same formula. This subsection shows what the molecules are made of, what a real spectrum looks like and which physical rules can become filters or features.

#### Atoms

The training table holds 275,810 structures from 31 to 3,400 Da. Table 6 compares three collections in the competition window of 157–1,159 Da: 273,199 training structures, 690,758 structures of the candidate pool and 105,339,427 structures of the PubChem tier. Each percentage is the share of structures that contain the element at least once, so a column does not sum to 100%.

The molecules are organic: C, H, O and N dominate, followed by S, the halogens and P. Nine training structures have no carbon. The only metal is tin, in a single structure of 553.2 Da; the heaviest element is iodine (atomic mass 126.9 Da). Sodium and potassium never appear inside a training structure, although 178,092 spectra are [M+Na]+, 18,955 are [M+K]+ and 53,452 are [M+NH4]+: in these ions the metal or ammonium only carries the charge.

In [ ]:
try:
    from pathlib import Path
    from IPython.display import display, Markdown, Image
    chem_path = Path(TABLE) / 'spectra_chemistry'
    display(Markdown((chem_path / 'element_table_v25.md').read_text(encoding='utf-8')))
except Exception:
    print('Chemistry table unavailable.')


In [ ]:
try:
    from pathlib import Path
    from IPython.display import display, Markdown, Image
    chem_path = Path(TABLE) / 'spectra_chemistry'
    display(Image(filename=str(chem_path / 'figures/elements.png'), width=1100))
except Exception:
    print('Chemistry figure unavailable.')


*Figure 1. Share of structures containing each of the ten most frequent training elements, 157–1,159 Da. Table 6 also lists the rare elements.*

#### What a spectrum contains — a real example

A tandem mass spectrum (MS/MS) lists the mass-to-charge ratio $m/z$ and the intensity of every product ion. In collision-induced dissociation (CID) the selected ion hits gas molecules, breaks and sometimes rearranges. Only the charged pieces are detected; the neutral pieces that leave are seen only as mass differences.

The example is the training molecule `TZJALUIVHRYQQB` (C33H40O15), a prenylated flavonoid with two sugars, measured as [M+H]+ on an Orbitrap at five collision energies. Its sugar losses make the path from the precursor to the large fragments easy to follow.

In [ ]:
try:
    from pathlib import Path
    from IPython.display import display, Markdown, Image
    chem_path = Path(TABLE) / 'spectra_chemistry'
    display(Image(filename=str(chem_path / 'figures/example_stick_spectra_v25.png'), width=1100))
except Exception:
    print('Chemistry figure unavailable.')


*Figure 2. Measured spectra of the example at three collision energies; intensity relative to the largest peak of each spectrum. Green peaks have a formula that fits the molecule, grey peaks have none; labels give the ion formula and m/z, a star marks peaks with more than one fitting formula. Structures are drawn only where exactly one piece of the molecule fits the peak, and are drawn as neutral molecules. The energy is given in the instrument's normalised units (NCE) and in eV.*

The main path is C33H41O15+ → C27H31O11+ → C21H21O6+ → C17H13O6+. The losses are C6H10O4 (146.0579 Da, a deoxyhexose such as rhamnose), C6H10O5 (162.0528 Da, a hexose such as glucose) and C4H8 (56.0626 Da, part of the prenyl side chain). The masses fit these losses; they do not prove the mechanism or where the charge sits.

In [ ]:
try:
    from pathlib import Path
    from IPython.display import display, Markdown, Image
    chem_path = Path(TABLE) / 'spectra_chemistry'
    display(Image(filename=str(chem_path / 'figures/fragmentation_tree_v25.png'), width=1100))
except Exception:
    print('Chemistry figure unavailable.')


*Figure 3. A possible fragmentation tree of the example. Nodes: ion formula, calculated m/z and the largest relative intensity over the five spectra; structures where only one piece of the molecule fits. Edges: the lost neutral with its mass. C21H21O6+ has a second formula of the same mass.*

The edges show possible losses, not a measured order of reactions.

#### Neutral losses

A neutral loss is the formula difference between two ions of a spectrum, either the precursor and a fragment or two fragments. Table 7 counts in how many spectra each loss appears. Differences between two fragments are possible, not proven, steps, so the percentages show how common a loss is, not how often it happens in one reaction.

In [ ]:
try:
    from pathlib import Path
    from IPython.display import display, Markdown, Image
    chem_path = Path(TABLE) / 'spectra_chemistry'
    display(Markdown((chem_path / 'neutral_loss_table_v25.md').read_text(encoding='utf-8')))
except Exception:
    print('Chemistry table unavailable.')


Most ions in these spectra have an even number of electrons, and such ions usually lose an even-electron neutral (water, CO, a sugar), not a radical:

$$\mathrm{EE}^{+}\rightarrow\mathrm{EE}^{+}+\text{even-electron neutral}.$$

This is the even-electron rule. In the training spectra 25.6% of 38,314 explained peaks need an odd-electron loss such as CH3·; among the 8,897 most reliable peaks (one fitting formula, error ≤ 10 ppm, intensity ≥ 5%) the share is 14.4%. The rule is a strong tendency with [known exceptions](https://pubmed.ncbi.nlm.nih.gov/17985325/), especially for methoxy groups and aromatic rings.

The products are not always the most stable ones. Which bond breaks depends on the energy barrier and on how fast each reaction runs in the short time before detection, so a detected ion need not be the lowest-energy product.

#### Are the fragments in the catalogues?

A spectrum lists fragment ions: pieces of the molecule that kept the charge. In a sample of 2,123 training spectra with known answers, formulas that are part of the known molecule explain 77% of the peaks and 89% of the signal. The rest are noise, isotope peaks, other compounds isolated together with the target and rearranged ions. Two thirds of the explained peaks have only one fitting formula.

The catalogue for the check is the PubChem tier of 105,875,489 structures from 150 to 1,250 Da, so small molecules such as water or methanol are not in it. Of the 273,199 training structures in 157–1,159 Da, 17,356 are not in the tier; they play the role of class-3 molecules here.

Two checks were made. By formula: the neutral formula of a fragment (one hydrogen added or removed) exists in the tier for 46% of the fragments. By structure: the actual piece of the molecule, cut out by breaking up to three bonds and completed with hydrogens, is in the tier for only 14% of the fragments.

Fragment ions are mostly not catalogued molecules. Acylium, iminium and oxonium ions, carbocations and radical ions exist only as charged species in the instrument and often have no stable neutral counterpart. Pieces below 150 Da are outside the tier, and many larger pieces have simply never been registered. A catalogue lookup of fragments therefore cannot serve as a hard filter.

For molecules that are not in PubChem, the large pieces are missing as well: fragments of 300–600 Da are catalogued in 5.2% of cases (29 of 559), against 9.4% for molecules that are in PubChem; fragments of 150–300 Da in 31.0% against 35.5%. Such a molecule cannot be found by looking up its pieces; a generator has to assemble it. Whether a generator can do this from fragments was not tested here.

In [ ]:
try:
    from pathlib import Path
    from IPython.display import display, Markdown, Image
    chem_path = Path(TABLE) / 'spectra_chemistry'
    display(Image(filename=str(chem_path / 'figures/fragment_catalogue_coverage.png'), width=1100))
except Exception:
    print('Chemistry figure unavailable.')


*Figure 4. Share of molecule pieces (up to three broken bonds, completed with hydrogens) found in the PubChem tier, by piece mass, for molecules in and not in the tier. Labels: found / checked. Below 150 Da the tier is empty by construction; the gap at 300–600 Da is real.*

Peaks were matched with a tolerance of max(0.002 Da, 10 ppm), above m/z 40 and above 1% of the largest peak; the precursor peak was excluded. A fitting formula does not fix the structure of an ion, and pieces cut from the intact molecule miss rearranged ions.

#### Physical rules as formulas

Each rule below is written as a formula and tested on the training structures: how many true structures break it, and whether it separates the true structure from candidates of the same mass (192 known molecules with up to eight same-mass candidates each; AUC is the share of pairs where the true structure scores higher, ties count one half, 0.5 means no separation). Table 8 at the end sums up the results.

**1. Mass accuracy.**

$$\Delta_{ppm}=10^6\,\frac{m_{obs}-m_{calc}}{m_{calc}},\qquad |\Delta_{ppm}|\leq\tau .$$

$m_{obs}$ and $m_{calc}$ are the measured and calculated precursor m/z, $\tau$ the tolerance in parts per million (ppm). A 10 ppm window keeps 95.4% of the spectra and a 30 ppm window 96.9% (2,004 of 2,068), so 10 ppm alone would lose one answer in twenty. The public pipeline searches at 10 ppm first and falls back to 30 ppm, and uses the error as a ranker feature.

**2. Adduct and charge.**

$$m/z=\frac{M+\Delta_{adduct}}{|z|}\qquad\Longleftrightarrow\qquad M=|z|\cdot m/z-\Delta_{adduct}.$$

$M$ is the neutral monoisotopic mass (Da), $z$ the charge and $\Delta_{adduct}$ the mass of the charge carrier including the electron: +1.007276 Da for [M+H]+, −1.007276 Da for [M−H]−, +22.989221 Da for [M+Na]+, +18.033826 Da for [M+NH4]+; for a dimer, $2M$ replaces $M$. The equation turns the measured precursor into the neutral mass that every search and generator starts from. It does not separate isomers. The pipeline already converts all adducts this way.

**3. Rings plus double bonds (RDBE).**

$$\mathrm{RDBE}=C+Si-\frac{H+X}{2}+\frac{N+P}{2}+1,\qquad X=F+Cl+Br+I .$$

Element symbols stand for atom counts. For a neutral molecule RDBE equals rings + double bonds + 2 × triple bonds and must be a whole number ≥ 0; for a protonated or deprotonated ion it is a half number ($\mathrm{RDBE}(M)=\mathrm{RDBE}([M+H]^+)+\tfrac12$). One of 275,249 training structures breaks it. All same-mass candidates pass, so it constrains a generator but does not rank. The pipeline only requires RDBE ≥ −1 for fragments.

**4. Nitrogen rule.**

$$A=\sum_e n_e\,a_e,\qquad A\bmod 2=N\bmod 2 .$$

$n_e$ is the count of element $e$, $a_e$ the mass number of its most common isotope, $A$ the nominal mass of the neutral molecule and $N$ the number of nitrogens: a molecule with an odd mass has an odd number of nitrogens. It holds for all 275,281 training structures, after the adduct is removed. Rounding the exact mass instead of summing $a_e$ gives the wrong parity for 3.2% of the structures, so the sum matters. All candidates pass; the rule saves formula enumeration in a generator. The pipeline has no explicit nitrogen-rule filter.

**5. SENIOR valence rules.**

$$V=\sum_e n_e\,v_e,\qquad V\ \text{even},\qquad V\geq 2\,v_{max},\qquad V\geq 2\,(N_{atoms}-1).$$

$v_e$ is the valence of element $e$ (P may be 3 or 5, S 2, 4 or 6), $v_{max}$ the largest valence present and $N_{atoms}$ the number of atoms. The conditions are necessary for the atoms to form one connected molecule. All 275,281 training structures pass, and so do all candidates: useful for a formula generator, not for ranking. The pipeline checks structures with RDKit but has no formula-level SENIOR filter.

**6. Element ratios (Seven Golden Rules).**

$$0.1\leq H/C\leq 6,\qquad n_e/C\leq u_e .$$

$u_e$ is the largest ratio found in known compounds: N 4, O 3, P 2, S 3, F 6, Cl 2, Br 2, Si 1 ([Kind and Fiehn, 2007](https://doi.org/10.1186/1471-2105-8-105)). 38 of 275,272 training structures break a ratio (0.014%), so the ratios fit a soft prior in a generator better than a hard cut. All candidates pass. The pipeline uses composition features but no ratio filter.

**7. Mass defect.**

$$d=M-\operatorname{round}(M).$$

$d$ is the difference between the exact and the nearest whole mass (Da); hydrogen-rich molecules have a positive defect, halogen-rich ones a negative defect. 98% of training structures lie between −0.44 and +0.40 Da. Same-formula candidates share $d$, so it can only help between different formulas. Not used in the pipeline; untested as a feature.

**8. Subformulas and explained intensity.**

$$0\leq n_e(f_k)\leq n_e(p)\ \text{for every element } e,\qquad EI=\frac{\sum_{k\in\mathcal M} I_k}{\sum_k I_k}.$$

A fragment formula $f_k$ cannot contain more atoms of any element than the precursor ion $p$. $I_k$ is the intensity of peak $k$ and $\mathcal M$ the set of peaks with a fitting formula; $EI$ is the explained share of the signal. The true formulas explain 76.7% of the peaks and 89.0% of the signal. Formula-level $EI$ separates the true structure from candidates with a different formula (AUC 0.736) but not from isomers (AUC 0.500). The structure-level score, which checks whether the fragment is an actual piece of the candidate, separates isomers with AUC 0.757 (656 pairs; mean 0.61 for the true structure against 0.29 for the others). The pipeline already uses both.

**9. Neutral losses and fragmentation trees.**

$$\ell_{uv}=f_u-f_v\geq 0,\qquad S_{tree}=\sum_{v}s_{peak}(v)+\sum_{(u,v)}\log P(\ell_{uv})-\lambda\,U .$$

$f_u$ and $f_v$ are the formulas of a parent and a child ion, $\ell_{uv}$ the lost neutral, $P(\ell_{uv})$ how common that loss is (Table 7), $s_{peak}$ the match score of a peak and $U$ the unexplained signal with weight $\lambda$. The best tree explains as much signal as possible with common losses; this is the idea behind [SIRIUS](https://v6.docs.sirius-ms.io/methods-background/), simplified. Neither a loss score nor a tree score was measured here; both are untested ideas for the ranker. The pipeline does not build trees.

**10. Chlorine and bromine isotopes.**

$$\frac{I_{M+2}}{I_M}\approx 0.32\,n_{Cl}+0.97\,n_{Br}.$$

$I_M$ and $I_{M+2}$ are the intensities of an ion's main peak and the peak 2 Da higher, $n_{Cl}$ and $n_{Br}$ the numbers of chlorine and bromine atoms; the factors are the natural ratios ³⁷Cl/³⁵Cl ≈ 0.32 and ⁸¹Br/⁷⁹Br ≈ 0.97 ([NIST](https://physics.nist.gov/cgi-bin/Compositions/stand_alone.pl)). In MS/MS the M+2 companion of a fragment is often missing, most likely because the isolation window passes mainly the main isotope of the precursor: it was found for only 281 of 1,437 halogenated ions, and only 81 of them were within ±50% of the expected ratio. A missing M+2 peak is therefore no evidence against a halogen. Not used in the pipeline.

**11. Collision energy.**

$$\bar m(E)=\frac{\sum_k I_k(E)\,(m/z)_k}{\sum_k I_k(E)} .$$

$\bar m(E)$ is the intensity-weighted mean fragment m/z at collision energy $E$. Higher energy usually shifts the signal to smaller fragments, but not always: in the example the values are 516, 249, 340 and 206 at 13.5, 20.3, 54.2 and 108.4 eV. NCE and eV also differ between instruments. Energy is therefore an input to spectrum models, not a filter; the pipeline passes it to them.

*Table 8. The rules in one view: share of true training structures that break the rule, separation from same-mass candidates and the suggested use.*

| Rule | True structures that fail | Separates same-mass candidates | Suggested use | In the public pipeline |
|---|---|---|---|---|
| Mass window 10 ppm / 30 ppm | 4.6% / 3.1% | no (isomers tie) | search window with fallback, error as feature | yes |
| Nitrogen rule | 0 of 275,281 | no | hard filter in a formula generator | no |
| SENIOR valences | 0 of 275,281 | no | hard filter in a formula generator | no |
| RDBE ≥ 0, whole number | 1 of 275,249 | no | hard filter in a generator | loose bound only |
| Element ratios | 38 of 275,272 | no | soft prior in a generator | no |
| Formula explained intensity | — | different formulas, AUC 0.736 | ranker feature | yes |
| Structure fragment score | — | isomers, AUC 0.757 | ranker feature | yes |
| Loss / tree score | — | not measured | untested idea | no |

#### How to use this

- **Filters for candidate generation.** The nitrogen rule and SENIOR never fail on true structures and RDBE fails once in 275,249, so a formula generator can enforce them, with exceptions for charged molecules and unusual elements. They save enumeration time but do not rank catalogue candidates, which pass them anyway. The mass window must keep the 30 ppm fallback: 10 ppm alone loses 4.6% of the answers.
- **Ranker features.** Only structure-aware fragment evidence separates isomers (AUC 0.757); formula-level evidence helps only between different formulas (AUC 0.736). These are separations on a test set, not measured score gains.
- **Generator constraints for class 3.** Element alphabet and maxima observed in 157–1,159 Da: C72, H122, N17, O36, S6, P6, F36, Cl10, Br6, I6, B1, Si6, As1, Se2, Sn1; ring count ≤ RDBE. Sugar losses of 162.0528 and 146.0579 Da suggest building glycosides from an aglycone plus sugars. Large pieces of class-3 molecules are rarely catalogued (5.2% at 300–600 Da), so a generator cannot rely on catalogued fragments. All of this is untested.
- **Limits.** Mass, parity, valence and ratios cannot tell apart isomers with the same formula, and a formula tree does not fix the structure. This is the job of forward spectrum models such as [ICEBERG and GLACIER](https://github.com/coleygroup/ms-pred), which the public pipeline uses; some spectra still leave several plausible answers.

The bond-cut analysis follows [MetFrag](https://doi.org/10.1186/s13321-016-0115-9) and [MAGMa](https://doi.org/10.1002/rcm.6364); the tree idea follows [SIRIUS](https://doi.org/10.1038/s41592-019-0344-8). The structure fragment score is the public MetFrag-lite implementation by Huseyin Emre Aksoy (Apache-2.0).

## 5. From spectra to a ranked answer

1. **Second search.** Models compare training structures, COCONUT, ChEBI, LIPID MAPS and PubChemLite candidates using reference spectra, fingerprints, related structures and fragment matches. Tree models combine these signals into proposals.
2. **PubChem search.** Candidates fall within 10 parts per million of the estimated neutral mass. FPNet A+B and the five fold-safe models predict structural features. The shortlist combines fingerprint matches with documentation leaders. Detailed scoring returns up to 25 proposals outside the main pool.
3. **Main search.** The pipeline searches the training-and-COCONUT pool near the estimated mass and generates candidates by modifying related reference structures. Fingerprint and ranking models score them. Documentation helps order the stored candidates.
4. **Predicted spectra.** ICEBERG and GLACIER score selected candidates. GLACIER uses protonated-ion spectra, [M+H]+. For molecules without one, it can use proxy spectra made from [M+NH4]+ or [M−H2O+H]+, following [flexonafft's N1 challenger](https://www.kaggle.com/code/flexonafft/casmi26-closing-the-evidence-gap?scriptVersionId=360486019). Proxy GLACIER scores can be combined with ICEBERG scores to distinguish candidates with the same formula.
5. **List combination.** PubChem proposals enter at six fixed positions, including rank 24, when the main list lacks a strong library match. Fingerprint and documentation thresholds, checked on the leading proposal's own scores, can promote it to rank 1. The main and second-search lists then support candidates according to their ranks, with less weight on the second search. Each scored identity receives one contribution per list.
6. **Final order.** Candidates with the same formula can be reordered using GLACIER, joint ICEBERG and proxy GLACIER, ICEBERG alone, or a MetFrag-lite fragment score, depending on available scores. Vacant ranks receive further distinct candidates. Finally, the second search's leading structure is placed at rank 2 unless it already leads.

### Key lessons

Removing PubChem-only candidates, with all PubChem features retained, reduced the public score from 0.464 to 0.409. Many answers in the public test are therefore reachable only through candidates outside the training structures and COCONUT.

Expanding the number of analogue parents from 6 to 20 changed the public score from 0.464 to 0.456. Replacing one fold-safe model with five left the score unchanged at 0.464.

One public molecule corresponds to about 0.008 MRR@25, so changes of that size are within noise. These comparisons provide limited evidence about ranking changes.

Public documentation favours familiar compounds. Training exclusions, candidate coverage and ranking measure different parts of the method: an excluded molecule can still be missed when its structure is absent from the proposed candidates.


In [ ]:
# EDA has finished: release scalar tables and RDKit objects before constructing the search index.
import gc
for _name in ("train", "unique_structures", "descriptor_sample", "desc", "fp_sample", "records",
              "preview_pool", "representatives", "mols", "X", "projection", "library_pairs"):
    globals().pop(_name, None)
plt.close("all")
gc.collect()


In [ ]:
'''CASMI26: Metric-Identity Fusion — Aggressive Experiment
One vote per official metric identity per engine; compact unique ranks before the 40-candidate fusion shortlist. Preserve the existing PubChem gate and same-formula GLACIER logic. This tests duplicate-vote bias, not a coefficient search. It can change top-1 and reduce the score.
Run the complete repaired @obstacledeveloper adaptation first, then this independent overlay. submission_reference.csv is the completed baseline from the same run; experiment_audit.csv/json record changes and file hashes. This is not stacked with the other experiment. Same 17 source inputs and GPU environment; no additional dataset required. No new public score has been measured. A visible smoke run is not validation.'''


In [ ]:
'''Credits & Attribution
@obstacledeveloper — direct source: CASMI26 V1.1 Metric Audit and Safe Tail, version 1, scriptVersionId 356393001. Source: https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail?scriptVersionId=356393001
The source page displays public score 0.433; this adaptation has not reproduced that result.
@bobthebot369 — inherited Sovereign Zenith ensemble and confidence-promotion ideas: https://www.kaggle.com/code/bobthebot369/enveda-casmi-2026-v18-sovereign-zenith/notebook
@ahmedberatozer — engine, ranker, pool, PubChem, full FPNet and simulator packages: https://www.kaggle.com/ahmedberatozer
@prvsiyan — analog propagation, spectral matching and fingerprint models: https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline
@dmitriigluzdov — PubChem popularity prior: https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior
@hengck23 — inherited hengck23-dreams-enveda-casmi26 structural candidate resource: https://www.kaggle.com/hengck23
Official metric implementation, version 14: https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank
ICEBERG/GLACIER research tooling: https://github.com/coleygroup/ms-pred
Competition: https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra
RDKit official metric wheel: https://www.kaggle.com/datasets/metric/rdkit-2026-3-3-wheel
ChEBI/LIPID MAPS and COCONUT resources remain credited in inherited code and input metadata.
The public notebook declares Apache-2.0; upstream datasets/components retain their own licences. Substantial code, including embedded modules, is reused verbatim. Our adaptation changes presentation and adds provenance, not authorship of the engines. Upstream comments are retained as provenance, not independently established component-performance claims.'''


In [ ]:
'''Experiment provenance and reasoning
Direct starting point: @obstacledeveloper, linked in Credits & Attribution below. The upstream engine/fusion/simulator code is substantially reused, not claimed as original. Our contribution here is the separately audited overlay described above. Previous analog expansion, top-1 shielding, adduct-specific merging and single-pass variants did not improve our confirmed 0.420 result. These experiments test structural ranking changes instead of repeating those parameter changes. Metric identity uses the inherited official canonicalization and heavy-atom composition helpers. GLACIER tooling: https://github.com/coleygroup/ms-pred ; research: https://arxiv.org/abs/2606.29161 . Global cross-formula ranking is our unvalidated hypothesis, not a claim made by that paper. Public leaderboard feedback does not establish private-set generalization.'''


In [ ]:
'''# CASMI26 V1.1 — metric audit and safe tail

Starting point: this account's V1 submission, public MRR **0.425** on 2026-10-08.
This is a conservative candidate-coverage experiment, not a claimed leaderboard improvement.

Changes: pin official RDKit 2026.03.3; preserve the V1 inference/ranking pipeline;
deduplicate only after ranking using the official metric identity; append ranked overflow
and mass-compatible DreaMS candidates into vacant slots; assert every original matchable
identity has equal or better rank. The old DreaMS branch was bypassed by `NameError: Path`.

No hidden-test labels, manual labels, or network access are used. The visible test overlaps
training data, so its score is an operational check and is **not** a validation estimate.

Source attribution: copied V1 ensemble from BobtheBot369 / Ahmed Berat Özer / prvsiyan
and the other authors credited in its code. Scoring reference: Kaggle Competition Metrics,
`https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank`, version 14, Apache-2.0.
'''


In [ ]:
'''## 1. Setup

Locates every dataset, installs the competition's RDKit wheel, sets the popularity-prior parameters for the PubChem channel (`POP_LAM = 0.25`, `POP_UNION = 200`) and checks the v4b model files against their manifest.
'''


In [ ]:
'''CASMI26 v4n = v4i + ICE_LAM = GL_LAM = 1.0 (v4l, LB 0.380) + engine FP bank fpnet_full1 (v4m, LB 0.378)'''
'''CASMI26 v4i = v4h + PubChem-only channel pass-1 depth N1 1000 -> 5000 (recall; gate/slots unchanged)'''
'''CASMI26 v4h = v4g + GLACIER third re-scoring term: z(ranker) + 0.5 z(ICEBERG) + 0.5 z(GLACIER) (casmi26-glacier)'''
'''casmi26 v28 = v17 + pool popularity re-rank of the base list (mu 0.15, mode A); v17 = Ahmed v4g + popularity prior in the PubChem list (see make_v17.py)'''
'''CASMI26 v4f = v4b (FPNet A engine bank, LB 0.354) + ICEBERG post-ranker isomer re-scoring = the v3.6 model (v1 engine + FPNet A + moderate generation + gated PubChem channel, LB 0.358)'''
'''+ fe_v4 feature families computed on the SAME v1 engine run (derivation evidence / class-3 priors, fragmentation 2.0,'''
'''analog-structure relations, DreamsFP views) + a ranker retrained on the v1-engine simulation sim1x with them.'''
'''Built by work/analysis/fe_v4/build_v4v1.py -- edit there.'''
import os, sys, glob, re, subprocess, time, json, hashlib, pickle
from pathlib import Path
T0 = time.time()
LIB_TAU, REL_TH = 0.9, 600.0
TOPN, ICE_LAM, ICE_BUDGET, ICE_PC = 60, 1.0, 5400, True
SLOTS_AGG, SLOTS_GENTLE = [2, 4, 6, 8, 10], [4, 8, 12, 16, 20]
PC_SIXTH_SLOT = 24
SLOTS_AGG = SLOTS_AGG + [PC_SIXTH_SLOT]
SLOTS_GENTLE = SLOTS_GENTLE + [PC_SIXTH_SLOT]
print("PUBCHEM_SIXTH_SLOT active=" + str(PC_SIXTH_SLOT) + " aggressive=" + str(SLOTS_AGG) + " gentle=" + str(SLOTS_GENTLE), flush=True)
os.makedirs('/kaggle/working/numba_cache', exist_ok=True)
os.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'
def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]
def sha256(path, n=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(n), b''):
            h.update(b)
    return h.hexdigest()
FOLD_CHECKPOINT_SHA256 = {'fpnet_fold0.pt': 'ddf6eaaa8e541a16e3f0357ed30f0346e529054f75d5619aedb6024dcb1bd7f0', 'fpnet_fold1.pt': '1739d5683fa94fd73fe3b37db98ac364cb5dfdc22ad5bea9789be9c48d7cadc4', 'fpnet_fold2.pt': '257062cb6ece333226027de6e313792e7b364517adf42a05e0df950b8e27bbe2', 'fpnet_fold3.pt': 'ac61125ef34415d857d58a115e017e0af2aede044e3a5d524029dd1364cbd58c', 'fpnet_fold4.pt': 'ad0b2a43343287f7deb4334c4ad766e8aa236d7f0bbeb91e3702910ee49aabe7'}
FOLD_PATHS = [str(FOLD / f'fpnet_fold{k}.pt') for k in range(5)]
FOLD0_PATH = FOLD_PATHS[0]
for _fold_path in FOLD_PATHS:
    assert Path(_fold_path).is_file(), _fold_path
    assert sha256(_fold_path) == FOLD_CHECKPOINT_SHA256[Path(_fold_path).name], ('Untested fold-safe checkpoint', _fold_path)
os.environ['CASMI_FPNET_FOLD0'] = FOLD0_PATH
os.environ['CASMI_FPNET_FOLDS'] = json.dumps(FOLD_PATHS)
print('PubChem ensemble rule: z_AB + 0.5 * mean(z_fold0..z_fold4)', flush=True)
POP_LAM, POP_UNION = 0.15, 200            # v17: popularity prior for the PubChem-only list
os.environ['CASMI_POP_DIR'] = str(POP); os.environ['CASMI_POP_LAM'] = str(POP_LAM); os.environ['CASMI_POP_UNION'] = str(POP_UNION)
print('v17 popularity prior', POP_LAM, POP_UNION, os.environ['CASMI_POP_DIR'])
SRC_BONUS, SRC_BITS = 0.5, 1   # documentation prior + bonus for depositor-source groups (pc_src_mask.npy, dataset v5)
os.environ['CASMI_SRC_BONUS'] = str(SRC_BONUS); os.environ['CASMI_SRC_BITS'] = str(SRC_BITS)
assert os.path.exists(os.path.join(os.environ['CASMI_POP_DIR'], 'pc_src_mask.npy')), 'pc_src_mask.npy missing'
print('depositor-source bonus', SRC_BONUS, 'bits', SRC_BITS, flush=True)
NP_BONUS = 1.0   # natural-product bonus: COCONUT, LOTUS or NPAtlas membership (pc_np.npy, popularity prior)
os.environ['CASMI_NP_BONUS'] = str(NP_BONUS)
assert os.path.exists(os.path.join(os.environ['CASMI_POP_DIR'], 'pc_np.npy')), 'pc_np.npy missing'
print('natural-product bonus', NP_BONUS, flush=True)
LOTUS_W = 0.5   # literature prior: ln(1 + LOTUS references) from the natural-product knowledge table
os.environ['CASMI_LOTUS_W'] = str(LOTUS_W)
assert (POP / 'aligned_public_priors.py').is_file(), 'aligned_public_priors.py missing'
assert sha256(POP / 'aligned_public_priors.py') == '89873c7312c16be2180d44bb9eb3c4e9ce19c821e74ab92bb41666ea0c0f9248', 'Untested aligned prior helper'
assert (POP / 'pc_lotus_refs.npy').is_file(), 'pc_lotus_refs.npy missing'
print('LOTUS literature prior', LOTUS_W, flush=True)
assert (POP / 'pc_lsynonyms.npy').is_file(), 'pc_lsynonyms.npy missing (popularity dataset of 19.10 or later)'
print('PubChem synonym prior 0.5 (pc_lsynonyms.npy)', flush=True)
for _n in ('delta_pc_mass.npy', 'growth_pc_mass.npy', 'delta_pc_lsynonyms.npy', 'growth_pc_lsynonyms.npy'):
    assert (POP / _n).is_file(), _n + ' missing (popularity dataset of 20.10 or later)'
import numpy as _np
print('PubChem supplements: legacy delta', len(_np.load(POP / 'delta_pc_mass.npy', mmap_mode='r')), 'rows + growth',
      len(_np.load(POP / 'growth_pc_mass.npy', mmap_mode='r')), 'rows', flush=True)
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl = [w for w in glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) if tag in w and Path(w).name.startswith('rdkit-2026.3.3-')]
assert whl, 'Official RDKit 2026.3.3 wheel is missing'
print('rdkit wheel:', whl)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
r.check_returncode()
import rdkit; print('rdkit', rdkit.__version__)
assert rdkit.__version__ == '2026.03.3'
V4CODE = os.path.dirname(os.path.dirname(find('casmi26-v4b-models/**/casmi/engine.py')))
V4MOD = os.path.dirname(find('casmi26-v4b-models/**/MANIFEST.json'))
V3CODE = os.path.dirname(os.path.dirname(find('casmi26-v3-models/**/casmi/engine.py')))
V3MOD = os.path.dirname(find('casmi26-v3-models/**/ranker_0.pkl'))
POOL_DIR = os.path.dirname(find('pool_meta.parquet'))
PC_DIR = os.path.dirname(find('pc_smiles.npy'))
COMP = os.path.dirname(find('test.parquet'))
MAN = json.load(open(os.path.join(V4MOD, 'MANIFEST.json')))
bad = []
for rel, h in sorted(MAN['files'].items()):
    p = os.path.join(V4CODE, rel[len('code/'):]) if rel.startswith('code/') else os.path.join(V4MOD, rel)
    if not os.path.exists(p) or sha256(p) != h:
        bad.append(rel)
assert not bad, f'dataset files differ from MANIFEST.json: {bad}'
print('V4CODE', V4CODE, '\nV4MOD', V4MOD, '\nV3CODE', V3CODE, '\nPOOL', POOL_DIR, '\nPC', PC_DIR, '\nCOMP', COMP)
print(f"{len(MAN['files'])} files match MANIFEST.json; families {MAN['families']}", f'{time.time()-T0:.0f}s')

'''Compatibility repair: select the simulator RDKit 2025.3.6 wheel by interpreter and platform tags, including bundled .whl.ice files. Fail before inference when no compatible build exists; preserve the main RDKit 2026.03.3 environment.'''

def simulator_rdkit_wheel(pkg):
    from pip._vendor.packaging.tags import sys_tags
    from pip._vendor.packaging.utils import parse_wheel_filename
    compatible = set(sys_tags())
    bundled = sorted(glob.glob(os.path.join(pkg, 'wheels', 'rdkit-2025.3.6-*.whl*')))
    external = sorted(glob.glob('/kaggle/input/**/rdkit-2025.3.6-*.whl', recursive=True))
    for path in bundled + external:
        name = os.path.basename(path).removesuffix('.ice')
        _, version, _, tags = parse_wheel_filename(name)
        if str(version) == '2025.3.6' and compatible.intersection(tags):
            return path
    raise FileNotFoundError(
        f'RDKit 2025.3.6 wheel compatible with Python {sys.version_info.major}.{sys.version_info.minor} '
        f'and this platform is required for ICEBERG/GLACIER; candidates: {bundled + external}')

_simulator_pkg = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))
print('Simulator RDKit wheel:', simulator_rdkit_wheel(_simulator_pkg))


In [ ]:
# v42 determinism policy, adapted from its restored tree to E006's original mounts.
# E006 cell6 has already verified the original v4b manifest and all its hashes.
def _v43_patch_engine(source_root, work_root):
    import hashlib
    from pathlib import Path
    source_root = Path(source_root).resolve(strict=True)
    work_root = Path(work_root)
    original = source_root / 'casmi' / 'engine.py'
    raw = original.read_bytes()
    old = 'ad = max(set(adducts), key=adducts.count)'
    new = 'ad = max(adducts, key=adducts.count)'
    text = raw.decode('utf-8')
    assert text.count(old) == 1, 'Expected exactly one source main-engine adduct tie'
    patched = text.replace(old, new, 1).encode('utf-8')
    assert source_root != work_root.resolve() and source_root not in work_root.resolve().parents
    work_root.mkdir(parents=True, exist_ok=True)
    # Preserve the original relative source tree. Only engine.py is copied/changed.
    for source in sorted(source_root.rglob('*')):
        if '__pycache__' in source.parts or source.suffix == '.pyc':
            continue
        dest = work_root / source.relative_to(source_root)
        if source.is_dir():
            dest.mkdir(parents=True, exist_ok=True)
        elif source == original:
            assert not dest.is_symlink(), 'Patched engine must be a work-tree file'
            if dest.exists():
                assert dest.read_bytes() == patched, 'Existing work-tree patch differs'
            else:
                dest.write_bytes(patched)
        else:
            if dest.is_symlink():
                assert dest.resolve(strict=True) == source.resolve(strict=True)
            else:
                assert not dest.exists(), 'Existing work-tree source differs'
                dest.symlink_to(source)
    assert original.read_bytes() == raw, 'Mounted source changed'
    assert (work_root / 'casmi/engine.py').read_bytes() == patched
    return dict(applied=True, original_source_root=str(source_root), work_source_root=str(work_root),
        original_sha256=hashlib.sha256(raw).hexdigest(), patched_sha256=hashlib.sha256(patched).hexdigest(),
        original_line=old, patched_line=new, subprocess_pythonhashseed='0',
        torch_seed=0, cuda_seed=0, cudnn_deterministic=True, cudnn_benchmark=False)

V43_DETERMINISM = _v43_patch_engine(V4CODE, '/kaggle/working/v43_v4code')
V4CODE = V43_DETERMINISM['work_source_root']
os.environ['PYTHONHASHSEED'] = '0'


In [ ]:
"""Read-only E004 observer, before the source smoke switch."""


def _e004_preflight(scope):
    import json
    import os
    import platform
    import subprocess
    import pandas as pd
    import rdkit

    slug = 'enveda-CASMI26-molecule-id-mass-spectra'
    comp = os.path.realpath(scope['COMP'])
    assert comp in {f'/kaggle/input/{slug}', f'/kaggle/input/competitions/{slug}'}
    assert platform.python_version_tuple()[:2] == ('3', '12')
    assert rdkit.__version__ == '2026.03.3'
    # Do not initialize a parent CUDA context before the source's Engine2 subprocess.
    gpu = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
    assert 'T4' in gpu, gpu
    ids = sorted(set(pd.read_parquet(os.path.join(comp, 'test.parquet'),
                                    columns=['molecule_id']).molecule_id.astype(str)))
    sample = pd.read_csv(os.path.join(comp, 'sample_submission.csv'), dtype=str)
    assert ids and set(sample.molecule_id) == set(ids)
    result = dict(runtime_input_path=comp, input_ids=ids, python=platform.python_version(),
                  rdkit=rdkit.__version__, gpu=gpu)
    print('E004_PREFLIGHT', json.dumps({k: v for k, v in result.items() if k != 'input_ids'}))
    return result


E004_PREFLIGHT = _e004_preflight(globals())
del _e004_preflight


In [ ]:
'''## 2. Commit switch

When the visible placeholder test is detected, the notebook runs only 12 molecules with a short ICEBERG budget, so saving a version is quick. The scoring rerun on the hidden test runs everything. Nothing else depends on this switch.
'''


In [ ]:
'''SMOKE switch: commits (dummy test) run 12 molecules; the scoring rerun (hidden test) runs everything.'''
import pandas as pd, hashlib
_te = pd.read_parquet(os.path.join(COMP, 'test.parquet'), columns=['molecule_id', 'spectrum_id'])
_sig = hashlib.md5(','.join(sorted(_te.spectrum_id.astype(str))).encode()).hexdigest()
IS_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN')) or _sig != '654e030e4173780e945252b4a5adf70b'
SMOKE_N = 12
if not IS_RERUN:
    SM = '/kaggle/working/smoke_comp'; os.makedirs(SM, exist_ok=True)
    _full = pd.read_parquet(os.path.join(COMP, 'test.parquet'))
    _keep = sorted(_full.molecule_id.unique())[:SMOKE_N]
    _full[_full.molecule_id.isin(_keep)].to_parquet(os.path.join(SM, 'test.parquet'))
    for _f in ['train.parquet', 'sample_submission.csv']:
        if not os.path.exists(os.path.join(SM, _f)):
            os.symlink(os.path.join(COMP, _f), os.path.join(SM, _f))
    COMP = SM
    ICE_BUDGET = 300
print('RERUN', IS_RERUN, '| sig', _sig, '| COMP', COMP, '| ICE_BUDGET', ICE_BUDGET)


In [ ]:
'''## 3. Engine 2: two-ranker engine (BIO + AFIX)

This engine runs first in its own process, while GPU memory is still empty. It writes a top-40 list per molecule with tautomer-canonical InChIKey-14 keys.

- **Candidates** come from COCONUT, ChEBI/LIPID MAPS (BIO) and the training structures.
- **Evidence** comes from library matching (including adduct-shifted matches), analog propagation, prvsiyan's fingerprint models and MetFrag-lite fragment matching.
- **Ranking** blends two gradient-boosted rankers.
'''


In [ ]:
'''Our engine (two-ranker + BIO + AFIX) in a subprocess, then weighted reciprocal-rank fusion with the v4g lists.'''
ENG_DIR = '/kaggle/working/eng'; os.makedirs(ENG_DIR, exist_ok=True)
ENG_FILES = {'pv.py': '"""prvsiyan analog-propagation pipeline components (public notebook, version of 2026-09-16), copied\nverbatim where possible so local simulations and the Kaggle notebook share one source of truth.\nAdditions are marked `# ADDED`.\n"""\nimport math\nimport numpy as np\nfrom numba import njit, prange\n\n\nclass CFG:\n    PPM_WIN = 10.0\n    PPM_FALLBACK = 30.0\n    INT_FLOOR = 0.002\n    MAX_PEAKS = 256\n    MZ_TOL = 0.01\n    INT_POWER = 1.0\n    ENT_WEIGHT = True\n    ANALOG_WIN = 200.0\n    N_ANALOG = 80\n    SIM_POWER = 3.0\n    W1_PRIORS = (0.30, 0.60)\n    SEEDS = (0, 1, 2, 3, 4, 5, 6, 7)  # 8-seed bag (ghazarosbarseghyan91 v14 seeds)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n    TOPN = 25\n\n\n# ----------------------------------------------------------------------------------------------\n# similarity kernels\n# ----------------------------------------------------------------------------------------------\n@njit(cache=True, fastmath=True)\ndef _clean(mz, it, floor, topk, power, ent_weight):\n    n = len(mz)\n    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    mx = 0.0\n    for i in range(n):\n        if it[i] > mx: mx = it[i]\n    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    thr = floor * mx; c = 0\n    for i in range(n):\n        if it[i] >= thr: c += 1\n    idx = np.empty(c, np.int64); j = 0\n    for i in range(n):\n        if it[i] >= thr: idx[j] = i; j += 1\n    if c > topk:\n        v = np.empty(c, np.float32)\n        for i in range(c): v[i] = it[idx[i]]\n        o = np.argsort(v)[c - topk:]\n        k2 = np.empty(topk, np.int64)\n        for i in range(topk): k2[i] = idx[o[i]]\n        k2.sort(); idx = k2; c = topk\n    om = np.empty(c, np.float32); oi = np.empty(c, np.float32)\n    s = 0.0\n    for i in range(c):\n        om[i] = mz[idx[i]]; v = it[idx[i]] ** power; oi[i] = v; s += v\n    if s > 0:\n        for i in range(c): oi[i] /= s\n    if ent_weight:\n        S = 0.0\n        for i in range(c):\n            if oi[i] > 0: S -= oi[i] * np.log(oi[i])\n        if S < 3.0:\n            w = 0.25 + 0.25 * S; s2 = 0.0\n            for i in range(c): oi[i] = oi[i] ** w; s2 += oi[i]\n            if s2 > 0:\n                for i in range(c): oi[i] /= s2\n    return om, oi\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim(qmz, qp, cmz, cp, tol):\n    i = 0; j = 0; n = len(qmz); m = len(cmz)\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])\n    SAB = 0.0; tot = 0.0\n    buf = np.empty(n + m, np.float64); b = 0\n    while i < n and j < m:\n        d = qmz[i] - cmz[j]\n        if d < -tol: buf[b] = qp[i]; i += 1; b += 1\n        elif d > tol: buf[b] = cp[j]; j += 1; b += 1\n        else: buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1\n    while i < n: buf[b] = qp[i]; i += 1; b += 1\n    while j < m: buf[b] = cp[j]; j += 1; b += 1\n    for x in range(b): tot += buf[x]\n    if tot <= 0: return 0.0\n    for x in range(b):\n        v = buf[x] / tot\n        if v > 0: SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\n    return out\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):\n    a = entropy_sim(qmz, qp, cmz, cp, tol)\n    if shift > -0.001 and shift < 0.001: return a\n    sm = np.empty(len(cmz), np.float32)\n    for i in range(len(cmz)): sm[i] = cmz[i] + shift\n    b = entropy_sim(qmz, qp, sm, cp, tol)\n    return a if a > b else b\n\n\n# ADDED: search over a pre-cleaned store (reps cleaned once instead of on every call)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_pre(qmz, qp, lo, hi, roff, rmz, rit, tol, shift):\n    out = np.zeros(hi - lo, np.float32)\n    for k in prange(hi - lo):\n        a = roff[lo + k]; b = roff[lo + k + 1]\n        if b <= a: continue\n        out[k] = entropy_sim_shift(qmz, qp, rmz[a:b], rit[a:b], tol, shift[k])\n    return out\n\n\n# ADDED: library search where each reference is shifted by (query precursor - reference precursor):\n# the same molecule measured as a different adduct keeps its neutral losses (modified-cosine idea)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_rows(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])\n    return out\n\n\n# ADDED: clean a list of spectra into a flat store\n@njit(cache=True)\ndef clean_store(rows, off, allmz, allin, floor, topk, power, ent_weight):\n    n = len(rows)\n    lens = np.zeros(n + 1, np.int64)\n    tmp_m = []\n    tmp_i = []\n    for k in range(n):\n        r = rows[k]\n        cm, cp = _clean(allmz[off[r]:off[r + 1]], allin[off[r]:off[r + 1]], floor, topk, power, ent_weight)\n        tmp_m.append(cm); tmp_i.append(cp)\n        lens[k + 1] = lens[k] + len(cm)\n    om = np.empty(lens[n], np.float32); oi = np.empty(lens[n], np.float32)\n    for k in range(n):\n        om[lens[k]:lens[k + 1]] = tmp_m[k]; oi[lens[k]:lens[k + 1]] = tmp_i[k]\n    return lens, om, oi\n\n\n# ----------------------------------------------------------------------------------------------\n# adducts\n# ----------------------------------------------------------------------------------------------\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\nE = 0.00054857990; PROTON = MASS[\'H\'] - E; H2O = 2 * MASS[\'H\'] + MASS[\'O\']\nNH4 = MASS[\'N\'] + 4 * MASS[\'H\']; FORMATE = MASS[\'C\'] + 2 * MASS[\'H\'] + 2 * MASS[\'O\']\nACETATE = 2 * MASS[\'C\'] + 4 * MASS[\'H\'] + 2 * MASS[\'O\']\nADDUCTS = {\n    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS[\'Na\'] - E),\n    "[M+K]+": (1, 1, MASS[\'K\'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),\n    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),\n    "[M+CH3OH+H]+": (1, 1, PROTON + MASS[\'C\'] + 4 * MASS[\'H\'] + MASS[\'O\']),\n    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS[\'C\'] + 3 * MASS[\'H\'] + MASS[\'N\']),\n    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),\n    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS[\'Cl\'] + E), "[M]-": (1, 1, E),\n    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS[\'Na\'] - 2 * PROTON),\n    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS[\'Na\'] - E), "[2M+NH4]+": (2, 1, NH4 - E),\n    "[2M+K]+": (2, 1, MASS[\'K\'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),\n    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS[\'Na\'] - 2 * PROTON),\n    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),\n}\n\n\ndef neutral_mass(mz, adduct):\n    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)\n    for a, (n, z, d) in ADDUCTS.items():\n        m = (ad == a)\n        if m.any(): out[m] = (mz[m] * z - d) / n\n    return out\n\n\n# ----------------------------------------------------------------------------------------------\n# ranking features (31)\n# ----------------------------------------------------------------------------------------------\nN_ANALOG = 80\nP_SIM = 3.0\nN_FEAT = 31\n\n\ndef _rank_norm(x):\n    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)\n\n\ndef _z(x):\n    s = x.std()\n    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)\n\n\ndef rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):\n    nc = cand_fp.shape[0]\n    cf = cand_fp.astype(np.float32); cs = cf.sum(1)\n    lv = np.asarray(cand_lib, np.float32)\n    lvmax = float(lv.max()) if nc else 0.0\n    if analog_fp is not None and len(analog_sim):\n        af = analog_fp.astype(np.float32); asum = af.sum(1)\n        inter = cf @ af.T\n        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)\n        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)\n        ap = (tan * (w ** P_SIM)[None, :]).max(1)\n        a1 = (tan * w[None, :]).max(1)\n        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])\n        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)\n    else:\n        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0\n    apmax = float(ap.max()) if nc else 0.0\n    if model_logits is not None:\n        raw = cf @ np.asarray(model_logits, np.float32)\n        nrm = raw / np.sqrt(np.maximum(cs, 1.0))\n        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),\n                 (raw == raw.max()).astype(np.float32)]\n    else:\n        mfeat = [np.zeros(nc, np.float32)] * 6\n    if model_logits is not None and nc:\n        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))\n        lbest = int(np.argmax(lv)) if lvmax > 0 else -1\n        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0\n        abest = int(np.argmax(ap)) if apmax > 0 else -1\n        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0\n        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),\n                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]\n    else:\n        xfeat = [np.zeros(nc, np.float32)] * 6\n    if frag is not None:\n        fr = np.asarray(frag, np.float32)\n        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]\n    else:\n        ffeat = [np.zeros(nc, np.float32)] * 4\n    return np.column_stack([\n        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),\n        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,\n        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),\n        np.full(nc, np.log(max(nc, 1))),\n        *mfeat, *ffeat, *xfeat,\n    ]).astype(np.float32)\n\n\n# ----------------------------------------------------------------------------------------------\n# MetFrag-lite\n# ----------------------------------------------------------------------------------------------\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\n\n\ndef mol_graph(smi):\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    n = m.GetNumAtoms()\n    w = np.zeros(n)\n    for a in m.GetAtoms():\n        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H\n    if (w == 0).any(): return None\n    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]\n    return w, bonds, n\n\n\ndef _components(n, bonds, drop):\n    adj = [[] for _ in range(n)]\n    for i, (a, b) in enumerate(bonds):\n        if i in drop: continue\n        adj[a].append(b); adj[b].append(a)\n    seen = np.zeros(n, bool); comps = []\n    for s in range(n):\n        if seen[s]: continue\n        stack = [s]; seen[s] = True; cur = [s]\n        while stack:\n            u = stack.pop()\n            for v in adj[u]:\n                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)\n        comps.append(cur)\n    return comps\n\n\ndef fragment_masses(smi, max_breaks=2, max_bonds=34):\n    g = mol_graph(smi)\n    if g is None: return np.zeros(0)\n    w, bonds, n = g\n    nb = len(bonds)\n    if nb == 0 or nb > max_bonds: return np.array([w.sum()])\n    out = {w.sum()}\n    for i in range(nb):\n        for c in _components(n, bonds, {i}):\n            out.add(float(w[c].sum()))\n    if max_breaks >= 2:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for c in _components(n, bonds, {i, j}):\n                    out.add(float(w[c].sum()))\n    return np.array(sorted(out))\n\n\ndef frag_masses_safe(smi):\n    try: return fragment_masses(smi)\n    except Exception: return np.zeros(0)\n\n\n# ADDED: charge carriers a fragment may keep, per precursor adduct (prvsiyan only used +-proton)\nCARRIERS = {\n    "[M+H]+": (PROTON,), "[M-H2O+H]+": (PROTON,), "[M-2H2O+H]+": (PROTON,),\n    "[M+NH4]+": (PROTON, NH4 - E), "[M+Na]+": (PROTON, MASS[\'Na\'] - E), "[M+K]+": (PROTON, MASS[\'K\'] - E),\n    "[M-H]-": (-PROTON,), "[M-H2O-H]-": (-PROTON,), "[M+CH2O2-H]-": (-PROTON, FORMATE - PROTON),\n    "[M+Cl]-": (-PROTON, MASS[\'Cl\'] + E),\n}\n\n\ndef explain_score_adduct(frag_mass, peak_mz, peak_int, adduct, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    car = CARRIERS.get(adduct, (PROTON,) if "+" in adduct[-2:] else (-PROTON,))\n    ion = np.sort(np.concatenate([frag_mass + dh * H + c for dh in h_shifts for c in car]))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)\n\n\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    ion = []\n    for dh in h_shifts:\n        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))\n    ion = np.sort(np.concatenate(ion))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)', 'pv_fp.py': '"""Spectrum -> fingerprint model (prvsiyan FPNet), kept apart so CPU worker processes never import torch."""\nimport math\nimport numpy as np\n# ----------------------------------------------------------------------------------------------\n# spectrum -> fingerprint model\n# ----------------------------------------------------------------------------------------------\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nFP_MAX_PEAKS = 128\nADDUCT_LIST = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",\n               "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",\n               "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",\n               "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]\nADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}\nINSTR_LIST = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]\n\n\ndef instr_family(s):\n    if s is None: return 4\n    t = str(s).lower()\n    if \'timstof\' in t: return 0\n    if \'orbitrap\' in t or \'qft\' in t or \'ftms\' in t or \'hybrid ft\' in t or \'itft\' in t or \'exactive\' in t: return 1\n    if \'tof\' in t: return 2\n    if \'trap\' in t or \'qq\' in t: return 3\n    return 4\n\n\ndef prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):\n    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = (mz <= prec_mz + 1.5)\n    mz, it = mz[keep], it[keep]\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    mx = it.max()\n    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = it >= floor * mx\n    mz, it = mz[keep], it[keep]\n    if len(mz) > max_peaks:\n        order = np.argsort(-it)\n        bucket = (mz // win).astype(np.int64)\n        cnt = {}; sel = []\n        for i in order:\n            b = bucket[i]; c = cnt.get(b, 0)\n            if c < per_win: cnt[b] = c + 1; sel.append(i)\n        sel = np.array(sel)\n        if len(sel) > max_peaks:\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\n        elif len(sel) < max_peaks:\n            rest = np.array([i for i in order if i not in set(sel.tolist())])\n            need = max_peaks - len(sel)\n            if len(rest): sel = np.concatenate([sel, rest[:need]])\n        mz, it = mz[sel], it[sel]\n    o = np.argsort(mz)\n    mz, it = mz[o], it[o]\n    v = np.sqrt(it / it.max())\n    return mz.astype(np.float32), v.astype(np.float32)\n\n\nclass SinEmb(nn.Module):\n    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):\n        super().__init__()\n        n = dim // 2\n        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)\n        self.register_buffer(\'inv\', (2 * math.pi) / wav)\n\n    def forward(self, x):\n        a = x.unsqueeze(-1) * self.inv\n        return torch.cat([torch.sin(a), torch.cos(a)], -1)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h, drop):\n        super().__init__(); self.h = h\n        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)\n        self.n2 = nn.LayerNorm(d)\n        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))\n        self.drop = nn.Dropout(drop)\n\n    def forward(self, x, pad):\n        B, N, D = x.shape; y = self.n1(x)\n        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\n        m = (~pad)[:, None, None, :]\n        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)\n        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))\n        return x + self.drop(self.ff(self.n2(x)))\n\n\nclass FPNet(nn.Module):\n    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):\n        super().__init__()\n        self.d = d\n        self.mz_emb = SinEmb(d)\n        self.nl_emb = SinEmb(d)\n        self.pk = nn.Linear(2 * d + 1, d)\n        self.prec_emb = SinEmb(d)\n        self.ad = nn.Embedding(len(ADDUCT_LIST), d)\n        self.ins = nn.Embedding(len(INSTR_LIST), d)\n        self.gl = nn.Linear(d + 3, d)\n        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\n        self.norm = nn.LayerNorm(d)\n        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\n\n    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\n        B, N = mz.shape\n        nl = (prec[:, None] - mz).clamp(min=0)\n        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\n        g = self.gl(torch.cat([self.prec_emb(prec),\n                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),\n                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)\n        x = torch.cat([g.unsqueeze(1), p], 1)\n        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\n        for b in self.blocks: x = b(x, pad)\n        x = self.norm(x)\n        cls = x[:, 0]\n        msk = (~pad[:, 1:]).float().unsqueeze(-1)\n        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)\n        return self.head(torch.cat([cls, mean], -1))\n\n\ndef load_fp_models(paths, dev):\n    single, merged = [], []\n    nbits = None\n    for pth in paths:\n        ck = torch.load(pth, map_location=\'cpu\', weights_only=False)\n        net = FPNet(ck[\'nbits\'], d=ck[\'d\'], layers=ck[\'layers\']).to(dev).eval()\n        net.load_state_dict(ck[\'model\'])\n        (merged if \'merged\' in str(pth).replace(\'\\\\\', \'/\').split(\'/\')[-1] else single).append(net)\n        nbits = ck[\'nbits\']\n    return single, merged, nbits\n\n\ndef merge_peaks(specs):\n    """specs: list of (mz, it). All peaks collapsed into one pseudo-spectrum."""\n    specs = [(a, b) for a, b in specs if len(a)]\n    if not specs: return np.zeros(0), np.zeros(0)\n    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])\n    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])\n    o = np.argsort(mz); mz, it = mz[o], it[o]\n    keep = np.ones(len(mz), bool)\n    for j in range(1, len(mz)):\n        if mz[j] - mz[j - 1] < 0.005:\n            if it[j] >= it[j - 1]: keep[j - 1] = False\n            else: keep[j] = False\n    return mz[keep], it[keep]\n\n\n@torch.no_grad()\ndef logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):\n    """P: list of (mz, it) already prepped; per-row arrays prec/adduct(str)/instr(str)/ce/mode."""\n    keep = [i for i, (a, _) in enumerate(P) if len(a)]\n    if not keep: return None\n    P = [P[i] for i in keep]\n    B = len(P); N = max(len(a) for a, _ in P)\n    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)\n    for i, (a, b) in enumerate(P):\n        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False\n    T = lambda x: torch.as_tensor(x, device=dev)\n    args = (T(mz), T(it), T(pad),\n            T(np.asarray(prec, np.float32)[keep]),\n            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX[\'<unk>\']) for a in np.asarray(adduct, object)[keep]])),\n            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),\n            T(np.asarray(ce, np.float32)[keep]),\n            T(np.asarray(mode, np.float32)[keep]))\n    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)\n\n\ndef molecule_logits(models, specs, prec, adduct, instr, ce, mode):\n    """prvsiyan model_logits(): mean of (single-model per-spectrum mean) and (merged-model on merged list).\n    specs: list of (mz, it); prec/adduct/instr/ce/mode: per-spectrum."""\n    single, merged, dev = models\n    out = []\n    if single:\n        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]\n        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)\n        if za is not None: out.append(za)\n    if merged:\n        mz, it = merge_peaks(specs)\n        pm = float(np.median(prec))\n        P = [prep_peaks(mz, it, pm)]\n        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])\n        if zb is not None: out.append(zb)\n    return np.mean(out, axis=0) if out else None', 'casmi_engine.py': '"""End-to-end inference (Kaggle notebook body). Mirrors e02_channels.py + e05_rank.py feature blocks exactly,\nbut reads train.parquet / test.parquet directly. Runs locally too (set CASMI_LOCAL=1).\n\nStages: library -> candidate pool -> index -> per-molecule channels -> MetFrag-lite -> features -> ranker -> submission\n"""\nimport os, sys, glob, time, pickle, math\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nimport numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa\nimport pv\nfrom pv import CFG\nimport re as _re\n_MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100, P=30.97376163,\n             Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473, Si=27.9769265325, Se=79.9165213,\n             Na=22.9897692809, K=38.96370668, B=11.0093054, As=74.9215965, Fe=55.9349375, D=2.0141017778)\n_TOK = _re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_mass(f):\n    """ADDED: neutral monoisotopic mass by arithmetic on molecular_formula (prvsiyan fix; immune to\n    riken\'s 0.005 Da precursor precision and gnps adduct mislabels). NaN if an element is untabulated."""\n    if not isinstance(f, str) or not f: return np.nan\n    m = 0.0\n    for el, n in _TOK.findall(f):\n        if el not in _MONO: return np.nan\n        m += _MONO[el] * (int(n) if n else 1)\n    return m\n\nPOOL = None\n\n\nclass RANK:\n    W_A = (0.35, 0.55)      # weight on class-1 simulation rows; two priors averaged\n    SEEDS = (0, 1, 2, 3, 4, 5)  # 6-seed bag (ghazarosbarseghyan91 v14 seeds)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n\nT0 = time.time()\nLOCAL = os.environ.get("CASMI_LOCAL") == "1"\nROOTS = [r"C:\\Users\\HW-LEE\\Desktop\\CASMI"] if LOCAL else ["/kaggle/input"]\n\n\ndef find(name):\n    for root in ROOTS:\n        hits = glob.glob(os.path.join(root, "**", name), recursive=True)\n        if hits: return sorted(hits, key=len)[0]\n    raise FileNotFoundError(name)\n\n\ndef log(msg):\n    print(f"[{time.time()-T0:6.0f}s] {msg}", flush=True)\n\n\n# ---------------------------------------------------------------------------------------------\n# library\n# ---------------------------------------------------------------------------------------------\ndef load_library(path):\n    """Row group by row group, peaks straight to float32: peak RAM stays near the final ~3.5 GB."""\n    f = pq.ParquetFile(path)\n    n_rows = f.metadata.num_rows\n    npk = 0\n    for i in range(f.num_row_groups):\n        c = f.read_row_group(i, columns=["ms2_mzs"]).column(0).combine_chunks()\n        npk += len(c.values)\n    off = np.zeros(n_rows + 1, np.int64)\n    allmz = np.empty(npk, np.float32); allin = np.empty(npk, np.float32)\n    prec, add, pol, ik, smi = [], [], [], [], []\n    fo, tims = [], []\n    r0 = p0 = 0\n    for i in range(f.num_row_groups):\n        t = f.read_row_group(i, columns=["inchikey14", "normalized_smiles", "adduct", "precursor_mz", "ionization_mode",\n                                         "ms2_mzs", "ms2_normalized_intensities", "molecular_formula", "instrument_type"])\n        mzc = t.column("ms2_mzs").combine_chunks(); itc = t.column("ms2_normalized_intensities").combine_chunks()\n        o = mzc.offsets.to_numpy().astype(np.int64)\n        n = len(o) - 1; k = int(o[-1] - o[0])\n        allmz[p0:p0 + k] = mzc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        allin[p0:p0 + k] = itc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        off[r0 + 1:r0 + n + 1] = p0 + (o[1:] - o[0])\n        prec.append(t.column("precursor_mz").to_numpy(zero_copy_only=False).astype(np.float64))\n        add += t.column("adduct").cast(pa.string()).to_pylist()\n        pol.append(np.array(t.column("ionization_mode").cast(pa.string()).to_pylist(), dtype=object) == "positive")\n        ik += t.column("inchikey14").cast(pa.string()).to_pylist()\n        smi += t.column("normalized_smiles").cast(pa.string()).to_pylist()\n        fo += t.column("molecular_formula").cast(pa.string()).to_pylist()\n        tims.append(np.array([isinstance(x, str) and x.lower().replace("-", "").replace(" ", "") == "timstof"\n                              for x in t.column("instrument_type").cast(pa.string()).to_pylist()]))\n        r0 += n; p0 += k\n        del t, mzc, itc\n    prec = np.concatenate(prec); add = np.asarray(add, dtype=object)\n    pol = np.where(np.concatenate(pol), 1, -1).astype(np.int8)\n    nmp = pv.neutral_mass(prec, add)\n    _fm = {x: formula_mass(x) for x in set(v for v in fo if isinstance(v, str))}\n    nmf = np.array([_fm.get(x, np.nan) if isinstance(x, str) else np.nan for x in fo])\n    nm = np.where(np.isfinite(nmf), nmf, nmp)\n    tims = np.concatenate(tims)\n    log(f"library mass index: formula {np.isfinite(nmf).mean():.1%}, timsTOF rows {tims.mean():.1%}")\n    del fo\n    return dict(off=off, mz=allmz, it=allin, prec=prec, nm=nm, pol=pol, tims=tims,\n                ik=np.asarray(ik, dtype=object), smi=np.asarray(smi, dtype=object))\n\n\n# ---------------------------------------------------------------------------------------------\n# candidate pool = COCONUT (prvsiyan fingerprints) U training structures (fingerprinted here)\n# ---------------------------------------------------------------------------------------------\n_g = {}\n\n\ndef _fp_init():\n    from rdkit import RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog("rdApp.*")\n    _g["bits"] = np.load(find("fp_bits.npy"))\n    _g["m2"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\n    _g["m3"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)\n    _g["rk"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)\n\n\ndef fp_and_mass(smi):\n    from rdkit import Chem\n    from rdkit.Chem import MACCSkeys\n    from rdkit.Chem.Descriptors import ExactMolWt\n    if not _g: _fp_init()\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    try:\n        fp = np.concatenate([_g["m2"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g["m3"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g["rk"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[_g["bits"]]\n        return np.packbits(fp), float(ExactMolWt(m))\n    except Exception:\n        return None\n\n\ndef canon_key(smi):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog("rdApp.*")\n    if "te" not in _g: _g["te"] = rdMolStandardize.TautomerEnumerator()\n    try:\n        m = Chem.MolFromSmiles(smi)\n        return Chem.MolToInchiKey(_g["te"].Canonicalize(m))[:14] if m is not None else None\n    except Exception:\n        return None\n\n\ndef build_pool(L, workers):\n    from multiprocessing import Pool as MPool\n    d = os.path.dirname(find("coco_fp.npy"))\n    cm = pickle.load(open(os.path.join(d, "coco_meta.pkl"), "rb"))\n    co_fp = np.load(os.path.join(d, "coco_fp.npy")); co_mass = np.load(os.path.join(d, "coco_mass.npy"))\n    co_keys = np.asarray(cm["keys"], dtype=object); co_smi = np.asarray(cm["smiles"], dtype=object)\n    # ADDED: ChEBI + LIPID MAPS (prvsiyan/chebi-lipidmaps-casmi26), same 6,930-bit packed fingerprints\n    bd = os.path.dirname(find(os.path.join("casmi-chebi-lmsd-structures", "bio_fp.npy"))); bm = pickle.load(open(os.path.join(bd, "bio_meta.pkl"), "rb"))\n    bkeys = np.asarray(bm["keys"], dtype=object); bnew = ~pd.Series(bkeys).isin(set(co_keys)).values\n    co_fp = np.vstack([co_fp, np.load(os.path.join(bd, "bio_fp.npy"))[bnew]])\n    co_mass = np.concatenate([co_mass, np.load(os.path.join(bd, "bio_mass.npy"))[bnew]])\n    co_keys = np.concatenate([co_keys, bkeys[bnew]])\n    co_smi = np.concatenate([co_smi, np.asarray(bm["smiles"], dtype=object)[bnew]])\n    log(f"+ ChEBI/LIPID MAPS {int(bnew.sum()):,} new structures")\n    # PubChemLite CC0 candidates (idea: guhongbin/casmi26-v447-pubchemlite; block as in ahmedberatozer/casmi26-v14-pubchemlite;\n    # data: franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints, copied into the attributed v4g assets).\n    # New arrays are built first and swapped in only if every step succeeds; any failure keeps the previous pool.\n    try:\n        pcd = os.path.dirname(find(os.path.join("casmi26-pubchemlite-cc0-candidate-fingerprints", "pubchemlite_fp.npy")))\n        pcm = pickle.load(open(os.path.join(pcd, "pubchemlite_meta.pkl"), "rb"))\n        pckeys = np.asarray(pcm["keys"], dtype=object)\n        pcnew = ~pd.Series(pckeys).isin(set(co_keys)).values\n        _fp = np.vstack([co_fp, np.load(os.path.join(pcd, "pubchemlite_fp.npy"))[pcnew]])\n        _ms = np.concatenate([co_mass, np.load(os.path.join(pcd, "pubchemlite_mass.npy"))[pcnew]])\n        _ks = np.concatenate([co_keys, pckeys[pcnew]])\n        _sm = np.concatenate([co_smi, np.asarray(pcm["smiles"], dtype=object)[pcnew]])\n        assert len(_fp) == len(_ms) == len(_ks) == len(_sm), (len(_fp), len(_ms), len(_ks), len(_sm))\n        co_fp, co_mass, co_keys, co_smi = _fp, _ms, _ks, _sm\n        log(f"+ PubChemLite CC0 {int(pcnew.sum()):,} new structures")\n    except Exception as _pe:\n        log(f"PubChemLite CC0 skipped: {_pe!r}")\n    tr = pd.DataFrame({"ik": L["ik"], "smi": L["smi"]}).dropna().drop_duplicates("ik")\n    tr = tr[~tr.ik.isin(set(co_keys))]\n    log(f"COCONUT {len(co_keys):,}; training structures to fingerprint {len(tr):,}")\n    with MPool(workers) as mp:\n        res = mp.map(fp_and_mass, list(tr.smi), chunksize=500)\n    ok = np.array([r is not None for r in res])\n    tr_fp = np.stack([r[0] for r in res if r is not None]); tr_mass = np.array([r[1] for r in res if r is not None])\n    fp = np.vstack([co_fp, tr_fp]); mass = np.concatenate([co_mass, tr_mass])\n    keys = np.concatenate([co_keys, tr.ik.values[ok]]); smis = np.concatenate([co_smi, tr.smi.values[ok]])\n    good = np.isfinite(mass)\n    o = np.argsort(np.where(good, mass, 1e18), kind="mergesort")[: good.sum()]\n    P = dict(fp=fp[o], mass=mass[o], keys=keys[o], smiles=smis[o])\n    P["k2i"] = pd.Series(np.arange(len(o)), index=P["keys"]); P["k2i"] = P["k2i"][~P["k2i"].index.duplicated()]\n    log(f"pool {len(o):,} structures")\n    return P\n\n\ndef pool_fps(idx):\n    return np.unpackbits(np.asarray(POOL["fp"][np.asarray(idx, np.int64)]), axis=1)[:, :6930]\n\n\ndef pool_window(t, ppm):\n    a = np.searchsorted(POOL["mass"], t * (1 - ppm / 1e6), "left")\n    b = np.searchsorted(POOL["mass"], t * (1 + ppm / 1e6), "right")\n    return np.arange(a, b)\n\n\n# ---------------------------------------------------------------------------------------------\n# index: neutral-mass sorted library rows + cleaned representatives (all / positive / negative)\n# ---------------------------------------------------------------------------------------------\ndef _reps(L, rows):\n    npk = np.diff(L["off"])[rows]\n    df = pd.DataFrame({"row": rows, "p": L["row_p"][rows], "npk": npk, "tims": L["tims"][rows]})\n    # ADDED: prefer the test instrument (timsTOF) representative, then the richest spectrum\n    df = df.sort_values(["p", "tims", "npk", "row"], ascending=[True, False, False, True]).drop_duplicates("p")\n    rep = df.row.values; rep_nm = L["nm"][rep]\n    o = np.argsort(rep_nm, kind="mergesort"); rep = rep[o]\n    roff, rmz, rit = pv.clean_store(rep, L["off"], L["mz"], L["it"], CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n    return dict(rep=rep, rep_p=L["row_p"][rep], rep_nm=L["nm"][rep], roff=roff, rmz=rmz, rit=rit)\n\n\ndef build_index(L):\n    ok = np.isfinite(L["nm"]) & (L["row_p"] >= 0)\n    rows = np.where(ok)[0]\n    o = np.argsort(L["nm"][rows], kind="mergesort")\n    I = dict(lib_rows=rows[o], lib_nm=L["nm"][rows[o]])\n    I["all"] = _reps(L, rows)\n    I["pos"] = _reps(L, rows[L["pol"][rows] == 1])\n    I["neg"] = _reps(L, rows[L["pol"][rows] == -1])\n    log(f"index: {len(rows):,} library rows; representatives {len(I[\'all\'][\'rep\']):,}")\n    return I\n\n\n# ---------------------------------------------------------------------------------------------\n# per-molecule channels (identical to e02_channels.molecule_channels)\n# ---------------------------------------------------------------------------------------------\ndef clean(mz, it):\n    return pv._clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\n                     CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n\n\ndef analog_search(R, Q, target):\n    lo = np.searchsorted(R["rep_nm"], target - CFG.ANALOG_WIN, "left")\n    hi = np.searchsorted(R["rep_nm"], target + CFG.ANALOG_WIN, "right")\n    asim = np.zeros(hi - lo, np.float32)\n    if hi > lo and Q:\n        shift = (target - R["rep_nm"][lo:hi]).astype(np.float32)\n        for qm, qp in Q:\n            asim = np.maximum(asim, pv.search_shift_pre(qm, qp, lo, hi, R["roff"], R["rmz"], R["rit"], CFG.MZ_TOL, shift))\n    return R["rep_p"][lo:hi], asim\n\n\ndef top_analogs(p, s):\n    if len(p) == 0: return []\n    df = pd.DataFrame({"p": p, "s": s}).groupby("p", sort=False).s.max().sort_values(ascending=False, kind="mergesort")\n    return list(zip(df.index[:CFG.N_ANALOG].astype(int), df.values[:CFG.N_ANALOG].astype(float)))\n\n\ndef molecule_channels(L, I, specs, precs, pols, target):\n    Q = [clean(a, b) for a, b in specs]\n    keep = [i for i, (a, _) in enumerate(Q) if len(a)]\n    Q = [Q[i] for i in keep]; precs = np.asarray(precs)[keep]; pols = np.asarray(pols)[keep]\n    tol = target * CFG.PPM_WIN / 1e6\n    lo = np.searchsorted(I["lib_nm"], target - tol, "left"); hi = np.searchsorted(I["lib_nm"], target + tol, "right")\n    crow = I["lib_rows"][lo:hi]\n    u, inv = np.unique(L["row_p"][crow], return_inverse=True)\n    M = np.zeros((max(len(Q), 1), len(u)), np.float32); MS = np.zeros_like(M)\n    for j, (qm, qp) in enumerate(Q):\n        if len(crow) == 0: break\n        s = pv.search(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                      CFG.INT_POWER, CFG.ENT_WEIGHT)\n        np.maximum.at(M[j], inv, s)\n        sh = (precs[j] - L["prec"][crow]).astype(np.float32)\n        s2 = pv.search_shift_rows(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                                  CFG.INT_POWER, CFG.ENT_WEIGHT, sh)\n        np.maximum.at(MS[j], inv, s2)\n    cnt = np.bincount(inv, minlength=len(u)).astype(np.float32)\n    z = np.zeros(0, np.float32)\n    lib = (u, M.max(0) if len(u) else z, M.mean(0) if len(u) else z, MS.max(0) if len(u) else z, cnt)\n    p, s = analog_search(I["all"], Q, target)\n    ana = top_analogs(p, s)\n    ps, ss = [], []\n    for sign, name in ((1, "pos"), (-1, "neg")):\n        QQ = [q for q, pl in zip(Q, pols) if pl == sign]\n        if QQ:\n            p, s = analog_search(I[name], QQ, target); ps.append(p); ss.append(s)\n    anapol = top_analogs(np.concatenate(ps), np.concatenate(ss)) if ps else []\n    return lib, ana, anapol\n\n\ndef parse_ce(v):\n    try:\n        a = np.abs(np.atleast_1d(np.asarray(v, float)))\n        return float(a.mean()) if len(a) else 25.0\n    except Exception:\n        return 25.0\n\n\n\n\n# ---------------------------------------------------------------------------------------------\n# ranking features: prvsiyan\'s 31 + our blocks (identical to e05_rank.py BLOCKS)\n# ---------------------------------------------------------------------------------------------\ndef _rel(v):\n    v = np.asarray(v, np.float32)\n    return [v, pv._rank_norm(v).astype(np.float32), v - (v.max() if len(v) else 0.0)]\n\n\ndef _analog_feats(cfp, ana):\n    nc = len(cfp)\n    if not ana: return np.zeros((nc, 5), np.float32)\n    afp = pool_fps([p for p, _ in ana]).astype(np.float32); cf = cfp.astype(np.float32)\n    inter = cf @ afp.T\n    tan = inter / (cf.sum(1)[:, None] + afp.sum(1)[None, :] - inter + 1e-9)\n    w = np.clip(np.array([s for _, s in ana], np.float32), 0, None)\n    ap = (tan * (w ** 3)[None, :]).max(1)\n    ap6 = (tan * (w ** 6)[None, :]).max(1)\n    return np.column_stack(_rel(ap) + [ap6, np.full(nc, w[0])]).astype(np.float32)\n\n\ndef blk_base(r, cfp):\n    ana = r["ana"]\n    afp = pool_fps([p for p, _ in ana]) if ana else None\n    sims = np.array([s for _, s in ana], np.float32)\n    return pv.rank_features(cfp, r["lib"], afp, sims, r["zlog"], r["frag"])\n\n\ndef blk_mass(r, cfp):\n    ppm = (POOL["mass"][r["cand"]] - r["target"]) / r["target"] * 1e6\n    return np.column_stack([ppm, np.abs(ppm), np.abs(ppm + 1.5)]).astype(np.float32)\n\n\ndef blk_libx(r, cfp):\n    cnt = np.log1p(r["lib_cnt"])\n    return np.column_stack(_rel(r["lib_mean"]) + _rel(r["libsh"]) + [cnt, cnt - cnt.max()]).astype(np.float32)\n\n\ndef blk_anapol(r, cfp):\n    return _analog_feats(cfp, r["anapol"])\n\n\ndef blk_fragx(r, cfp):\n    return np.column_stack(_rel(r["frag_ad"]) + _rel(r["frag_mean"]) + [r["frag_pol"]]).astype(np.float32)\n\n\nBLOCKS = {"base": blk_base, "mass": blk_mass, "libx": blk_libx, "anapol": blk_anapol, "fragx": blk_fragx}\n\n\n# ---------------------------------------------------------------------------------------------\n# run\n# ---------------------------------------------------------------------------------------------\ndef compute_channels(L, I, models, te):\n    mols = list(te.groupby("molecule_id", sort=True))\n    recs = []\n    for gi, (mid, sub) in enumerate(mols):\n        specs = []\n        for r in sub.itertuples():\n            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)\n            k_ = a_ <= float(r.precursor_mz) + 2.0\n            a_, b_ = a_[k_], b_[k_]\n            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))\n        nms = pv.neutral_mass(sub.precursor_mz.values.astype(np.float64), sub.adduct.astype(str).values)\n        nms = nms[np.isfinite(nms)]\n        rec = dict(mid=mid, cand=np.zeros(0, np.int64))\n        if len(nms):\n            target = float(np.median(nms))\n            modes = np.where(sub.ionization_mode.astype(str).values == "positive", 1.0, -1.0)\n            lib, ana, anapol = molecule_channels(L, I, specs, sub.precursor_mz.values.astype(np.float64), modes, target)\n            cand = pool_window(target, CFG.PPM_WIN)\n            if len(cand) == 0: cand = pool_window(target, CFG.PPM_FALLBACK)\n            ces = np.array([parse_ce(v) for v in sub.collision_energy_ev.values], np.float32)\n            import pv_fp\n            zlog = pv_fp.molecule_logits(models, specs, sub.precursor_mz.values.astype(np.float32),\n                                         list(sub.adduct.astype(str).values), list(sub.instrument_type.astype(str).values),\n                                         ces, modes)\n            u, lmax, lmean, lsh, lcnt = lib\n            pos = {int(x): i for i, x in enumerate(u)}\n            ci = np.array([pos.get(int(c), -1) for c in cand], np.int64); has = ci >= 0\n            rec.update(target=target, cand=cand, zlog=None if zlog is None else zlog.astype(np.float32), specs=specs,\n                       mode=float(np.mean(modes)), modes=modes, adducts=list(sub.adduct.astype(str).values),\n                       ana=ana, anapol=anapol)\n            for nm_, arr in (("lib", lmax), ("lib_mean", lmean), ("libsh", lsh), ("lib_cnt", lcnt)):\n                v = np.zeros(len(cand), np.float32); v[has] = arr[ci[has]]; rec[nm_] = v\n        recs.append(rec)\n        if gi % 50 == 0: log(f"  channels {gi}/{len(mols)}")\n    return mols, recs\n\n\ndef compute_frag(recs, workers):\n    from multiprocessing import Pool as MPool\n    uc = np.unique(np.concatenate([r["cand"] for r in recs]))\n    log(f"MetFrag-lite on {len(uc):,} candidates")\n    with MPool(workers) as mp:\n        fr = mp.map(pv.frag_masses_safe, list(POOL["smiles"][uc]), chunksize=16)\n    fmap = dict(zip(uc.tolist(), fr))\n    for r in recs:\n        if not len(r["cand"]): continue\n        peaks = []\n        for a_, b_ in r.pop("specs"):\n            m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)\n            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n        F1 = np.zeros((len(r["cand"]), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)\n        for j, c in enumerate(r["cand"]):\n            fm = fmap[int(c)]\n            for k, (x, y) in enumerate(peaks):\n                F1[j, k] = pv.explain_score(fm, x, y, mode=r["mode"], tol=CFG.MZ_TOL)\n                F2[j, k] = pv.explain_score_adduct(fm, x, y, r["adducts"][k], tol=CFG.MZ_TOL)\n                F3[j, k] = pv.explain_score(fm, x, y, mode=r["modes"][k], tol=CFG.MZ_TOL)\n        r["frag"] = F1.max(1); r["frag_ad"] = F2.max(1); r["frag_mean"] = F3.mean(1); r["frag_pol"] = F3.max(1)\n\n\ndef _hgb(X, Y, W, seed, gbm):\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    m = HistGradientBoostingClassifier(random_state=seed, **gbm); m.fit(X, Y, sample_weight=W)\n    return m\n\n\ndef fit_rankers(rank_train_path):\n    """Our ranker: simulation rows from 7 query sets (S = 0 class-1 sim, 1 = class-2 sim)."""\n    z = np.load(rank_train_path, allow_pickle=True)\n    X, Y, S = z["X"], z["Y"], z["S"]\n    rankers = [_hgb(X, Y, np.where(S == 0, wA, 1.0 - wA), sd, RANK.GBM) for wA in RANK.W_A for sd in RANK.SEEDS]\n    log(f"our ranker: {len(rankers)} GBMs on {X.shape[0]:,} rows x {X.shape[1]} features")\n    return rankers, [str(b) for b in z["blocks"]]\n\n\ndef fit_pv_rankers(pv_rank_path, priors=(0.30, 0.60), seeds=(0, 1, 2, 3)):\n    """prvsiyan\'s ranker, exactly as in his notebook: shipped rank_train.npz (M = 0 class-1 sim)."""\n    z = np.load(pv_rank_path)\n    rankers = [_hgb(z["X"], z["Y"], np.where(z["M"] == 0, w1, 1.0 - w1), sd, CFG.GBM) for w1 in priors for sd in seeds]\n    log(f"prvsiyan ranker: {len(rankers)} GBMs on {z[\'X\'].shape[0]:,} rows x {z[\'X\'].shape[1]} features")\n    return rankers, z["X"].shape[1]\n\n\ndef score_molecules(recs, rankers, blocks, use_fp):\n    out = {}\n    for r in recs:\n        if not len(r["cand"]): continue\n        rr = r if use_fp else dict(r, zlog=None)\n        cfp = pool_fps(r["cand"])\n        Xm = np.hstack([BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\n        out[r["mid"]] = np.mean([m.predict_proba(Xm)[:, 1] for m in rankers], axis=0)\n    return out\n\n\ndef blend_scores(a, b, wa):\n    """Within-molecule rank blend (both inputs are per-candidate probabilities)."""\n    out = {}\n    for mid in a:\n        ra = 1.0 - pv._rank_norm(a[mid]); rb = 1.0 - pv._rank_norm(b[mid])\n        out[mid] = wa * ra + (1.0 - wa) * rb + 1e-6 * a[mid]\n    return out\n\n\ndef make_submissions(mols, recs, score_sets, sample_path, workers, topn=25):\n    """score_sets: {name: {mid: scores}} -> {name: submission DataFrame}; one canonicalisation pass for all."""\n    from multiprocessing import Pool as MPool\n    cand = {r["mid"]: r["cand"] for r in recs if len(r["cand"])}\n    ordered = {n: {mid: cand[mid][np.argsort(-sc, kind="mergesort")][:topn + 15] for mid, sc in S.items()}\n               for n, S in score_sets.items()}\n    allc = [v for o in ordered.values() for v in o.values()]\n    short = np.unique(np.concatenate(allc)) if allc else np.zeros(0, np.int64)\n    with MPool(workers) as mp:\n        ck = mp.map(canon_key, list(POOL["smiles"][short]), chunksize=64)\n    ckey = dict(zip(short.tolist(), ck))\n    samp = pd.read_csv(sample_path)\n    subs = {}\n    for n, o in ordered.items():\n        rows = []\n        for mid, _ in mols:\n            out, seen = [], set()\n            for c in o.get(mid, []):\n                k = ckey.get(int(c)) or POOL["keys"][c]\n                if k in seen: continue\n                seen.add(k); out.append(POOL["smiles"][c])\n                if len(out) == topn: break\n            while len(out) < topn:\n                out.append("CCO")\n            rows.append((mid, ";".join(out[:topn])))\n        sub = samp[["molecule_id"]].merge(pd.DataFrame(rows, columns=["molecule_id", "smiles"]), on="molecule_id", how="left")\n        sub["smiles"] = sub["smiles"].fillna("CCO")\n        assert len(sub) == len(samp) and sub.molecule_id.duplicated().sum() == 0 and sub.smiles.isnull().sum() == 0\n        assert (sub.smiles.str.split(";").map(len) <= topn).all()\n        subs[n] = sub\n    return subs\n\n\ndef main(test_path, train_path, sample_path, our_rank_path, pv_rank_path, fp_model_paths, workers=4, limit=0, w_pv=0.88):\n    global POOL\n    import torch\n    import pv_fp\n    L = load_library(train_path); log(f"library {len(L[\'prec\']):,} spectra")\n    POOL = build_pool(L, workers)\n    L["row_p"] = POOL["k2i"].reindex(L["ik"]).fillna(-1).values.astype(np.int64)\n    del L["ik"], L["smi"]\n    I = build_index(L)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    single, merged, _ = pv_fp.load_fp_models(fp_model_paths, dev)\n    log(f"fp models: {len(single)} single + {len(merged)} merged on {dev}")\n    te = pq.read_table(test_path).to_pandas()\n    if limit:\n        keep = sorted(te.molecule_id.unique())[:limit]; te = te[te.molecule_id.isin(keep)]\n    log(f"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules")\n    mols, recs = compute_channels(L, I, (single, merged, dev), te)\n    del L, I\n    compute_frag(recs, workers)\n    ours, blocks = fit_rankers(our_rank_path)\n    s_ours = score_molecules(recs, ours, blocks, use_fp=False)\n    del ours\n    pvr, nfeat = fit_pv_rankers(pv_rank_path)\n    s_pv = score_molecules(recs, pvr, ["base"], use_fp=True)\n    del pvr\n    subs = make_submissions(mols, recs, {"blend": blend_scores(s_pv, s_ours, w_pv), "pv": s_pv, "ours": s_ours},\n                            sample_path, workers)\n    log("submissions ready")\n    return subs, recs, {"pv": s_pv, "ours": s_ours}\n'}
for _n, _src in ENG_FILES.items():
    open(os.path.join(ENG_DIR, _n), 'w').write(_src)
open(os.path.join(ENG_DIR, 'eng_runner.py'), 'w').write('"""Our engine (two-ranker, BIO + AFIX) -> {molecule_id: {smiles: [...], keys: [...]}} (blend order, top 40)."""\nimport os, sys, json, time\nimport numpy as np\nif __name__ == \'__main__\':\n    here = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\n    test, train, sample, out = sys.argv[1:5]\n    import casmi_engine as E\n    _orig_find = E.find\n    def _find(n):                                  # v4g ships its own fp_bits.npy (10,226 bits): take prvsiyan\'s 6,930\n        if n == \'fp_bits.npy\':\n            return os.path.join(os.path.dirname(_orig_find(\'coco_fp.npy\')), \'fp_bits.npy\')\n        return _orig_find(n)\n    E.find = _find\n    E.RANK.W_A = (0.35, 0.55); E.RANK.SEEDS = (0, 1); E.CFG.N_ANALOG = 200\n    import glob\n    fp_models = sorted(p for p in glob.glob(\'/kaggle/input/**/fp_*.pt\', recursive=True) if \'casmi26-fp-models-v2\' in p)\n    print(\'engine fp models\', fp_models, flush=True)\n    T0 = time.time()\n    subs, recs, S = E.main(test, train, sample, E.find(\'sim_rank_rows_nofp.npz\'), E.find(\'rank_train.npz\'), fp_models,\n                           workers=os.cpu_count(), w_pv=0.88)\n    bl = E.blend_scores(S[\'pv\'], S[\'ours\'], 0.88)\n    res = {}\n    for r in recs:\n        mid = r[\'mid\']\n        if mid not in bl:\n            continue\n        order = np.argsort(-bl[mid], kind=\'mergesort\')[:80]\n        smis, keys, seen = [], [], set()\n        for c in r[\'cand\'][order]:\n            s = E.POOL[\'smiles\'][c]; k = E.canon_key(s)\n            if k is None or k in seen:\n                continue\n            seen.add(k); smis.append(s); keys.append(k)\n            if len(smis) >= 40:\n                break\n        res[str(mid)] = dict(smiles=smis, keys=keys)\n    json.dump(res, open(out, \'w\'))\n    print(\'engine lists\', len(res), f\'{time.time()-T0:.0f}s\', flush=True)\n')
ENG = {}
try:
    _t0 = time.time()
    rr = subprocess.run([sys.executable, os.path.join(ENG_DIR, 'eng_runner.py'), os.path.join(COMP, 'test.parquet'),
                         os.path.join(COMP, 'train.parquet'), os.path.join(COMP, 'sample_submission.csv'),
                         '/kaggle/working/eng_lists.json'], capture_output=True, text=True, timeout=4 * 3600, cwd=ENG_DIR, env={**os.environ, 'PYTHONHASHSEED': '0'})
    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])
    ENG = json.load(open('/kaggle/working/eng_lists.json'))
    print('engine lists', len(ENG), f'{time.time()-_t0:.0f}s')
except Exception as e:
    print('ENGINE FAILED -> v4g lists kept:', repr(e))


In [ ]:
'''## 4. PubChem-only channel with a popularity prior

This channel searches about 105 million PubChem structures that are not in the main candidate pool. Its fingerprint logits combine FPNet A+B with the panel-excluded fold-safe FPNet (normalized fold0 fraction 0.333333). The raw score retains the A+B coefficient for fixed promotion-margin thresholds. Its score combines the spectrum fit with how well documented a structure is.

$$\\text{score} = z(f\\cdot z) + 0.15\\,\\big(\\log(1+\\#\\text{PubChem substances}) + \\log(1+\\#\\text{PubMed papers})\\big)$$

The 200 most documented structures in the mass window are always added to the shortlist. A gate later decides how many PubChem candidates enter the final list. This prior added +0.004 on the leaderboard.
'''


In [ ]:
V17_PATCH = '\n\n# ---- v17 patch: popularity prior (log1p SID + log1p PMID from PubChem, mapped onto this store by stereo-stripped SMILES) ----\n_POP_DIR = os.environ.get(\'CASMI_POP_DIR\'); _POP_LAM = float(os.environ.get(\'CASMI_POP_LAM\', \'0\')); _POP_UNION = int(os.environ.get(\'CASMI_POP_UNION\', \'0\'))\n_SRC_BONUS = float(os.environ.get(\'CASMI_SRC_BONUS\', \'0\')); _SRC_BITS = int(os.environ.get(\'CASMI_SRC_BITS\', \'0\'))   # v5 depositor-source bonus\n_NP_BONUS = float(os.environ.get(\'CASMI_NP_BONUS\', \'0\'))   # natural-product membership bonus (data notebook v5)\n_LOTUS_W = float(os.environ.get(\'CASMI_LOTUS_W\', \'0\'))   # pre-aligned literature prior\n_SYN_W = 0.5\n_PARENT_PENALTY = 0.0\n_AMBIGUITY = False\n_init_worker0 = init_worker\n\n\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\n    _init_worker0(pc_dir, bits_path, pool_meta_path, code_dir)\n    if _POP_DIR:\n        _n = len(_W[\'mass\'])\n        if _SYN_W:\n            _W[\'syn\'] = np.load(os.path.join(_POP_DIR, \'pc_lsynonyms.npy\'), mmap_mode=\'r\')\n            assert _W[\'syn\'].shape == (_n,), \'Synonym tier-row alignment differs\'\n        if _PARENT_PENALTY:\n            _W[\'parent_rows\'] = np.load(os.path.join(_POP_DIR, \'pc_different_parent_rows.npy\'), mmap_mode=\'r\')\n            assert _W[\'parent_rows\'].ndim == 1\n            assert not len(_W[\'parent_rows\']) or int(_W[\'parent_rows\'][-1]) < _n\n        if _AMBIGUITY:\n            _W[\'formula_count\'] = np.load(os.path.join(_POP_DIR, \'pc_formula_isomers.npy\'), mmap_mode=\'r\')\n            assert _W[\'formula_count\'].shape == (_n,), \'Formula tier-row alignment differs\'\n        _W[\'ls\'] = np.load(os.path.join(_POP_DIR, \'pc_lsid.npy\'), mmap_mode=\'r\')\n        _W[\'lp\'] = np.load(os.path.join(_POP_DIR, \'pc_lpmid.npy\'), mmap_mode=\'r\')\n        if _SRC_BONUS > 0 and _SRC_BITS:\n            _W[\'sm\'] = np.load(os.path.join(_POP_DIR, \'pc_src_mask.npy\'), mmap_mode=\'r\')\n        if _NP_BONUS > 0:\n            _W[\'np\'] = np.load(os.path.join(_POP_DIR, \'pc_np.npy\'), mmap_mode=\'r\')\n        if _LOTUS_W > 0:\n            import sys as _sys\n            _sys.path.insert(0, _POP_DIR)\n            from aligned_public_priors import load_lotus_refs, lotus_log_prior\n            _W[\'lotus\'] = load_lotus_refs(_POP_DIR, tier_rows=len(_W[\'mass\']))\n            _W[\'lotus_log\'] = lotus_log_prior\n\n\ndef probe_one(task):\n    mid, target, z = task\n    t0 = time.time()\n    chem = _W[\'chem\']; Chem = _W[\'Chem\']; gen = _W[\'ecfp4\']\n    tol = float(target) * PPM * 1e-6\n    smis, window_priors = _window_bundle(float(target))\n    n = len(smis)\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0, fz_top=None)\n    if n == 0:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    raw_e = _W[\'raw_e\']; z_e = z[_W[\'sel_e\']].astype(np.float32)\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\n    for i, s in enumerate(smis):\n        m = Chem.MolFromSmiles(s)\n        if m is None:\n            continue\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\n    sc1 = E @ z_e\n    sc1[~ok] = -np.inf\n    n1 = min(PC_N1, int(ok.sum()))\n    diag[\'n_pass1\'] = n1\n    if n1 == 0:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\n    use_pop = (\'ls\' in _W) and _POP_LAM > 0\n    pop = (np.asarray(window_priors[\'ls\'], np.float32) + np.asarray(window_priors[\'lp\'], np.float32)) if use_pop else np.zeros(n, np.float32)\n    if use_pop and \'sm\' in _W:\n        pop = pop + _SRC_BONUS * ((np.asarray(window_priors[\'sm\']) & _SRC_BITS) != 0).astype(np.float32)\n    if use_pop and \'np\' in _W:\n        pop = pop + _NP_BONUS * (np.asarray(window_priors[\'np\']) != 0).astype(np.float32)\n    if use_pop and \'lotus\' in _W:\n        _lr = _W[\'lotus_log\'](window_priors[\'lotus\'])\n        pop = pop + _LOTUS_W * _lr\n    if use_pop and \'syn\' in _W:\n        pop = pop + _SYN_W * np.asarray(window_priors[\'syn\'], np.float32)\n    if use_pop and \'parent_rows\' in _W:\n        _flags = np.zeros(n, dtype=bool)\n        pop = pop - _PARENT_PENALTY * _flags.astype(np.float32)\n    if use_pop and _POP_UNION > 0:\n        pp = np.where(ok, pop, -np.inf); ku = min(_POP_UNION, int(ok.sum()))\n        top1 = np.union1d(top1, np.argpartition(-pp, ku - 1)[:ku])\n    bits = _W[\'bits\']\n    fz, idx = [], []\n    for i in top1:\n        fp = chem.raw_fingerprint(smis[i])\n        if fp is None:\n            continue\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\n    diag[\'n_pass2\'] = len(idx)\n    if not idx:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    fz = np.asarray(fz, np.float64); idx = np.asarray(idx)\n    keyc = {}\n    def key(j):\n        if j not in keyc:\n            keyc[j] = chem.score_key(smis[idx[j]]); diag[\'n_keyed\'] += 1\n        return keyc[j]\n    for j in np.argsort(-fz, kind=\'stable\'):        # gate quantity = v16\'s pc_fz[0] (best non-pool structure by raw f.z)\n        k = key(j)\n        if k is not None and k not in _W[\'pool_keys\']:\n            diag[\'fz_top\'] = float(fz[j]); break\n    zf = (fz - fz.mean()) / (fz.std() + 1e-9)\n    _lam = _POP_LAM\n    if use_pop and \'formula_count\' in _W:\n        _count = np.maximum(np.asarray(np.ones(n, dtype=np.uint16), np.float64)[idx], 1.0)\n        _lam = _POP_LAM * np.clip(np.log1p(_count) / np.log1p(1000.0), 0.5, 1.5)\n    sc = zf + _lam * pop[idx]\n    order = np.argsort(-sc, kind=\'stable\')\n    promotion_evidence = {}\n    out, out_s, out_k, out_p, out_sc, seen = [], [], [], [], [], set()\n    for j in order:\n        k = key(j)\n        if k is None or k in seen:\n            continue\n        seen.add(k)\n        if k in _W[\'pool_keys\']:\n            diag[\'n_in_pool\'] += 1\n            continue\n        out.append(smis[idx[j]]); out_s.append(float(sc[j])); out_k.append(k); out_p.append(float(pop[idx[j]])); out_sc.append(float(sc[j]))\n        promotion_evidence[k] = dict(smiles=smis[idx[j]], raw_fz=float(fz[j]), S=float(sc[j]), pop=float(pop[idx[j]]))\n        if len(out) >= K:\n            break\n    diag[\'n_listed\'] = len(out)\n    diag[\'promotion_evidence\'] = promotion_evidence\n    diag[\'S\'] = float(out_sc[0]) if out_sc else 0.0\n    diag[\'top_pop\'] = float(out_p[0]) if out_p else 0.0\n    diag[\'secs\'] = time.time() - t0\n    return mid, out, out_s, out_k, diag\n\n\n"""Appended to the current embedded PubChem module; no full-tier reordering."""\n_init_worker_single_store = init_worker\n\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\n    _init_worker_single_store(pc_dir, bits_path, pool_meta_path, code_dir)\n    assert _POP_DIR, \'Aligned popularity dataset required for PubChem store union\'\n    assert not _AMBIGUITY and not _PARENT_PENALTY, \'Supplementary formula/parent priors have not been measured\'\n    base = dict(name=\'tier\', mass=_W[\'mass\'], off=_W[\'off\'], buf=_W[\'buf\'])\n    for key in (\'ls\', \'lp\', \'sm\', \'np\', \'lotus\', \'syn\'):\n        if key in _W:\n            assert _W[key].shape == (len(base[\'mass\']),), (key, \'tier-row alignment\')\n            base[key] = _W[key]\n    _W[\'pc_stores\'] = [base]\n    for prefix in (\'delta_pc\', \'growth_pc\'):\n        mass_path = os.path.join(_POP_DIR, prefix + \'_mass.npy\')\n        if not os.path.isfile(mass_path):\n            continue\n        store = dict(name=prefix)\n        for key, suffix in [(\'mass\',\'mass\'), (\'off\',\'off\'), (\'buf\',\'smiles\'), (\'ls\',\'lsid\'), (\'lp\',\'lpmid\'),\n                            (\'sm\',\'src_mask\'), (\'np\',\'np\'), (\'lotus\',\'lotus_refs\'), (\'syn\',\'lsynonyms\'),\n                            (\'src_observed\',\'src_observed\')]:\n            path = os.path.join(_POP_DIR, prefix + \'_\' + suffix + \'.npy\')\n            assert os.path.isfile(path), \'Missing aligned supplementary prior: \' + path\n            store[key] = np.load(path, mmap_mode=\'r\')\n        n = len(store[\'mass\'])\n        assert store[\'off\'].shape == (n + 1,) and store[\'buf\'].ndim == 1\n        assert int(store[\'off\'][0]) == 0 and int(store[\'off\'][-1]) == len(store[\'buf\'])\n        assert np.all(store[\'mass\'][1:] >= store[\'mass\'][:-1]), prefix + \' masses are not sorted\'\n        for key in (\'ls\',\'lp\',\'sm\',\'np\',\'lotus\',\'syn\',\'src_observed\'):\n            assert store[key].shape == (n,), (prefix, key, \'row alignment\')\n        _W[\'pc_stores\'].append(store)\n\ndef _window_bundle(target):\n    """Return SMILES and row-aligned priors gathered from independent sorted stores."""\n    target = float(target)\n    tol = target * PPM * 1e-6\n    smis, parts, counts = [], {key: [] for key in (\'ls\',\'lp\',\'sm\',\'np\',\'lotus\',\'syn\',\'src_observed\')}, {}\n    for store in _W[\'pc_stores\']:\n        a = int(np.searchsorted(store[\'mass\'], target - tol, \'left\'))\n        b = int(np.searchsorted(store[\'mass\'], target + tol, \'right\'))\n        counts[store[\'name\']] = b - a\n        if b <= a:\n            continue\n        off = np.asarray(store[\'off\'][a:b + 1], np.int64)\n        raw = bytes(store[\'buf\'][off[0]:off[-1]])\n        first = int(off[0])\n        smis.extend(raw[int(off[i])-first:int(off[i+1])-first].decode(\'ascii\') for i in range(b-a))\n        for key in parts:\n            if key in store:\n                values = np.asarray(store[key][a:b])\n            else:\n                values = np.full(b-a, 1 if key == \'src_observed\' else 0, np.float32)\n            assert values.shape == (b-a,), (store[\'name\'], key)\n            parts[key].append(values)\n    priors = {key: np.concatenate(values) if values else np.zeros(0, np.float32) for key, values in parts.items()}\n    assert all(len(values) == len(smis) for values in priors.values())\n    _W[\'last_window_counts\'] = counts\n    return smis, priors\n'
'''PubChem-only channel in its own process (S1 code snapshot + FPNet A+B of casmi26-v3-models) -- identical to v3.6'''
CORE = '"""P_pubchem probe core (lever 3). Embedded verbatim into the Kaggle notebook (written to /kaggle/working and imported,\nso worker processes can import it) and imported directly by the local smoke test.\n\nPer molecule:\n  1. f.z logits of the A+B FPNet ensemble, exactly like Engine.model_logits: z = 0.5 * (mean per-spectrum logits +\n     mean per-polarity merged-spectrum logits); models averaged inside ModelBank.\n  2. PubChem window: +-10 ppm around the target neutral mass (median over the molecule\'s spectra), no subsampling.\n  3. pass 1 (src/casmi/pubchem.py tier): partial f.z on the ECFP4 block of the selected bits, keep top PC_N1 = 1000.\n  4. pass 2: full selected-bit fingerprints, full f.z, sort descending.\n  5. walk down the f.z order, compute the metric score key (tautomer-canonical InChIKey14); keep a structure only if\n     its key is NOT in the candidate pool (train U COCONUT) and not already listed; stop at 25.\n  An empty list becomes the single \'CCO\' fallback (never padded).\nReading: sum of reciprocal ranks of PubChem-only truths / ~130 public molecules.\n"""\nfrom __future__ import annotations\n\nimport os\nimport time\n\nimport numpy as np\n\nPC_N1 = 5000\nPPM = 10.0\nK = 25\n\n_W = {}\n\n\n# ------------------------------------------------------------------------------------------ model part (main process)\ndef molecule_logits(bank, spectra):\n    """spectra: list of dict(mz, it, mode, adduct, prec, ce, ce_n). Returns z_avg (nbits,) float32 or None."""\n    from casmi import chem, fpnet\n    from casmi.spectra import merge_spectra\n    items_s = []\n    for s in spectra:\n        pm, pi = fpnet.prep_peaks(s[\'mz\'], s[\'it\'], s[\'prec\'])\n        ce_ok = s[\'ce\'] is not None and not np.isnan(s[\'ce\'])\n        items_s.append(dict(mz=pm, it=pi, prec=float(s[\'prec\']), adduct_ix=chem.adduct_index(s[\'adduct\']),\n                            ce=float(s[\'ce\']) if ce_ok else 0.0, ce_known=1.0 if ce_ok else 0.0,\n                            n_merged=min(int(s.get(\'ce_n\', 1)), 8), mode=float(s[\'mode\'])))\n    items_m = []\n    for md in (1, -1):\n        grp = [s for s in spectra if s[\'mode\'] == md]\n        if not grp:\n            continue\n        mm, ii = merge_spectra([(s[\'mz\'], s[\'it\']) for s in grp])\n        prec = float(np.median([s[\'prec\'] for s in grp]))\n        pm, pi = fpnet.prep_peaks(mm, ii, prec)\n        ces = [s[\'ce\'] for s in grp if s[\'ce\'] is not None and not np.isnan(s[\'ce\'])]\n        adducts = [s[\'adduct\'] for s in grp]\n        ad = max(adducts, key=adducts.count)          # deterministic tie-break (first occurrence)\n        items_m.append(dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(ad),\n                            ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\n                            n_merged=min(sum(max(1, int(s.get(\'ce_n\', 1))) for s in grp), 8), mode=float(md)))\n    if not items_s:\n        return None\n    zs, _ = bank.logits_el(items_s)\n    zm, _ = bank.logits_el(items_m)\n    # The bank already returns z_AB + 0.5 * mean(z_fold0, ..., z_fold4).\n    return (0.5 * (zs.mean(0) + zm.mean(0))).astype(np.float32)\n\n\n# ------------------------------------------------------------------------------------------ PubChem part (workers)\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\n    import sys\n    if code_dir and code_dir not in sys.path:\n        sys.path.insert(0, code_dir)\n    import pandas as pd\n    from casmi import chem\n    _W[\'chem\'] = chem\n    _W[\'mass\'] = np.load(os.path.join(pc_dir, \'pc_mass.npy\'), mmap_mode=\'r\')\n    _W[\'off\'] = np.load(os.path.join(pc_dir, \'pc_off.npy\'), mmap_mode=\'r\')\n    _W[\'buf\'] = np.load(os.path.join(pc_dir, \'pc_smiles.npy\'), mmap_mode=\'r\')\n    bits = np.load(bits_path)\n    _W[\'bits\'] = bits\n    _W[\'sel_e\'] = np.where(bits < 4096)[0]\n    _W[\'raw_e\'] = bits[_W[\'sel_e\']]\n    _W[\'pool_keys\'] = set(pd.read_parquet(pool_meta_path, columns=[\'key\']).key.values.tolist())\n    r = chem._rdkit()\n    _W[\'ecfp4\'] = r[\'gen\'].GetMorganGenerator(radius=2, fpSize=4096)\n    _W[\'Chem\'] = r[\'Chem\']\n\n\ndef _window(target):\n    tol = target * PPM * 1e-6\n    a = int(np.searchsorted(_W[\'mass\'], target - tol, \'left\')); b = int(np.searchsorted(_W[\'mass\'], target + tol, \'right\'))\n    off = np.asarray(_W[\'off\'][a:b + 1]).astype(np.int64)\n    if b <= a:\n        return []\n    raw = bytes(_W[\'buf\'][off[0]:off[-1]])\n    base = off[0]\n    return [raw[off[i] - base:off[i + 1] - base].decode(\'ascii\') for i in range(b - a)]\n\n\ndef probe_one(task):\n    """task = (molecule_id, target_mass, z (nbits,) float32). Returns (molecule_id, [smiles], [f.z], [score keys], diag dict)."""\n    mid, target, z = task\n    t0 = time.time()\n    chem = _W[\'chem\']; Chem = _W[\'Chem\']; gen = _W[\'ecfp4\']\n    smis = _window(float(target))\n    n = len(smis)\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0)\n    if n == 0:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    raw_e = _W[\'raw_e\']; z_e = z[_W[\'sel_e\']].astype(np.float32)\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\n    for i, s in enumerate(smis):\n        m = Chem.MolFromSmiles(s)\n        if m is None:\n            continue\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\n    sc1 = E @ z_e\n    sc1[~ok] = -np.inf\n    n1 = min(PC_N1, int(ok.sum()))\n    diag[\'n_pass1\'] = n1\n    if n1 == 0:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\n    bits = _W[\'bits\']\n    fz, idx = [], []\n    for i in top1:\n        fp = chem.raw_fingerprint(smis[i])\n        if fp is None:\n            continue\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\n    diag[\'n_pass2\'] = len(idx)\n    order = np.argsort(-np.asarray(fz), kind=\'stable\')\n    out, out_fz, out_k, seen = [], [], [], set()\n    for o in order:\n        s = smis[idx[o]]\n        k = chem.score_key(s)\n        diag[\'n_keyed\'] += 1\n        if k is None or k in seen:\n            continue\n        seen.add(k)\n        if k in _W[\'pool_keys\']:\n            diag[\'n_in_pool\'] += 1\n            continue\n        out.append(s); out_fz.append(float(fz[o])); out_k.append(k)\n        if len(out) >= K:\n            break\n    diag[\'n_listed\'] = len(out)\n    diag[\'secs\'] = time.time() - t0\n    return mid, out, out_fz, out_k, diag\n'
RUNNER = '"""PubChem-only channel for the final submission (runs in its own process so its casmi package\n(S1 snapshot, FPNet A+B) never collides with the v1 engine\'s casmi package).\n\nPer molecule: z = z_AB + 0.5 * mean(z_fold0..z_fold4), with the unchanged molecule views;\nbest_pool_fz = max f.z over the pool window (+-10 ppm, fallback 30 ppm) with the same z (as P1 logits.py);\nPubChem-only list = probe_core2.probe_one (top 25 by f.z, keys not in the pool) with f.z values and score keys.\nWrites a json {molecule_id: {pc, pc_fz, pc_keys, best_pool_fz, target}}.\nusage: python pc_runner.py <v3_code_dir> <v3_models_dir> <pool_dir> <pubchem_dir> <test_parquet> <out_json> <n_workers> [max_mols]\n"""\nimport os, sys, json, time\nimport numpy as np, pandas as pd\n\nif __name__ == \'__main__\':\n    V3CODE, V3MOD, POOL_DIR, PC_DIR, TEST, OUT, NW = sys.argv[1:8]\n    MAXM = int(sys.argv[8]) if len(sys.argv) > 8 else 0\n    NW = int(NW)\n    sys.path.insert(0, V3CODE); sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n    import torch, glob\n    import probe_core2 as pc\n    from casmi import fpnet, chem\n    T0 = time.time()\n    models = sorted(glob.glob(os.path.join(V3MOD, \'fpnet_*.pt\')))\n    assert len(models) == 2, models\n    fold0 = os.environ[\'CASMI_FPNET_FOLD0\']\n    assert os.path.isfile(fold0), fold0\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    folds = json.loads(os.environ[\'CASMI_FPNET_FOLDS\'])\n    pinned_hashes = {\'fpnet_fold0.pt\': \'ddf6eaaa8e541a16e3f0357ed30f0346e529054f75d5619aedb6024dcb1bd7f0\', \'fpnet_fold1.pt\': \'1739d5683fa94fd73fe3b37db98ac364cb5dfdc22ad5bea9789be9c48d7cadc4\', \'fpnet_fold2.pt\': \'257062cb6ece333226027de6e313792e7b364517adf42a05e0df950b8e27bbe2\', \'fpnet_fold3.pt\': \'ac61125ef34415d857d58a115e017e0af2aede044e3a5d524029dd1364cbd58c\', \'fpnet_fold4.pt\': \'ad0b2a43343287f7deb4334c4ad766e8aa236d7f0bbeb91e3702910ee49aabe7\'}\n    assert len(folds) == 5 and [os.path.basename(p) for p in folds] == list(pinned_hashes), folds\n    assert all(os.path.isfile(p) for p in folds), folds\n    import hashlib\n    stage = os.path.dirname(fold0)\n    manifest = json.load(open(os.path.join(stage, \'MANIFEST_FIVEFOLD.json\')))\n    validation = json.load(open(os.path.join(stage, \'FIVEFOLD_VALIDATION.json\')))\n    assert manifest[\'complete\'] and manifest[\'updates_per_fold\'] == 82855 and manifest[\'bits\'] == 10226\n    assert validation[\'passed\'] and validation[\'recipe_replay\'][\'passed\'] and validation[\'fold0_metric_replay\'][\'passed\']\n    hashes = {item[\'file\']: item[\'sha256\'] for item in manifest[\'files\']}\n    for k, path in enumerate(folds):\n        assert json.load(open(os.path.join(stage, f\'check_inputs_fold{k}.json\')))[\'passed\']\n        h = hashlib.sha256()\n        with open(path, \'rb\') as checkpoint_file:\n            for chunk in iter(lambda: checkpoint_file.read(4 << 20), b\'\'):\n                h.update(chunk)\n        assert h.hexdigest() == hashes[os.path.basename(path)] == pinned_hashes[os.path.basename(path)], path\n    # Explicit groups: z_AB + 0.5 * mean(z_fold0, ..., z_fold4).\n    # Keep the original A+B coefficient and original 150/600 raw-margin units.\n    class ResidualBank:\n        def __init__(self):\n            self.ab = fpnet.ModelBank(models, device=dev)\n            self.folds = fpnet.ModelBank(folds, device=dev)\n            self.nbits = self.ab.nbits\n            assert self.nbits == self.folds.nbits\n        def logits_el(self, items):\n            za, _ = self.ab.logits_el(items)\n            zf, _ = self.folds.logits_el(items)\n            return za + 0.5 * zf, None\n    bank = ResidualBank()\n    for k, path in enumerate(folds):\n        print(\'FPNET_FOLD_LOADED\', k, os.path.basename(path), \'sha256=\' + pinned_hashes[os.path.basename(path)], flush=True)\n    print(\'FPNET_ENSEMBLE_LOADED ab_models=2 fold_models=5 rule=z_AB+0.5*mean(z_fold0..z_fold4)\', flush=True)\n    bits = np.load(os.path.join(POOL_DIR, \'fp_bits.npy\')); nbits = len(bits)\n    assert bank.nbits == nbits, (bank.nbits, nbits)\n    assert np.array_equal(bits, np.load(os.path.join(os.path.dirname(fold0), \'fp_bits.npy\'))), \'Fingerprint bit order differs\'\n    for checkpoint in folds:\n        assert np.array_equal(bits, np.load(os.path.join(os.path.dirname(checkpoint), \'fp_bits.npy\'))), checkpoint\n    pmass = pd.read_parquet(os.path.join(POOL_DIR, \'pool_meta.parquet\'), columns=[\'mass\']).mass.values.astype(np.float64)\n    assert np.all(np.diff(pmass) >= 0), \'pool not mass-sorted\'\n    pfp = np.load(os.path.join(POOL_DIR, \'pool_fp.npy\'), mmap_mode=\'r\')\n    te = pd.read_parquet(TEST)\n    te[\'nm\'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)\n    tasks, extra = [], {}\n    groups = list(te.groupby(\'molecule_id\', sort=False))\n    if MAXM:\n        groups = groups[:MAXM]\n    for mid, sub in groups:\n        nms = sub.nm.values[np.isfinite(sub.nm.values)]\n        if not len(nms):\n            continue\n        target = float(np.median(nms))\n        spectra = []\n        for r in sub.itertuples():\n            ce = r.collision_energy_ev\n            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)\n            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                                mode=1 if r.ionization_mode == \'positive\' else -1, adduct=r.adduct,\n                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,\n                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))\n        spectra = spectra[:16]\n        try:\n            z = pc.molecule_logits(bank, spectra)\n        except Exception as e:\n            print(\'ERROR logits\', mid, repr(e), flush=True); z = None\n        if z is None:\n            continue\n        tol = target * 10e-6\n        a = np.searchsorted(pmass, target - tol, \'left\'); b = np.searchsorted(pmass, target + tol, \'right\')\n        if b <= a:\n            tol = target * 30e-6\n            a = np.searchsorted(pmass, target - tol, \'left\'); b = np.searchsorted(pmass, target + tol, \'right\')\n        if b > a:\n            F = np.unpackbits(np.asarray(pfp[a:b]), axis=1)[:, :nbits].astype(np.float64)\n            best = float((F @ z.astype(np.float64)).max())\n        else:\n            best = float(\'nan\')\n        extra[mid] = dict(best_pool_fz=best, target=target)\n        tasks.append((mid, target, z))\n    del bank; torch.cuda.empty_cache()\n    print(f\'logits for {len(tasks)} molecules {time.time()-T0:.0f}s\', flush=True)\n    res = {}\n    import multiprocessing as mp\n    ctx = mp.get_context(\'fork\' if sys.platform != \'win32\' else \'spawn\')\n    with ctx.Pool(NW, initializer=pc.init_worker, initargs=(PC_DIR, os.path.join(POOL_DIR, \'fp_bits.npy\'),\n                                                            os.path.join(POOL_DIR, \'pool_meta.parquet\'), V3CODE)) as P:\n        for k, (mid, smis, fzs, keys, d) in enumerate(P.imap_unordered(pc.probe_one, tasks, chunksize=1)):\n            res[mid] = dict(pc=smis, pc_fz=fzs, pc_keys=keys, fz_top=d.get(\'fz_top\'), S=d.get(\'S\'), top_pop=d.get(\'top_pop\', 0.0), promotion_evidence=d.get(\'promotion_evidence\', {}), **extra[mid])\n            if (k + 1) % 25 == 0 or k + 1 == len(tasks):\n                print(f\'  pubchem {k+1}/{len(tasks)} {time.time()-T0:.0f}s\', flush=True)\n    json.dump(res, open(OUT, \'w\'))\n    print(\'pc_runner done\', len(res), f\'{time.time()-T0:.0f}s\', flush=True)\n'
open('/kaggle/working/probe_core2.py', 'w').write(CORE + V17_PATCH)
open('/kaggle/working/pc_runner.py', 'w').write(RUNNER)
print('v17 patch appended to probe_core2:', 'def probe_one' in V17_PATCH)
PC = {}
try:
    rr = subprocess.run([sys.executable, '/kaggle/working/pc_runner.py', V3CODE, V3MOD, POOL_DIR, PC_DIR,
                         os.path.join(COMP, 'test.parquet'), '/kaggle/working/pc_lists.json', '4'],
                        capture_output=True, text=True, timeout=4 * 3600, env={**os.environ, 'PYTHONHASHSEED': '0'})
    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])
    PC = json.load(open('/kaggle/working/pc_lists.json'))
except Exception as e:
    print('PUBCHEM CHANNEL FAILED -> base lists only:', repr(e))
print('pubchem lists', len(PC), f'{time.time()-T0:.0f}s')


In [ ]:
'''## 5. Engine 1: Ahmed Berat Özer's v4n

This step builds the spectral library from `train.parquet` and links the precomputed candidate pool: training structures plus COCONUT, about 711k in total. The pool comes with FPNet fingerprints and fragment tables.
'''


In [ ]:
'''v1 engine code (the 0.358 model's engine) + fe_v4; pool symlinks; spectrum cache from train.parquet'''
sys.path.insert(0, V4CODE)
os.makedirs('work', exist_ok=True)
for f in ['pool_meta.parquet', 'pool_fp.npy', 'pool_frag_off.npy', 'pool_frag_mass.npy', 'fp_bits.npy', 'train_structs.parquet', 'train_fp_sel.npy']:
    src = os.path.join(POOL_DIR, f)
    dst = f'work/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)
from casmi.build import build_spec_cache
build_spec_cache(os.path.join(COMP, 'train.parquet'), 'work')

"""Notebook-embedded, trained-SID-only extension. No runtime structure chemistry."""

def _build_external_reference_cache(base, table, pins):
    import hashlib, os, shutil, tempfile
    from pathlib import Path
    import numpy as np
    import pyarrow as pa
    import pyarrow.parquet as pq
    from casmi import chem
    from casmi.build import LIB_IX, instr_family
    from casmi.spectra import clean_peaks

    def check(ok, reason):
        if not ok:
            raise ValueError(reason)

    def digest(path):
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            for block in iter(lambda: f.read(4 << 20), b''):
                h.update(block)
        return h.hexdigest()

    target = None
    try:
        check(table is not None, 'knowledge table unavailable')
        source = Path(table) / 'external_reference_spectra'
        base = Path(base)
        for name, expected in pins['files'].items():
            path = source / name
            check(path.is_file() and path.stat().st_size <= 32 << 20, 'external file unavailable or too large')
            check(digest(path) == expected, 'external file checksum differs')
        for name, expected in pins['base_files'].items():
            check(digest(base / name) == expected, 'training structure/pool version differs')
        links = pq.read_table(source / 'external_scorer_sid_links.parquet').to_pylist()
        check(len(links) == pins['keys'], 'link count differs')
        k2link = {r['scorer_key']: r for r in links}
        check(len(k2link) == len(links), 'duplicate scorer links')
        quality = pq.read_table(source / 'external_record_quality.parquet').to_pylist()
        pf = pq.ParquetFile(source / 'external_delta.parquet')
        check(pf.metadata.num_rows == len(quality) == pins['delta_rows'], 'delta/quality length differs')
        off = np.load(base / 'spec_off.npy', mmap_mode='r')
        mz = np.load(base / 'spec_mz.npy', mmap_mode='r')
        it = np.load(base / 'spec_it.npy', mmap_mode='r')
        meta = pq.ParquetFile(base / 'spec_meta.parquet')
        n = meta.metadata.num_rows
        check(off.dtype == np.int64 and mz.dtype == it.dtype == np.float32, 'CSR dtype differs')
        check(len(off) == n + 1 and off[0] == 0 and off[-1] == len(mz) == len(it), 'CSR size differs')
        for start in range(0, n, 1 << 18):
            check(np.all(np.diff(off[start:min(n, start + (1 << 18)) + 1]) >= 0), 'CSR offsets decrease')
        for start in range(0, len(mz), 1 << 20):
            a, b = mz[start:start + (1 << 20)], it[start:start + (1 << 20)]
            check(np.all(np.isfinite(a)) and np.all(np.isfinite(b)) and np.all(a > 0) and np.all(b >= 0), 'invalid base peaks')
        required = {'sid','lib','instr','adduct_ix','mode','precursor_mz','n_clean','nm','adduct',
                    'precursor_error_ppm','collision_energy_orig_units','base_peak_intensity','num_peaks',
                    'ce_mean','ce_min','ce_max','ce_n'}
        check(set(meta.schema_arrow.names) == required, 'cache metadata schema differs')
        meta_row = 0
        for batch in meta.iter_batches(batch_size=65536, columns=['sid','n_clean']):
            sids = np.asarray(batch.column(0))
            check(np.all((sids >= 0) & (sids < pins['structure_rows'])), 'base SID out of range')
            check(np.array_equal(np.asarray(batch.column(1)), np.diff(off[meta_row:meta_row + len(sids) + 1])), 'base peak counts differ')
            meta_row += len(sids)
        extra_meta, extra_mz, extra_it, used_keys = [], [], [], set()
        sources = {'MassBank':'massbank', 'MoNA':'mona', 'GNPS':'gnps', 'MSnLib':'pluskal_ms2'}
        row = 0
        for batch in pf.iter_batches(batch_size=128):
            for r in batch.to_pylist():
                q = quality[row]
                check(q['delta_row'] == row and all(q[c] == r[c] for c in ['source','record_id','scorer_key','spectrum_sha256','licence']), 'quality join differs')
                row += 1
                link = k2link.get(r['scorer_key'])
                check(link is not None, 'scorer link missing')
                if link['sid'] is None or not r['in_train_structure'] or q['mass_check_status'] != 'within_diagnostic_threshold':
                    continue
                check(0 <= link['sid'] < pins['structure_rows'] and r['formal_charge'] == 0, 'invalid linked SID/charge')
                check(abs(r['neutral_mono_mass'] - link['mass']) < 1e-5, 'linked structure mass differs')
                check(r['adduct'] in chem.ADDUCTS and r['ionization_mode'] in ['positive','negative'], 'unsupported ion metadata')
                check(bool(r['licence']), 'record licence missing')
                mode = 1 if r['ionization_mode'] == 'positive' else -1
                if chem.ADDUCTS[r['adduct']][3] != mode:
                    continue
                x, y = np.asarray(r['ms2_mzs'], np.float64), np.asarray(r['ms2_normalized_intensities'], np.float64)
                check(len(x) == len(y) and np.all(np.isfinite(x)) and np.all(np.isfinite(y)) and np.all(x > 0) and np.all(y >= 0), 'invalid external peaks')
                prec = float(r['precursor_mz'])
                check(np.isfinite(prec) and prec > 0, 'invalid precursor')
                x, y = clean_peaks(x, y, prec, 0.001, 512, 2.0)
                if not len(x):
                    continue
                check(np.all(np.diff(x) >= 0) and np.all(np.isfinite(y)) and len(x) <= 512, 'cleaned peaks invalid')
                ce = r['collision_energy_ev']
                known_ce = ce is not None and np.isfinite(ce) and r['collision_energy_units'] == 'eV'
                ce = float(ce) if known_ce else None
                d = dict(adduct=r['adduct'], precursor_mz=prec, precursor_error_ppm=None,
                         collision_energy_orig_units=r['collision_energy_units'], base_peak_intensity=None,
                         num_peaks=len(r['ms2_mzs']), ce_mean=ce, ce_min=ce, ce_max=ce,
                         ce_n=1 if known_ce else 0, sid=int(link['sid']), lib=LIB_IX[sources[r['source']]],
                         instr=instr_family(r['instrument_type']), adduct_ix=chem.adduct_index(r['adduct']),
                         mode=mode, nm=float(chem.neutral_mass([prec], [r['adduct']])[0]), n_clean=len(x))
                extra_meta.append(d); extra_mz.append(x); extra_it.append(y); used_keys.add(r['scorer_key'])
        check(len(extra_meta) == pins['added_rows'] and len(used_keys) == pins['added_keys'], 'eligible extension count differs')
        check(len(extra_meta) > 0, 'no eligible external references')
        target = Path(tempfile.mkdtemp(prefix='external_reference_', dir=base.parent))
        os.symlink((base / 'train_structs.parquet').resolve(), target / 'train_structs.parquet')
        counts = np.array([len(x) for x in extra_mz], np.int64)
        eo = np.r_[0, np.cumsum(counts)]
        for name, original, additions in [('spec_mz.npy', mz, extra_mz), ('spec_it.npy', it, extra_it)]:
            out = np.lib.format.open_memmap(target / name, mode='w+', dtype=np.float32, shape=(len(original) + int(eo[-1]),))
            for start in range(0, len(original), 1 << 20):
                end = min(len(original), start + (1 << 20))
                out[start:end] = original[start:end]
            for j, values in enumerate(additions):
                out[len(original) + eo[j]:len(original) + eo[j + 1]] = values
            out.flush(); del out
        out = np.lib.format.open_memmap(target / 'spec_off.npy', mode='w+', dtype=np.int64, shape=(n + len(extra_meta) + 1,))
        for start in range(0, len(off), 1 << 20):
            out[start:min(len(off), start + (1 << 20))] = off[start:start + (1 << 20)]
        out[n + 1:] = eo[1:] + off[-1]
        out.flush(); del out
        with pq.ParquetWriter(target / 'spec_meta.parquet', meta.schema_arrow) as writer:
            for batch in meta.iter_batches(batch_size=65536):
                writer.write_batch(batch)
            writer.write_table(pa.Table.from_pylist(extra_meta, schema=meta.schema_arrow))
        # Verify every copied base byte by bounded array comparison; no second full CSR in RAM.
        for name, original in [('spec_off.npy',off), ('spec_mz.npy',mz), ('spec_it.npy',it)]:
            extended = np.load(target / name, mmap_mode='r')
            for start in range(0, len(original), 1 << 20):
                end = min(len(original), start + (1 << 20))
                check(np.array_equal(original[start:end], extended[start:end]), 'base CSR changed')
            if name == 'spec_off.npy':
                check(extended[-1] == len(mz) + eo[-1] and np.all(np.diff(extended[n:]) > 0), 'appended offsets invalid')
            del extended
        check(pq.ParquetFile(target / 'spec_meta.parquet').metadata.num_rows == n + len(extra_meta), 'extended metadata size differs')
        return dict(path=str(target), added=len(extra_meta), keys=len(used_keys), base_rows=n, error=None)
    except Exception as exc:
        if target is not None:
            shutil.rmtree(target, ignore_errors=True)
        return dict(path=None, added=0, keys=0, error=f'{type(exc).__name__}: ' + ' '.join(str(exc).split())[:200])


def _external_reference_library(state):
    import gc
    import numpy as np
    from casmi.library import Library
    if state['path'] is not None:
        try:
            library = Library(state['path'])
            if len(library.sid) != state['base_rows'] + state['added'] or not np.all(np.diff(library.smass_sorted) >= 0):
                raise ValueError('extended Library index differs')
            if np.any(library.sid < 0) or np.any(library.sid >= len(library.struct_mass)):
                raise ValueError('structure/SID alignment differs')
            return library
        except Exception as exc:
            state['error'] = f'{type(exc).__name__}: ' + ' '.join(str(exc).split())[:200]
            state['path'] = None
            library = None
            gc.collect()
    return Library('work')


def _external_reference_engine(state, library, pool, train_fp, bank):
    import gc
    from casmi.library import Library
    from casmi.engine import Engine, EngineCfg
    if state['path'] is not None:
        try:
            if len(train_fp) != len(library.struct_mass):
                raise ValueError('fingerprint/SID alignment differs')
            engine = Engine(library, pool, train_fp, EngineCfg(generate=True), bank)
            print(f"external reference spectra: {state['added']} added for {state['keys']} structures", flush=True)
            return library, engine
        except Exception as exc:
            state['error'] = f'{type(exc).__name__}: ' + ' '.join(str(exc).split())[:200]
            state['path'] = None
            library = engine = None
            gc.collect()
        library = Library('work')
    print('external reference spectra disabled: ' + str(state['error']), flush=True)
    return library, Engine(library, pool, train_fp, EngineCfg(generate=True), bank)

_EXTERNAL_REFERENCE_PINS = {'files': {'external_delta.parquet': 'e75558832935f28e85a5419022c4cfba4d0d4b49f1063544c317c8ae61c2ac41', 'external_record_quality.parquet': '4f5b227d1c874624182c76e8f8406ba3ee86521744831cb461d6198c898b6c9e', 'external_scorer_sid_links.parquet': '8cd61b8f110bfd6686684d330609a1d0f0d7cc96eab32b9cdd3352939ef443ef'}, 'base_files': {'train_structs.parquet': 'f20aa1cac23af42e430a6633707fda34e48dea97114bdc3e443897f692df3656', 'pool_meta.parquet': '82b8c40315611289b9a4004b3af18d6d3402b3cd5baa4ee5723da07b7e2f6bb0'}, 'keys': 782, 'delta_rows': 1384, 'structure_rows': 275810, 'added_rows': 751, 'added_keys': 469}
_EXTERNAL_REFERENCE_STATE = _build_external_reference_cache('work', TABLE, _EXTERNAL_REFERENCE_PINS)

import casmi; print('casmi from', casmi.__file__)
sys.path.insert(1, os.path.join(V4CODE, 'fe_v4'))
print(f'cache built {time.time()-T0:.0f}s')


In [ ]:
'''v1 engine exactly as in v3.6 (FPNet A, EngineCfg(generate=True)) wrapped by V1FE (recording hooks + families)'''
import numpy as np, pandas as pd, torch, lightgbm as lgb
from casmi.library import Library
from casmi.pool import Pool
from casmi.engine import Engine, EngineCfg, FEATURES
from casmi import fpnet, chem
import v1engine
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
L = _external_reference_library(_EXTERNAL_REFERENCE_STATE); P = Pool('work')
tfp = np.load('work/train_fp_sel.npy')
_FULL = glob.glob('/kaggle/input/**/casmi26-fpnet-full1/**/fpnet_full1.pt', recursive=True)   # v4m
bank = fpnet.ModelBank([_FULL[0] if _FULL else os.path.join(V4MOD, 'fpnet_0.pt')], device=dev)
print('engine FP bank:', _FULL[0] if _FULL else 'fpnet_0 (A) FALLBACK', flush=True)
L, E = _external_reference_engine(_EXTERNAL_REFERENCE_STATE, L, P, tfp, bank)
V = v1engine.V1FE(E, [os.path.join(V4MOD, m['file']) for m in MAN['fe_models']], MAN['families'], device=dev)
if 'derivation' in MAN['families']:
    n_bad = V.mods['derivation'].self_test()
    print('derivation prior self-test mismatches:', n_bad)
    assert n_bad == 0, 'RDKit build differs from the one the derivation priors were built with'
RK = pickle.load(open(os.path.join(V4MOD, MAN['ranker']['file']), 'rb'))
RK_FEATS = list(RK['features']); RK_B = [lgb.Booster(model_str=s) for s in RK['boosters']]
def rank_score(X, names):
    pos = {n: i for i, n in enumerate(names)}
    miss = [f for f in RK_FEATS if f not in pos]
    assert not miss, f'ranker features not produced (would be zero-filled): {miss}'
    Xs = np.ascontiguousarray(X[:, [pos[f] for f in RK_FEATS]], dtype=np.float32)
    return np.mean([b.predict(Xs, num_threads=4) for b in RK_B], axis=0)
print('engine ready; ranker features', len(RK_FEATS), 'boosters', len(RK_B), f'{time.time()-T0:.0f}s')


In [ ]:
'''## 6. Main candidate lists + pool popularity re-rank

For each molecule, v4n scores candidates with FPNet fingerprints, its feature families and a 4-booster LightGBM ranker, and keeps the top 60. The list is then re-ranked by

$$z(\\text{ranker score}) + 0.15\\,\\big(\\log(1+\\#\\text{substances}) + \\log(1+\\#\\text{papers})\\big)$$

Only pool structures are re-ranked. Generated (novel) candidates keep their positions and form their own simulator groups. This step added +0.010 on the leaderboard.
'''


In [ ]:
'''all molecules -> base lists (v3.6 spectra construction and ordering)'''
te = pd.read_parquet(os.path.join(COMP, 'test.parquet'))
te['nm'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)
mols = list(te.groupby('molecule_id', sort=False))
BASE, n_err = {}, 0
POOLPOP_MU = 0.15
POOL_POP = np.load(find('pool_lsid.npy')).astype(np.float64) + np.load(find('pool_lpmid.npy')).astype(np.float64)
assert len(POOL_POP) == len(P.key), (len(POOL_POP), len(P.key))
n_pp = 0
print('v19 pool popularity re-rank, mu', POOLPOP_MU, 'pool', len(POOL_POP))
for gi, (mid, sub) in enumerate(mols):
    nms = sub.nm.values[np.isfinite(sub.nm.values)]
    smis, keys, lib_max, scs, forms = [], [], 0.0, [], []
    pids_ = []
    if len(nms):
        target = float(np.median(nms))
        spectra = []
        for r in sub.itertuples():
            ce = r.collision_energy_ev
            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)
            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),
                                mode=1 if r.ionization_mode == 'positive' else -1, adduct=r.adduct,
                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,
                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))
        try:
            C, X, F, names, info = V.run(spectra, target)
            lib_max = float(info.get('lib_max', 0.0))
            if len(C.pid):
                score = rank_score(np.concatenate([X, F], 1), list(FEATURES) + list(names))
                order = np.argsort(-score, kind='stable')
                seen = set()
                for i in order:
                    k = C.key[i]
                    if k in seen: continue
                    seen.add(k); smis.append(C.smiles[i]); keys.append(k); scs.append(float(score[i])); forms.append(C.formula[i]); pids_.append(int(C.pid[i]))
                    if len(smis) >= TOPN: break
        except Exception as e:
            print('ERROR', mid, repr(e)); n_err += 1
    if POOLPOP_MU > 0 and len(scs) > 1:            # v19: popularity re-rank of the top-60 base list
        try:
            s_ = np.asarray(scs, np.float64); z_ = (s_ - s_.mean()) / (s_.std() + 1e-9)
            pid_ = np.asarray(pids_); isg_ = pid_ < 0
            f_ = z_ + POOLPOP_MU * np.where(isg_, 0.0, POOL_POP[np.maximum(pid_, 0)])
            pool_i = np.where(~isg_)[0]; o_ = np.empty(len(pid_), np.int64)
            o_[isg_] = np.where(isg_)[0]; o_[~isg_] = pool_i[np.argsort(-f_[pool_i], kind='stable')]   # generated keep their slots
            smis = [smis[i] for i in o_]; keys = [keys[i] for i in o_]; scs = [float(f_[i]) for i in o_]
            forms = [forms[i] + ('|gen' if isg_[i] else '') for i in o_]      # generated form their own ICEBERG groups
            n_pp += 1
        except Exception as e:
            print('poolpop failed', mid, repr(e))
    BASE[mid] = (smis, keys, lib_max, scs, forms)
    if (gi + 1) % 25 == 0 or gi == len(mols) - 1:
        print(f'  {gi+1}/{len(mols)} molecules {time.time()-T0:.0f}s', flush=True)
print('errors', n_err, '| popularity re-ranked lists', n_pp)
'''Keep ranked overflow for append-only final coverage.'''
BASE_TAIL_CANDIDATES = {str(k): list(v[0]) for k, v in BASE.items()}


In [ ]:
'''## 7. ICEBERG + GLACIER spectrum simulators

Both models predict each candidate's spectrum and compare it with the measured one. Candidates with the same molecular formula are then re-ranked by

$$z(\\text{score}) + 1.0\\,z(\\text{ICEBERG}) + 1.0\\,z(\\text{GLACIER})$$

Most wrong answers share the right answer's formula, so this is where isomers get separated.

- Second-engine candidates join the simulator input, so they are scored too.
- **GLACIER uses [M+H]+ spectra only.** Restricting it this way added +0.006 on the leaderboard.
'''


In [ ]:
'''--- CLAW FIX (b418/b419): stage health is now RECORDED, never silently swallowed ---'''
'''`glob('/kaggle/input/**/x')` WITHOUT recursive=True matches one directory level only, so it'''
'''misses /kaggle/input/datasets/<owner>/<slug>/. That silently disabled ICEBERG+GLACIER:'''
'''ICE FAILED -> ranker order kept: FileNotFoundError('cp313 rdkit wheel not found')  [b417 log]'''
'''rdkit-2025.3.6-cp312-...whl is not a supported wheel on this platform               [b417 log]'''
'''recursive=True finds the wheel under BOTH mount layouts (same 2025.3.6 cp313 file either way).'''
CLAW_ICE_HEALTH = 'ICE NOT RUN'
CLAW_GL_HEALTH = 'GL NOT RUN'
BASE_ICE = False         # b421: base-list ICEBERG fusion OFF (b420 showed it costs 0.009); see header
'''ICEBERG post-ranker isomer re-scoring (casmi26-iceberg). Optional: any failure keeps the ranker order.'''
import gc
ICE_SCORES, ice_stats = {}, dict(molecules=0, changed_top25=0, changed_top1=0)


'''--- CLAW FIX (b415c): make ICEBERG/GLACIER runnable on a cp313 Kaggle image ---'''
'''The ICEBERG package bundles rdkit-2025.3.6 built for **cp312**; this image is cp313,'''
'''so pip rejects that wheel and install_site() raises before anything is scored'''
'''(n_mols_scored == 0 for ICEBERG *and* GLACIER). The MANIFEST pins 2025.3.6 because'''
'''"FragmentEngine Kekulize differs in 2026.03", so swapping in the runtime's 2026.03'''
'''rdkit would change the scoring. Instead we install the SAME 2025.3.6 release built'''
'''for cp313, so the pinned version is honoured exactly and RDKIT_REQ passes natively.'''
''''''
'''install_site() is idempotent via <site>/.ice_site_ok, which it compares against the'''
'''name list of `pkg/wheels`. We pre-build the site and write that exact list, so both'''
'''runners short-circuit and never invoke the failing pip.'''
def _claw_preseed_site(pkg, site):
    import glob as _g, os as _o, shutil as _s, subprocess as _sp, sys as _sys
    srcs = sorted(_g.glob(_o.path.join(pkg, 'wheels', '*.whl.ice')) +
                  _g.glob(_o.path.join(pkg, 'wheels', '*.whl')))
    # install everything EXCEPT the cp-tag-mismatched rdkit
    keep = [w for w in srcs if not _o.path.basename(w).lower().startswith('rdkit')]
    dropped = [_o.path.basename(w) for w in srcs if w not in keep]
    new = [simulator_rdkit_wheel(pkg)]
    keep = keep + new[:1]
    _o.makedirs(site, exist_ok=True)
    wdir = _o.path.join(site, '.wheels')
    _o.makedirs(wdir, exist_ok=True)
    whl = []
    for w in keep:
        dst = _o.path.join(wdir, _o.path.basename(w)[:-4] if w.endswith('.ice') else _o.path.basename(w))
        _s.copyfile(w, dst)
        whl.append(dst)
    r = _sp.run([_sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps',
                 '--no-compile', '--disable-pip-version-check', '--upgrade', '--target', site] + whl,
                stdout=_sp.PIPE, stderr=_sp.STDOUT, text=True)
    print('[claw] site preseed rc', r.returncode, (r.stdout or '')[-400:], flush=True)
    if r.returncode != 0:
        raise RuntimeError('claw preseed pip failed: ' + (r.stdout or '')[-600:])
    _s.rmtree(wdir, ignore_errors=True)
    # the marker MUST equal exactly what install_site() derives from the PACKAGE list
    want = '\n'.join(_o.path.basename(w) for w in srcs)
    with open(_o.path.join(site, '.ice_site_ok'), 'w') as f:
        f.write(want)
    print('[claw] site ready; substituted', dropped, '->', [_o.path.basename(w) for w in new[:1]], flush=True)
    return want


try:
    ICE_PKG = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))
    sys.path.append(ICE_PKG)
    import fuse as ice_fuse
    del V, E, bank; gc.collect(); torch.cuda.empty_cache()
    # priority: molecules without a library-level match first (class 1 rarely changes), then the rest
    mids = sorted(BASE, key=lambda m: BASE[m][2])
    cands = {m: (ice_fuse.ice_candidates(BASE[m][0], BASE[m][4], TOPN) if BASE_ICE else []) for m in mids}
    if ENG:                              # ADDED (ICE_UNION): also score our engine's candidates (same-formula groups
        from rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF   # formed over the v4g top-N + engine top-40)
        from rdkit import Chem as _Ch
        def _f2(s):
            try:
                return _CMF(_Ch.MolFromSmiles(s))
            except Exception:
                return s
        n_add = 0
        for m in mids:
            e = ENG.get(str(m))
            if not e or not e.get('smiles'):
                continue
            bs, bk, bf = list(BASE[m][0][:TOPN]), set(BASE[m][1][:TOPN]), list(BASE[m][4][:TOPN])
            es = [s for s, k in zip(e['smiles'][:40], e['keys'][:40]) if k not in bk]
            ef = [_f2(s) for s in es]
            extra = ice_fuse.ice_candidates(bs + es, bf + ef, len(bs) + len(es))
            before = len(cands.get(m, []))
            cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))
            n_add += len(cands[m]) - before
        print('ICE_UNION: engine candidates added to ICE input:', n_add, flush=True)
    if ICE_PC:                          # also the PubChem-only list of molecules the v3.6 gate may use (lib_max < LIB_TAU)
        from rdkit.Chem.rdMolDescriptors import CalcMolFormula
        from rdkit import Chem as _Chem
        def _form(s):
            try:
                return CalcMolFormula(_Chem.MolFromSmiles(s))
            except Exception:
                return s
        for m in mids:
            p = PC.get(m) if isinstance(PC, dict) else None
            if p and p.get('pc') and BASE[m][2] < LIB_TAU:
                p['pc_form'] = [_form(x) for x in p['pc']]
                extra = ice_fuse.ice_candidates(p['pc'], p['pc_form'], 25)
                cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))
    cands = {m: c for m, c in cands.items() if c}
    items = ice_fuse.build_ice_input(te, cands)
    print('ICE input molecules', len(items), 'candidates', sum(len(it['cands']) for it in items), f'{time.time()-T0:.0f}s', flush=True)
    _claw_preseed_site(ICE_PKG, '/kaggle/working/ice_site')
    ICE_SCORES = ice_fuse.run_ice(ICE_PKG, items, workdir='/kaggle/working/ice_work', device='cuda', budget_s=ICE_BUDGET,
                                  site='/kaggle/working/ice_site', extra_args=['--any-rdkit'])
    meta_f = '/kaggle/working/ice_work/ice_out.json.meta.json'
    if os.path.exists(meta_f):
        print('ICE meta', open(meta_f).read()[:1500])
except Exception as e:
    CLAW_ICE_HEALTH = f'ICE FAILED: {e!r}'
    print('ICE FAILED -> ranker order kept:', repr(e), flush=True)
'''v4h: GLACIER third re-scoring term (casmi26-glacier) on the SAME runner input; any GLACIER failure -> v4g (ICE only)'''
GL_SCORES, gl_stats, GL_LAM, GL_BUDGET = {}, dict(molecules=0, changed_vs_ice_top25=0, changed_vs_ice_top1=0), 1.0, 4000
try:
    GL_PKG = os.path.dirname(find('casmi26-glacier/**/gl_runner.py'))
    sys.path.append(GL_PKG)
    import gl_fuse
    gc.collect(); torch.cuda.empty_cache()   # ICE ran in its own (finished) process; free the notebook's cached GPU memory
    gl_items = [dict(it, spectra=[s for s in it['spectra'] if s.get('adduct') == '[M+H]+']) for it in items]   # v29: GLACIER on [M+H]+ only (v27: +0.006)
    gl_items = [it for it in gl_items if it['spectra']]
    print('v29 GLACIER [M+H]+ only: molecules', len(gl_items), 'of', len(items))
    GL_SCORES = gl_fuse.run_gl(GL_PKG, gl_items, workdir='/kaggle/working/gl_work', device='cuda', budget_s=GL_BUDGET,
                               site='/kaggle/working/ice_site', wheels=os.path.join(ICE_PKG, 'wheels'), extra_args=['--any-rdkit'])
    meta_f = '/kaggle/working/gl_work/gl_out.json.meta.json'
    print('GL meta', open(meta_f).read()[:1500] if os.path.exists(meta_f) else 'missing')
except Exception as e:
    GL_SCORES = {}
    CLAW_GL_HEALTH = f'GL FAILED: {e!r}'
    print('GL FAILED -> v4g behaviour (ICE only):', repr(e), flush=True)
# N1A complement: Seyit Kaan Gunes; integration as in Ahmed Berat Ozer v15 / guhongbin v465.
# ---- N1A_MCE (gold roadmap 2026-10-09 section 3.2: N1 arm A + I14b merged CE on the proxies only) ---------------
# Molecules WITHOUT a real [M+H]+ spectrum are GLACIER-blind in H1. N1 arm A scores their [M+NH4]+ / [M-H2O+H]+
# spectra as [M+H]+ proxies (NH4: M = prec - 18.033823, peaks >= M + 1.007276 - 1.5 dropped; water loss:
# M = prec + 18.010565 - 1.007276, peaks < prec - 1.5 kept; precursor -> M + 1.007276, relabelled [M+H]+;
# [M-2H2O+H]+ / Na / K never), and a merged-CE proxy (>= 2 distinct CE buckets) is predicted at EVERY bucket and the
# predictions merge()d like the SUM_CE path (I14b). = bench kdev_glbench_n1 arm N1A_D on the blind molecules (521
# bench molecules: 21 blind molecules with a proxy, lift over random +0.270 [0.117, 0.443]; vs H1 on all np molecules
# +0.027 [0.010, 0.047]; merged vs mean CE on those 21 +0.0026). Arm B (proxies also where [M+H]+ exists, -0.047) and
# merged CE for real [M+H]+ spectra (no gain) are NOT done: molecules WITH a real [M+H]+ keep the H1 run above
# untouched (stock gl_runner, mean CE, byte-identical input). The bench's own code runs here, embedded verbatim
# (sha256-checked): glb_core.py (n1_proxy / build_variants / mol_flags / arm_tags / arm_scores) in this process,
# glb_driver.py (driver around the attached gl_runner.py: same env, model, Predictor, canonicalisation, CE bucketing,
# merge, entropy similarity) in its own process like gl_runner. N1 scores are added to GL_SCORES (-> the PubChem-list
# re-rank below and the fused re-rank of cell 19, whose `if ice:` gate is opened ONLY for N1 molecules with scores).
# Any failure -> no N1 scores = H1 behaviour (the commit raises instead). The 12 dummy commit molecules never
# trigger N1, so the commit adds a forced exercise: synthetic [M+H]+-less copies 'n1x:<mid>' ([M+H]+ -> [M+NH4]+ with
# an injected NH4 precursor peak, the first one also -> [M-H2O+H]+) run through the same code + asserts.
N1_ARM = 'N1A_D'                 # bench arm: blind molecule -> spectrum tags (NH4p, WLp), CE mode 'merged'
N1_BUDGET = 1200.0 if IS_RERUN else 600.0
N1_WORK, N1_SITE, N1_DEVICE, N1_DRIVER_EXTRA = '/kaggle/working/n1_work', '/kaggle/working/ice_site', 'cuda', []
N1_MIDS, N1_GL_MIDS, N1_STATS, N1_DRV, N1_HEALTH = set(), set(), {}, {}, 'N1 NOT RUN'
N1X_MIDS, N1X_GL, N1X_STATS = set(), {}, {}
# k2 (E006 port), variant A = N1-complement. E006 (cell 31) re-ranks a GLACIER-blind molecule with ICEBERG when its ICE
# dict has finite scores (in practice [M+Na]+ without [M+H]+: `elif ice_fallback and ice and not gl`). N1 GL scores
# would move such a molecule to the GLACIER branch, so they are kept ONLY for N1 molecules WITHOUT any finite ICE
# score (E006 has no forward-model evidence there: its ICE fallback on an all-None dict is a no-op). False = the plain
# port (N1 GL scores for every N1 molecule, E2 behaviour). Molecules with a real [M+H]+ are never touched either way.
# Challenger route: flexonafft, casmi26-closing-the-evidence-gap v47.
N1_COMPLEMENT = False
print('[N1 challenger] configuration: N1_COMPLEMENT=False; real [M+H]+ stays on H1', flush=True)
N1_ICE_DROPPED = set()           # N1 molecules left to E006's ICE fallback by the complement filter
N1_CORE_SHA256 = 'f7a3b8cd8802a7ba3b69a297603241784a63c069044cf7bd6efda7d68cc8fdfa'      # = subs/out/kdev_glbench_n1_full/glb_core.py
N1_DRIVER_SHA256 = 'e919a154b57bda5a52573481348e38d8b4e7c97883532ea5b385bc3cb6f98502'    # = subs/out/kdev_glbench_n1_full/glb_driver.py
N1_GL_RUNNER_SHA256 = '5c4f74313ac54263d05d987c39d012d81ec57c5f35e1095ecd49a5bcb6440fca'   # casmi26-glacier gl_runner.py (B1 MANIFEST, bench)
# kdev_glbench_n1 glb_core.py, verbatim (proxy construction, variants, arm resolution, aggregation)
N1_CORE_SRC = ''.join([
    '"""casmi26 kdev_glbench_n1 -- pure-python core of the roadmap 3.2 GLACIER bench (no torch / rdkit import).\n',
    '\n',
    'Query handling (N1, roadmap 3.2), on fuse.build_ice_input(..., covered_only=False) spectrum dicts:\n',
    '  [M+NH4]+    M = prec - 18.033823; prec -> M + 1.007276; drop peaks mz >= M + 1.007276 - 1.5; relabel [M+H]+\n',
    '  [M-H2O+H]+  M = prec + 18.010565 - 1.007276; prec -> M + 1.007276; keep peaks mz < original prec - 1.5; relabel\n',
    '  (a proxy left without peaks is dropped; [M-2H2O+H]+, [M+Na]+, [M+K]+ are never proxied)\n',
    '  Hcut (diagnostic): a real [M+H]+ spectrum with the same precursor-region cut (peaks mz >= prec - 1.5 dropped), the\n',
    '  like-for-like reference for the proxies (separates "different adduct" from "precursor region removed").\n',
    '\n',
    'Scoring: the driver returns per-spectrum GLACIER similarities for the frozen bench candidates in two CE modes\n',
    "('mean' = production gl_runner: one prediction at the bucket of mean(ce_ev); 'merged' = I14b: one prediction per\n",
    "distinct CE bucket of a merged-CE spectrum, merged like the SUM_CE path). A molecule's arm score per candidate =\n",
    "mean over the arm's spectra with a similarity = gl_ex_mean restricted to those spectra.\n",
    '\n',
    'Ranking metric: expected reciprocal rank of the truth (cands[mid][0]) under a UNIFORM RANDOM tie-break (no truth-first\n',
    'bias: a truth tied with k decoys gets mean(1/(r..r+k))); unscored candidates rank below every scored one (tied among\n',
    'themselves). Baseline = a random order of all frozen candidates, mean(1/1..1/n). An arm "covers" a molecule when the\n',
    'truth and >= 2 candidates are scored; elsewhere the arm has no evidence (= the random baseline in fill-in totals).\n',
    '"""\n',
    'import math\n',
    'from collections import Counter, OrderedDict\n',
    '\n',
    'import numpy as np\n',
    '\n',
    'PROTON = 1.007276\n',
    'NH4_ADD = 18.033823           # [M+NH4]+ precursor m/z - M\n',
    'H2O = 18.010565\n',
    'CUT_DA = 1.5\n',
    'MIN_PEAKS = 1\n',
    "PROXY_TAG = OrderedDict([('[M+NH4]+', 'NH4p'), ('[M-H2O+H]+', 'WLp')])\n",
    'EPS_TIE = 1e-12\n',
    'N_BOOT = 2000\n',
    'BOOT_SEED = 12345\n',
    '\n',
    "# arm -> (spectrum tags, CE mode, description); 'N1A' is resolved per molecule (see arm_tags)\n",
    'ARMS = OrderedDict([\n',
    "    ('A_H', (('H',), 'mean', '(a) [M+H]+ only, mean CE = B1 production GL')),\n",
    "    ('B_NH4', (('NH4p',), 'mean', '(b) [M+NH4]+ proxies relabelled [M+H]+')),\n",
    "    ('C_WL', (('WLp',), 'mean', '(c) [M-H2O+H]+ proxies relabelled [M+H]+')),\n",
    "    ('D_Hmerged', (('H',), 'merged', '(d) [M+H]+ only, merged-CE per-bucket prediction (I14b)')),\n",
    "    ('A_Hcut', (('Hcut',), 'mean', 'diag: [M+H]+ with the proxy-style precursor cut')),\n",
    "    ('B_NH4m', (('NH4p',), 'merged', 'diag: NH4 proxies, merged CE')),\n",
    "    ('C_WLm', (('WLp',), 'merged', 'diag: water-loss proxies, merged CE')),\n",
    "    ('HNa', (('H', 'Na'), 'mean', 'diag: [M+H]+ and [M+Na]+ (old fwdbench COV; P4 analogue)')),\n",
    "    ('AB', (('H', 'NH4p', 'WLp'), 'mean', 'diag: N1 arm B = [M+H]+ plus all proxies')),\n",
    "    ('N1A', (None, 'mean', 'N1 arm A: [M+H]+ if the molecule has one, else its proxies')),\n",
    "    ('N1A_D', (None, 'merged', 'diag: N1 arm A with merged CE (N1A + I14b)')),\n",
    '])\n',
    '\n',
    '\n',
    '# ------------------------------------------------------------------------------------------------ query handling --\n',
    'def _keep_below(sp, cut, min_peaks):\n',
    "    mz, it = sp.get('mz') or [], sp.get('it') or []\n",
    '    keep = [(float(m), float(i)) for m, i in zip(mz, it) if float(m) < cut]\n',
    '    if len(keep) < min_peaks or not any(i > 0 for _, i in keep):\n',
    '        return None\n',
    '    return [m for m, _ in keep], [i for _, i in keep]\n',
    '\n',
    '\n',
    'def n1_proxy(sp, min_peaks=MIN_PEAKS):\n',
    '    """[M+NH4]+ / [M-H2O+H]+ positive spectrum dict -> relabelled [M+H]+ proxy dict, or None."""\n',
    "    if not isinstance(sp, dict) or sp.get('mode') != 'positive':\n",
    '        return None\n',
    "    ad = sp.get('adduct')\n",
    '    try:\n',
    "        prec = float(sp['prec'])\n",
    '    except Exception:\n',
    '        return None\n',
    '    if not math.isfinite(prec):\n',
    '        return None\n',
    "    if ad == '[M+NH4]+':\n",
    '        M = prec - NH4_ADD\n',
    '        cut = M + PROTON - CUT_DA\n',
    "    elif ad == '[M-H2O+H]+':\n",
    '        M = prec + H2O - PROTON\n',
    '        cut = prec - CUT_DA\n',
    '    else:\n',
    '        return None\n',
    '    k = _keep_below(sp, cut, min_peaks)\n',
    '    if k is None:\n',
    '        return None\n',
    '    out = dict(sp)\n',
    "    out.update(mz=k[0], it=k[1], prec=M + PROTON, adduct='[M+H]+', mode='positive', n1_src=ad)\n",
    '    return out\n',
    '\n',
    '\n',
    'def h_cut(sp, min_peaks=MIN_PEAKS):\n',
    '    """Real [M+H]+ spectrum with the proxy-style cut (peaks mz >= prec - 1.5 dropped); None if nothing is left."""\n',
    "    if not isinstance(sp, dict) or sp.get('mode') != 'positive' or sp.get('adduct') != '[M+H]+':\n",
    '        return None\n',
    "    k = _keep_below(sp, float(sp['prec']) - CUT_DA, min_peaks)\n",
    '    if k is None:\n',
    '        return None\n',
    '    return dict(sp, mz=k[0], it=k[1])\n',
    '\n',
    '\n',
    'def build_variants(items_all):\n',
    '    """fuse.build_ice_input(Q, cands, covered_only=False) items -> driver items whose spectra carry sid / tag\n',
    '    (tags H, Hcut, Na, NH4p, WLp), plus counts. sid = \'<index of the spectrum in the molecule>:<tag>\'."""\n',
    '    out, st = [], Counter()\n',
    '    for it in items_all:\n',
    '        sps = []\n',
    "        for j, sp in enumerate(it.get('spectra') or []):\n",
    "            if sp.get('mode') != 'positive':\n",
    '                continue\n',
    "            ad = sp.get('adduct')\n",
    "            if ad == '[M+H]+':\n",
    "                sps.append(dict(sp, sid=f'{j}:H', tag='H')); st['H'] += 1\n",
    '                c = h_cut(sp)\n',
    '                if c is None:\n',
    "                    st['Hcut_empty'] += 1\n",
    '                else:\n',
    "                    sps.append(dict(c, sid=f'{j}:Hcut', tag='Hcut')); st['Hcut'] += 1\n",
    "            elif ad == '[M+Na]+':\n",
    "                sps.append(dict(sp, sid=f'{j}:Na', tag='Na')); st['Na'] += 1\n",
    '            elif ad in PROXY_TAG:\n',
    '                tag = PROXY_TAG[ad]\n',
    '                p = n1_proxy(sp)\n',
    '                if p is None:\n',
    "                    st[tag + '_empty'] += 1\n",
    '                else:\n',
    "                    sps.append(dict(p, sid=f'{j}:{tag}', tag=tag)); st[tag] += 1\n",
    "        out.append(dict(mid=str(it['mid']), cands=list(it['cands']), spectra=sps))\n",
    '    return out, dict(st)\n',
    '\n',
    '\n',
    'def proxy_prec_check(items):\n',
    '    """Real-data check of the adduct arithmetic: proxy precursor (M + 1.007276) vs the molecule\'s real [M+H]+\n',
    '    precursor m/z (median over its [M+H]+ spectra), in mDa, per proxy tag."""\n',
    '    out = {}\n',
    '    for tag in PROXY_TAG.values():\n',
    '        d = []\n',
    '        for it in items:\n',
    "            H = [float(sp['prec']) for sp in it['spectra'] if sp.get('tag') == 'H']\n",
    '            if H:\n',
    "                d += [1000.0 * (float(sp['prec']) - float(np.median(H))) for sp in it['spectra'] if sp.get('tag') == tag]\n",
    '        out[tag] = dict(n=len(d), median_mda=float(np.median(d)) if d else None,\n',
    '                        max_abs_mda=float(np.max(np.abs(d))) if d else None)\n',
    '    return out\n',
    '\n',
    '\n',
    '# ---------------------------------------------------------------------------------------------------- ranking ----\n',
    'def rr_random_tb(vals, ti=0, eps=EPS_TIE):\n',
    '    """Expected reciprocal rank of vals[ti] under a uniform random tie-break; None entries rank below all scored\n',
    '    ones. None if vals[ti] is None."""\n',
    '    t = vals[ti]\n',
    '    if t is None:\n',
    '        return None\n',
    '    gt = eq = 0\n',
    '    for j, v in enumerate(vals):\n',
    '        if j == ti or v is None:\n',
    '            continue\n',
    '        if v > t + eps:\n',
    '            gt += 1\n',
    '        elif abs(v - t) <= eps:\n',
    '            eq += 1\n',
    '    return sum(1.0 / (gt + 1 + k) for k in range(eq + 1)) / (eq + 1)\n',
    '\n',
    '\n',
    'def rr_truth_first(vals, ti=0, eps=EPS_TIE):\n',
    '    """Old-bench convention with the truth winning every tie (the bias this bench removes)."""\n',
    '    t = vals[ti]\n',
    '    if t is None:\n',
    '        return None\n',
    '    return 1.0 / (1 + sum(1 for j, v in enumerate(vals) if j != ti and v is not None and v > t + eps))\n',
    '\n',
    '\n',
    'def rr_split(vals, ti=0, eps=EPS_TIE):\n',
    '    """fwdbench bench_core.mrr_table convention (ties count half)."""\n',
    '    t = vals[ti]\n',
    '    if t is None:\n',
    '        return None\n',
    '    gt = sum(1 for j, v in enumerate(vals) if j != ti and v is not None and v > t + eps)\n',
    '    eq = sum(1 for j, v in enumerate(vals) if j != ti and v is not None and abs(v - t) <= eps)\n',
    '    return 1.0 / (1 + gt + 0.5 * eq)\n',
    '\n',
    '\n',
    'def rr_rand(n):\n',
    "    return sum(1.0 / k for k in range(1, n + 1)) / n if n > 0 else float('nan')\n",
    '\n',
    '\n',
    'def covered(vals, ti=0):\n',
    '    return vals[ti] is not None and sum(v is not None for v in vals) >= 2\n',
    '\n',
    '\n',
    '# ------------------------------------------------------------------------------------------------ aggregation ----\n',
    'def mol_flags(item):\n',
    "    tags = Counter(sp['tag'] for sp in item['spectra'])\n",
    "    return dict(has_H=tags['H'] > 0, has_NH4p=tags['NH4p'] > 0, has_WLp=tags['WLp'] > 0, has_Na=tags['Na'] > 0)\n",
    '\n',
    '\n',
    'def arm_tags(arm, flags):\n',
    '    tags, mode, _ = ARMS[arm]\n',
    '    if tags is None:                                        # N1 arm A: proxies only where no real [M+H]+ exists\n',
    "        tags = ('H',) if flags['has_H'] else ('NH4p', 'WLp')\n",
    '    return tags, mode\n',
    '\n',
    '\n',
    'def arm_scores(mol_out, tags, mode):\n',
    '    """mol_out = driver output of one molecule {\'cands\': [...], \'spec\': {sid: {\'tag\',\'nb\',\'mean\',\'merged\'}}} ->\n',
    '    per-candidate list (aligned with mol_out[\'cands\']) of the mean similarity over the arm\'s spectra (None if none)."""\n',
    "    cands = mol_out.get('cands') or []\n",
    "    cols = [s[mode] for sid, s in sorted((mol_out.get('spec') or {}).items(), key=lambda kv: _sid_key(kv[0]))\n",
    "            if s.get('tag') in tags and s.get(mode) is not None]\n",
    '    out = []\n',
    '    for i in range(len(cands)):\n',
    '        xs = [c[i] for c in cols if i < len(c) and c[i] is not None]\n',
    '        out.append(float(np.mean(xs)) if xs else None)\n',
    '    return out\n',
    '\n',
    '\n',
    'def _sid_key(sid):\n',
    "    a, _, b = str(sid).partition(':')\n",
    '    try:\n',
    '        return (int(a), b)\n',
    '    except ValueError:\n',
    '        return (10 ** 9, str(sid))\n',
    '\n',
    '\n',
    'def molecule_table(items, out, done):\n',
    '    """-> list of row dicts, one per molecule the driver finished: rr per arm (None = arm does not cover it)."""\n',
    '    rows = []\n',
    '    for it in items:\n',
    "        mid = it['mid']\n",
    '        if mid not in done:\n',
    '            continue\n',
    "        mo = out.get(mid) or {'cands': it['cands'], 'spec': {}}\n",
    "        assert list(mo.get('cands') or it['cands']) == list(it['cands']), ('candidate order changed', mid)\n",
    '        fl = mol_flags(it)\n',
    "        n = len(it['cands'])\n",
    "        nb_h = [s.get('nb', 1) for s in (mo.get('spec') or {}).values() if s.get('tag') == 'H']\n",
    "        r = dict(mid=mid, set=mid.split(':')[0], n=n, rand=rr_rand(n), has_merged_H=any(b >= 2 for b in nb_h), **fl)\n",
    '        for arm in ARMS:\n',
    '            tags, mode = arm_tags(arm, fl)\n',
    '            v = arm_scores(mo, tags, mode)\n',
    "            r['rr_' + arm] = rr_random_tb(v) if covered(v) else None\n",
    "            if arm == 'A_H':\n",
    "                r['rrtf_A_H'] = rr_truth_first(v) if covered(v) else None\n",
    "                r['rrsp_A_H'] = rr_split(v) if covered(v) else None\n",
    "                r['top1_A_H'] = (None if not covered(v) else float(rr_random_tb(v) == 1.0))\n",
    '        rows.append(r)\n',
    '    return rows\n',
    '\n',
    '\n',
    '# ----------------------------------------------------------------------------------------------------- stats -----\n',
    'def _boot_idx(n, seed=BOOT_SEED, nb=N_BOOT):\n',
    '    return np.random.default_rng(seed).integers(0, n, (nb, n))\n',
    '\n',
    '\n',
    "def paired(rows, a, b, sel=None, label=''):\n",
    '    """Paired comparison of arms a and b over the rows both cover (and sel(row) if given)."""\n',
    "    R = [r for r in rows if r['rr_' + a] is not None and r['rr_' + b] is not None and (sel is None or sel(r))]\n",
    '    n = len(R)\n',
    '    res = dict(label=label, a=a, b=b, n=n)\n',
    '    if n == 0:\n',
    '        return res\n',
    "    ra = np.array([r['rr_' + a] for r in R]); rb = np.array([r['rr_' + b] for r in R])\n",
    "    rn = np.array([r['rand'] for r in R])\n",
    '    d = rb - ra\n',
    '    idx = _boot_idx(n)\n',
    '    la, lb = ra - rn, rb - rn\n',
    "    with np.errstate(divide='ignore', invalid='ignore'):\n",
    '        bo_ratio = lb[idx].mean(1) / la[idx].mean(1)\n',
    "    ratio = float(lb.mean() / la.mean()) if la.mean() > 0 else float('nan')\n",
    '    res.update(mrr_a=float(ra.mean()), mrr_b=float(rb.mean()), rand=float(rn.mean()), lift_a=float(la.mean()),\n',
    '               lift_b=float(lb.mean()), delta=float(d.mean()),\n',
    '               delta_ci=[float(x) for x in np.percentile(d[idx].mean(1), [2.5, 97.5])],\n',
    '               ratio=ratio, ratio_ci=[float(x) for x in np.nanpercentile(bo_ratio, [2.5, 97.5])] if n > 1 else [ratio, ratio],\n',
    '               wins=int((d > 1e-9).sum()), ties=int((np.abs(d) <= 1e-9).sum()), losses=int((d < -1e-9).sum()),\n',
    "               by_set={s: int(sum(r['set'] == s for r in R)) for s in sorted({r['set'] for r in R})})\n",
    '    return res\n',
    '\n',
    '\n',
    "def arm_table(rows, arm, sets=('np', 'e180', 'all')):\n",
    '    out = {}\n',
    '    for s in sets:\n',
    "        R = [r for r in rows if r['rr_' + arm] is not None and (s == 'all' or r['set'] == s)]\n",
    '        if not R:\n',
    '            out[s] = dict(n=0)\n',
    '            continue\n',
    "        rr = np.array([r['rr_' + arm] for r in R]); rn = np.array([r['rand'] for r in R])\n",
    '        out[s] = dict(n=len(R), mrr=float(rr.mean()), rand=float(rn.mean()), lift=float((rr - rn).mean()),\n',
    '                      top1=float(np.mean(rr >= 1.0 - 1e-12)))\n',
    '    return out\n',
    '\n',
    '\n',
    "def filled_delta(rows, a, b, sel=None, label=''):\n",
    '    """Mean over ALL selected molecules with a missing arm filled by the random baseline (no evidence)."""\n',
    '    R = [r for r in rows if sel is None or sel(r)]\n',
    '    n = len(R)\n',
    '    res = dict(label=label, a=a, b=b, n=n)\n',
    '    if n == 0:\n',
    '        return res\n',
    "    fa = np.array([r['rr_' + a] if r['rr_' + a] is not None else r['rand'] for r in R])\n",
    "    fb = np.array([r['rr_' + b] if r['rr_' + b] is not None else r['rand'] for r in R])\n",
    '    d = fb - fa\n',
    '    idx = _boot_idx(n)\n',
    '    res.update(mrr_a=float(fa.mean()), mrr_b=float(fb.mean()), delta=float(d.mean()),\n',
    '               delta_ci=[float(x) for x in np.percentile(d[idx].mean(1), [2.5, 97.5])],\n',
    '               n_changed=int((np.abs(d) > 1e-9).sum()))\n',
    '    return res\n',
    '\n',
    '\n',
    'def fmt_p(p):\n',
    "    if not p.get('n'):\n",
    '        return f"{p.get(\'label\', \'\')}: {p[\'a\']} vs {p[\'b\']} n=0"\n',
    '    s = (f"{p.get(\'label\', \'\')}: n={p[\'n\']} {p[\'a\']} {p[\'mrr_a\']:.4f} vs {p[\'b\']} {p[\'mrr_b\']:.4f} (rand {p[\'rand\']:.4f}) "\n',
    '         f"delta {p[\'delta\']:+.4f} CI [{p[\'delta_ci\'][0]:+.4f}, {p[\'delta_ci\'][1]:+.4f}] "\n',
    '         f"lift {p[\'lift_a\']:.4f} -> {p[\'lift_b\']:.4f} ratio {p[\'ratio\']:.3f} CI [{p[\'ratio_ci\'][0]:.3f}, {p[\'ratio_ci\'][1]:.3f}] "\n',
    '         f"W/T/L {p[\'wins\']}/{p[\'ties\']}/{p[\'losses\']} sets {p[\'by_set\']}")\n',
    '    return s\n',
    '\n',
    '\n',
    'def gates(rows):\n',
    '    """Roadmap 3.2 gates. N1-NH4 / N1-WL: kill if the proxy lift < 50% of the paired [M+H]+ lift (same molecules).\n',
    '    I14b: adopt only if delta >= +0.01 on merged-CE e180 molecules and >= 0 on np merged-CE molecules."""\n',
    '    g = OrderedDict()\n',
    "    for key, arm in (('N1_NH4', 'B_NH4'), ('N1_WL', 'C_WL')):\n",
    "        p = paired(rows, 'A_H', arm, label=f'{key} paired [M+H]+ vs proxy')\n",
    "        pc = paired(rows, 'A_Hcut', arm, label=f'{key} paired [M+H]+cut vs proxy (diag)')\n",
    "        if not p.get('n'):\n",
    "            v = 'NO DATA'\n",
    "        elif not math.isfinite(p['ratio']):\n",
    "            v = 'UNDEFINED ([M+H]+ lift <= 0)'\n",
    '        else:\n',
    "            v = 'PASS' if p['ratio'] >= 0.5 else 'KILL'\n",
    "        g[key] = dict(verdict=v, rule='proxy lift >= 0.5 x paired [M+H]+ lift', paired=p, paired_cut=pc)\n",
    "    pe = paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'e180' and r['has_merged_H'], label='I14b e180 merged-CE')\n",
    "    pn = paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'np' and r['has_merged_H'], label='I14b np merged-CE')\n",
    "    if not pe.get('n') or not pn.get('n'):\n",
    "        v = 'NO DATA'\n",
    '    else:\n',
    "        v = 'ADOPT' if (pe['delta'] >= 0.01 and pn['delta'] >= 0.0) else 'REJECT'\n",
    "    g['I14b'] = dict(verdict=v, rule='delta >= +0.01 on e180 merged-CE and >= 0 on np merged-CE', e180=pe, np=pn,\n",
    "                     np_all=paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'np', label='I14b np all'),\n",
    "                     e180_all=paired(rows, 'A_H', 'D_Hmerged', sel=lambda r: r['set'] == 'e180', label='I14b e180 all'))\n",
    '    return g\n',
    '\n',
    '\n',
    'def blind_report(rows):\n',
    '    """Molecules with NO real [M+H]+ spectrum (GLACIER-blind in B1): what N1 arm A recovers there."""\n',
    "    B = [r for r in rows if not r['has_H']]\n",
    "    res = OrderedDict(n_blind=len(B), n_blind_np=sum(r['set'] == 'np' for r in B),\n",
    "                      n_blind_with_proxy=sum(r['has_NH4p'] or r['has_WLp'] for r in B),\n",
    "                      n_blind_proxy_covered=sum(r['rr_N1A'] is not None for r in B))\n",
    "    C = [r for r in B if r['rr_N1A'] is not None]\n",
    '    if C:\n',
    "        rr = np.array([r['rr_N1A'] for r in C]); rn = np.array([r['rand'] for r in C])\n",
    '        idx = _boot_idx(len(C))\n',
    '        res.update(mrr_proxy=float(rr.mean()), rand=float(rn.mean()), lift=float((rr - rn).mean()),\n',
    '                   lift_ci=[float(x) for x in np.percentile((rr - rn)[idx].mean(1), [2.5, 97.5])],\n',
    '                   top1=float(np.mean(rr >= 1.0 - 1e-12)),\n',
    "                   by_src={k: int(sum(1 for r in C if r[k])) for k in ('has_NH4p', 'has_WLp')},\n",
    "                   mids=[r['mid'] for r in C])\n",
    "    res['overall_np'] = filled_delta(rows, 'A_H', 'N1A', sel=lambda r: r['set'] == 'np', label='N1A vs A, all np (blind filled with rand)')\n",
    "    res['overall_all'] = filled_delta(rows, 'A_H', 'N1A', label='N1A vs A, all molecules')\n",
    "    res['overall_np_D'] = filled_delta(rows, 'A_H', 'N1A_D', sel=lambda r: r['set'] == 'np', label='N1A+I14b vs A, all np')\n",
    '    return res\n',
    '\n',
    '\n',
    'def score_corr(new, old):\n',
    '    """new / old: {mid: {smiles: score|None}} -> pooled Pearson + mean per-molecule Spearman over common scored."""\n',
    '    xs, ys, sp = [], [], []\n',
    '    for mid, d in new.items():\n',
    '        o = old.get(mid) or {}\n',
    '        pairs = [(v, o.get(s)) for s, v in d.items() if v is not None and o.get(s) is not None]\n',
    '        if len(pairs) < 3:\n',
    '            continue\n',
    '        a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])\n',
    '        xs.extend(a); ys.extend(b)\n',
    '        ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))\n',
    '        if ra.std() > 0 and rb.std() > 0:\n',
    '            sp.append(float(np.corrcoef(ra, rb)[0, 1]))\n',
    '    if len(xs) < 3:\n',
    '        return dict(n_mols=0)\n',
    '    return dict(n_mols=len(sp), n_pairs=len(xs), pearson=float(np.corrcoef(xs, ys)[0, 1]),\n',
    "                spearman_mean=float(np.mean(sp)) if sp else float('nan'))\n",
    '\n',
    '\n',
    '# -------------------------------------------------------------------------------------------------- self-test ----\n',
    'def selftest():\n',
    '    """Forced unit tests of the pure functions (run on Kaggle before any GPU work and locally by the build)."""\n',
    '    M = 300.0\n',
    '    nh4 = dict(mz=[100.0, 250.0, 299.5, 300.0, 301.0, 318.03], it=[0.1, 1.0, 0.2, 0.3, 0.4, 0.5], prec=M + NH4_ADD,\n',
    "               adduct='[M+NH4]+', mode='positive', instrument='timsTOF', ce_ev=[20.0], ce_orig='20', ce_units='eV')\n",
    '    p = n1_proxy(nh4)\n',
    "    assert p is not None and p['mz'] == [100.0, 250.0, 299.5] and p['it'] == [0.1, 1.0, 0.2], p\n",
    "    assert abs(p['prec'] - (M + PROTON)) < 1e-9 and p['adduct'] == '[M+H]+' and p['n1_src'] == '[M+NH4]+'\n",
    "    assert p['ce_ev'] == [20.0] and p['instrument'] == 'timsTOF' and nh4['adduct'] == '[M+NH4]+'   # input untouched\n",
    "    wl = dict(nh4, mz=[100.0, 281.4, 281.5, 283.0], it=[1.0, 0.5, 0.5, 0.9], prec=M + PROTON - H2O, adduct='[M-H2O+H]+')\n",
    '    p = n1_proxy(wl)\n',
    "    assert p is not None and p['mz'] == [100.0, 281.4] and abs(p['prec'] - (M + PROTON)) < 1e-9, p\n",
    "    for ad in ('[M+Na]+', '[M+K]+', '[M-2H2O+H]+', '[M+H]+', '[M-H]-'):\n",
    '        assert n1_proxy(dict(nh4, adduct=ad)) is None, ad\n',
    "    assert n1_proxy(dict(nh4, mode='negative')) is None\n",
    '    assert n1_proxy(dict(nh4, mz=[310.0, 315.0], it=[1.0, 1.0])) is None             # nothing below the cut\n',
    '    assert n1_proxy(dict(nh4, mz=[100.0], it=[0.0])) is None                           # no positive intensity\n',
    "    h = dict(nh4, adduct='[M+H]+', prec=M + PROTON, mz=[100.0, 299.5, 299.6, 301.0], it=[1.0, 1.0, 1.0, 1.0])\n",
    '    c = h_cut(h)\n',
    "    assert c is not None and c['mz'] == [100.0, 299.5] and c['prec'] == h['prec'] and c['adduct'] == '[M+H]+', c\n",
    '    assert h_cut(nh4) is None\n',
    "    items = [dict(mid='np:X', cands=['A', 'B'], spectra=[h, nh4, wl, dict(h, mode='negative', adduct='[M-H]-'),\n",
    "                                                        dict(h, adduct='[M+Na]+'), dict(h, adduct='[M+K]+')])]\n",
    '    v, st = build_variants(items)\n',
    "    assert [s['sid'] for s in v[0]['spectra']] == ['0:H', '0:Hcut', '1:NH4p', '2:WLp', '4:Na'], v\n",
    '    assert st == dict(H=1, Hcut=1, NH4p=1, WLp=1, Na=1), st\n',
    '    # random tie-break expected RR\n',
    '    assert abs(rr_random_tb([0.5, 0.9, 0.5, 0.1]) - (1 / 2 + 1 / 3) / 2) < 1e-12\n',
    '    assert rr_random_tb([None, 0.2, 0.1]) is None and rr_random_tb([0.2, None, 0.1]) == 1.0\n',
    '    assert abs(rr_random_tb([0.0, 0.0, 0.0, 0.0]) - rr_rand(4)) < 1e-12\n',
    '    assert rr_truth_first([0.0, 0.0, 0.0, 0.0]) == 1.0 and abs(rr_split([0.0, 0.0, 0.0]) - 0.5) < 1e-12\n',
    '    assert abs(rr_rand(3) - (1 + 0.5 + 1 / 3) / 3) < 1e-12\n',
    '    assert covered([0.1, None, 0.2]) and not covered([0.1, None]) and not covered([None, 0.1, 0.2])\n',
    "    # aggregation: mean over the arm's spectra, None-aware; N1A tag resolution\n",
    "    mo = {'cands': ['A', 'B', 'C'], 'spec': {\n",
    "        '0:H': dict(tag='H', nb=3, mean=[0.2, 0.4, None], merged=[0.3, 0.1, None]),\n",
    "        '1:H': dict(tag='H', nb=1, mean=[0.4, None, None], merged=[0.4, None, None]),\n",
    "        '2:NH4p': dict(tag='NH4p', nb=1, mean=[0.9, 0.1, 0.5], merged=[0.9, 0.1, 0.5])}}\n",
    "    am = arm_scores(mo, ('H',), 'mean')\n",
    '    assert len(am) == 3 and np.allclose(am[:2], [0.3, 0.4]) and am[2] is None, am\n',
    "    assert np.allclose(arm_scores(mo, ('H',), 'merged')[:2], [0.35, 0.1])\n",
    "    assert arm_tags('N1A', dict(has_H=True)) == (('H',), 'mean')\n",
    "    assert arm_tags('N1A', dict(has_H=False)) == (('NH4p', 'WLp'), 'mean')\n",
    "    assert arm_tags('N1A_D', dict(has_H=False)) == (('NH4p', 'WLp'), 'merged')\n",
    '    # end-to-end on two synthetic molecules\n',
    "    its = [dict(mid='np:X', cands=['A', 'B', 'C'], spectra=[dict(tag='H'), dict(tag='NH4p')]),\n",
    "           dict(mid='np:Y', cands=['A', 'B', 'C'], spectra=[dict(tag='NH4p')]),\n",
    "           dict(mid='e180:Z', cands=['A', 'B'], spectra=[dict(tag='H')])]\n",
    "    out = {'np:X': mo, 'np:Y': {'cands': ['A', 'B', 'C'], 'spec': {'0:NH4p': dict(tag='NH4p', nb=1, mean=[0.1, 0.5, 0.0], merged=[0.1, 0.5, 0.0])}}}\n",
    "    rows = molecule_table(its, out, done={'np:X', 'np:Y'})\n",
    "    assert [r['mid'] for r in rows] == ['np:X', 'np:Y']\n",
    '    x, y = rows\n',
    "    assert abs(x['rr_A_H'] - 0.5) < 1e-12 and x['rr_B_NH4'] == 1.0 and x['rr_N1A'] == x['rr_A_H'] and x['has_merged_H']\n",
    "    assert y['rr_A_H'] is None and abs(y['rr_N1A'] - 0.5) < 1e-12 and not y['has_H']\n",
    "    p = paired(rows, 'A_H', 'B_NH4')\n",
    "    assert p['n'] == 1 and abs(p['delta'] - 0.5) < 1e-12\n",
    "    f = filled_delta(rows, 'A_H', 'N1A')\n",
    "    assert f['n'] == 2 and abs(f['delta'] - (0.5 - rr_rand(3)) / 2) < 1e-12\n",
    '    g = gates(rows)                                          # A lift < 0 here (0.5 vs rand 0.611)\n',
    "    assert g['N1_NH4']['verdict'].startswith('UNDEFINED') and g['N1_WL']['verdict'] == 'NO DATA', g['N1_NH4']['verdict']\n",
    "    assert g['I14b']['verdict'] == 'NO DATA'\n",
    '\n',
    '    def row(s, a, b, d, mh=True):\n',
    "        return dict(mid=f'{s}:{a}{b}{d}', set=s, rand=0.2, rr_A_H=a, rr_B_NH4=b, rr_A_Hcut=None, rr_C_WL=None,\n",
    '                    rr_D_Hmerged=d, has_merged_H=mh)\n',
    "    gk = gates([row('np', 1.0, 0.5, 1.0), row('np', 0.6, 0.4, 0.6), row('e180', 1.0, None, 1.0)])   # ratio 0.25/0.6\n",
    "    assert gk['N1_NH4']['verdict'] == 'KILL' and abs(gk['N1_NH4']['paired']['ratio'] - 0.25 / 0.6) < 1e-12\n",
    "    assert gk['I14b']['verdict'] == 'REJECT' and gk['I14b']['e180']['delta'] == 0.0      # +0.01 needed on e180\n",
    "    gp = gates([row('np', 1.0, 0.9, 1.0), row('np', 0.6, 0.6, 0.5), row('e180', 0.5, None, 1.0)])   # ratio 1.1/1.2\n",
    "    assert gp['N1_NH4']['verdict'] == 'PASS' and gp['I14b']['verdict'] == 'REJECT'          # np merged delta < 0\n",
    "    ga = gates([row('np', 0.5, 0.5, 0.5), row('e180', 0.5, None, 0.52)])\n",
    "    assert ga['I14b']['verdict'] == 'ADOPT' and abs(ga['I14b']['e180']['delta'] - 0.02) < 1e-12\n",
    '    b = blind_report(rows)\n',
    "    assert b['n_blind'] == 1 and b['n_blind_proxy_covered'] == 1\n",
    "    sc = score_corr({'m': {'a': 1.0, 'b': 2.0, 'c': 3.0}}, {'m': {'a': 2.0, 'b': 4.0, 'c': 6.5}})\n",
    "    assert sc['n_mols'] == 1 and sc['pearson'] > 0.99 and abs(sc['spearman_mean'] - 1.0) < 1e-12\n",
    "    return 'glb_core selftest ok (proxy NH4/WL/Hcut, variants, random tie-break RR, aggregation, paired, gates, blind)'\n",
])
# kdev_glbench_n1 glb_driver.py, verbatim (GLACIER driver: mean + merged-CE per spectrum, own process)
N1_DRIVER_SRC = ''.join([
    '#!/usr/bin/env python\n',
    '"""casmi26 kdev_glbench_n1 -- GLACIER bench driver (runs in its OWN process, like gl_runner.py).\n',
    '\n',
    'Imports the attached casmi26-glacier gl_runner.py as a module and reuses ITS environment setup (offline wheel site,\n',
    'shims, RDKit 2025.03 check), model loading, batched Predictor (canonical-SMILES dedupe, OOM split, fail-fast), query\n',
    "cleaning, CE bucketing, summed-CE merge() and entropy similarity. The 'mean' mode calls gl_runner.score_molecule on\n",
    "one spectrum at a time, so a molecule's mean over its [M+H]+ spectra is exactly the production gl_ex_mean.\n",
    "Added: per-spectrum output, and the I14b 'merged' mode (a merged-CE eV spectrum = >= 2 distinct CE buckets is\n",
    'predicted at each bucket and the predictions are merge()d like the SUM_CE path; single-CE / NCE / no-CE spectra:\n',
    'merged = mean).\n',
    '\n',
    'usage: python glb_driver.py <gl_pkg> <in.json> <out.json> [--device cuda] [--budget S] [--site DIR] [--wheels DIR]\n',
    '                            [--bs 32] [--chunk-jobs 3000] [--fake]\n',
    'in.json : [{"mid", "cands": [smiles], "spectra": [{"sid", "tag", "mz", "it", "prec", "adduct", "mode", "instrument",\n',
    '            "ce_ev", "ce_orig", "ce_units"}]}]\n',
    'out.json: {mid: {"cands": [smiles], "spec": {sid: {"tag", "nb", "mean": [sim|null], "merged": [sim|null]}}}}\n',
    '          (lists aligned with "cands"; null = rejected SMILES / failed prediction)\n',
    'out.json.meta.json: status, counts, timings, versions, errors, "done" = molecules fully scored (input order).\n',
    '--fake: no torch / model / wheel site (local dry run): deterministic pseudo-random predictions, SMILES accepted as is.\n',
    '"""\n',
    'import os, sys, json, time, math, argparse, traceback, hashlib, importlib.util\n',
    '\n',
    'sys.dont_write_bytecode = True\n',
    "NEED = ('setup_env', 'load_model', 'Predictor', 'smiles_ok', '_CANON', 'molecule_jobs', 'score_molecule',\n",
    "        'prep_spectrum', 'prep_query', 'entropy_sim', 'merge', 'post', 'ce_bucket', 'spectrum_ce', 'is_covered',\n",
    "        'instr_token', 'SUM_CE', 'FLOOR', 'TOPK', 'POWER', 'ENTW', 'TOL', 'PPM', 'write_json', 'Log')\n",
    '\n',
    '\n',
    'def sha256(path):\n',
    '    h = hashlib.sha256()\n',
    "    with open(path, 'rb') as f:\n",
    "        for b in iter(lambda: f.read(1 << 20), b''):\n",
    '            h.update(b)\n',
    '    return h.hexdigest()\n',
    '\n',
    '\n',
    'def load_gl(pkg):\n',
    '    """gl_runner.py of the package as a module (no side effects beyond its imports: numpy, numba)."""\n',
    "    spec = importlib.util.spec_from_file_location('gl_runner', os.path.join(pkg, 'gl_runner.py'))\n",
    '    G = importlib.util.module_from_spec(spec)\n',
    "    sys.modules['gl_runner'] = G\n",
    '    spec.loader.exec_module(G)\n',
    '    miss = [n for n in NEED if not hasattr(G, n)]\n',
    '    if miss:\n',
    "        raise RuntimeError(f'gl_runner API differs from the one this driver was written for: missing {miss}')\n",
    '    return G\n',
    '\n',
    '\n',
    '# ------------------------------------------------------------------------------------------- per-spectrum plan ----\n',
    'def merged_buckets(G, sp):\n',
    '    """I14b: sorted distinct CE buckets of a merged-CE eV spectrum when there are >= 2, else None."""\n',
    "    if sp.get('ce_units') == 'NCE':\n",
    '        return None\n',
    '    b = set()\n',
    "    for x in sp.get('ce_ev') or []:\n",
    '        try:\n',
    '            x = float(x)\n',
    '        except (TypeError, ValueError):\n',
    '            continue\n',
    '        if math.isfinite(x):\n',
    '            cb = G.ce_bucket(x)\n',
    '            if math.isfinite(cb):\n',
    '                b.add(float(cb))\n',
    '    return sorted(b) if len(b) >= 2 else None\n',
    '\n',
    '\n',
    'def spec_plan(G, sp):\n',
    '    """None if gl_runner would not score this spectrum, else {\'entry\': molecule_plan entry, \'mb\': buckets|None}."""\n',
    '    if not G.is_covered(sp):\n',
    '        return None\n',
    '    qm, qp = G.prep_spectrum(sp)\n',
    "    return dict(entry=(qm, qp, sp['adduct'], G.instr_token(sp), G.ce_bucket(G.spectrum_ce(sp))), mb=merged_buckets(G, sp))\n",
    '\n',
    '\n',
    'def spec_jobs(G, plan, cands):\n',
    "    J = set(G.molecule_jobs([plan['entry']], cands))\n",
    "    if plan['mb']:\n",
    "        _, _, ad, ins, _ = plan['entry']\n",
    '        for c in cands:\n',
    "            for b in plan['mb']:\n",
    '                J.add((c, ad, ins, b))\n',
    '    return J\n',
    '\n',
    '\n',
    'def spec_scores(G, plan, cands, cache):\n',
    '    """-> (mean {c: sim|None}, merged {c: sim|None}, n_buckets)."""\n',
    "    mean = G.score_molecule([plan['entry']], cands, cache)\n",
    "    if not plan['mb']:\n",
    '        return mean, dict(mean), 1\n',
    "    qm, qp, ad, ins, _ = plan['entry']\n",
    '    merged = {}\n',
    '    for c in cands:\n',
    "        parts = [cache.get((c, ad, ins, b)) for b in plan['mb']]\n",
    '        if any(p is None for p in parts):            # = the SUM_CE path: every component must be predicted\n',
    '            merged[c] = None\n',
    '            continue\n',
    '        pr = G.merge(parts)\n',
    '        pm, pp = G.prep_query(pr[0], pr[1], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\n',
    '        s = float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))\n',
    '        merged[c] = s if math.isfinite(s) else None\n',
    "    return mean, merged, len(plan['mb'])\n",
    '\n',
    '\n',
    '# ------------------------------------------------------------------------------------------------ fake mode ------\n',
    'class FakePredictor:\n',
    '    """Same interface as gl_runner.Predictor (predict(jobs, deadline) -> bool, .cache); deterministic random peaks."""\n',
    '\n',
    '    def __init__(self, G):\n',
    '        self.G, self.cache = G, {}\n',
    '        self.n_pred = self.n_fail = self.n_feat_fail = 0\n',
    '        self.t_pred, self.bs = 0.0, 0\n',
    '\n',
    '    def predict(self, jobs, deadline):\n',
    '        import numpy as np\n',
    '        t0 = time.time()\n',
    '        for j in sorted(jobs):\n',
    '            if j in self.cache:\n',
    '                continue\n',
    '            if time.time() > deadline:\n',
    '                self.t_pred += time.time() - t0\n',
    '                return False\n',
    '            c = self.G._CANON.get(j[0], (None,))[0]\n',
    '            if c is None:\n',
    '                self.cache[j] = None\n',
    '                continue\n',
    '            r = np.random.default_rng(int(hashlib.md5(repr((c, j[1], j[2], float(j[3]))).encode()).hexdigest()[:8], 16))\n',
    '            mz = np.round(r.uniform(40.0, 400.0, 25), 2)\n',
    '            o = self.G.post(np.stack([mz, r.uniform(0.05, 1.0, 25)], 1).astype(np.float32))\n',
    '            self.cache[j] = (o[0].astype(np.float64), o[1].astype(np.float64))\n',
    '            self.n_pred += 1\n',
    '        self.t_pred += time.time() - t0\n',
    '        return True\n',
    '\n',
    '\n',
    'def fake_smiles_ok(G):\n',
    '    def ok(smi):\n',
    '        good = isinstance(smi, str) and bool(smi)\n',
    "        G._CANON[smi] = (smi if good else None, max(1, len(smi) // 2) if good else 0, '' if good else 'empty')\n",
    "        return good, ('' if good else 'empty')\n",
    '    return ok\n',
    '\n',
    '\n',
    '# ------------------------------------------------------------------------------------------------- self-test -----\n',
    'def selftest(G):\n',
    '    """Forced unit test of the merged-CE logic on a synthetic cache (no model); run by the notebook before the GPU\n',
    '    job and by the local build."""\n',
    '    import numpy as np\n',
    "    sp = dict(mz=[50.0, 80.0, 120.0, 150.5], it=[0.2, 1.0, 0.5, 0.3], prec=151.0, adduct='[M+H]+', mode='positive',\n",
    "              instrument='timsTOF', ce_ev=[20.0, 40.0, 60.0], ce_orig='20,40,60', ce_units='eV')\n",
    '    p = spec_plan(G, sp)\n',
    "    assert p is not None and p['mb'] == [20.0, 40.0, 60.0], p\n",
    "    assert p['entry'][2:] == ('[M+H]+', 'QTOF', 40.0), p['entry'][2:]\n",
    "    cands = ['CCO', 'CCC']\n",
    '    J = spec_jobs(G, p, cands)\n',
    "    assert J == {(c, '[M+H]+', 'QTOF', b) for c in cands for b in (20.0, 40.0, 60.0)}, J\n",
    '    rng = np.random.default_rng(0)\n',
    '    cache = {}\n',
    '    for j in sorted(J):\n',
    '        mz = np.sort(rng.choice([50.0, 80.0, 120.0, 150.5, 60.0, 99.0], 4, replace=False))\n',
    '        cache[j] = (mz.astype(np.float64), rng.uniform(0.1, 1.0, 4).astype(np.float64))\n',
    "    cache[('CCC', '[M+H]+', 'QTOF', 60.0)] = None\n",
    '    mean, merged, nb = spec_scores(G, p, cands, cache)\n',
    "    assert nb == 3 and merged['CCC'] is None and mean['CCC'] is not None and mean['CCO'] is not None, (mean, merged)\n",
    "    pr = G.merge([cache[('CCO', '[M+H]+', 'QTOF', b)] for b in (20.0, 40.0, 60.0)])\n",
    '    qm, qp = G.prep_spectrum(sp)\n',
    '    pm, pp = G.prep_query(pr[0], pr[1], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\n',
    "    assert abs(merged['CCO'] - float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))) < 1e-12\n",
    "    pm, pp = G.prep_query(*cache[('CCO', '[M+H]+', 'QTOF', 40.0)], G.FLOOR, G.TOPK, G.POWER, G.ENTW)\n",
    "    assert abs(mean['CCO'] - float(G.entropy_sim(qm, qp, pm, pp, G.TOL, G.PPM))) < 1e-12\n",
    '    p1 = spec_plan(G, dict(sp, ce_ev=[40.0]))                         # single CE: merged == mean\n',
    '    m1, g1, n1 = spec_scores(G, p1, cands, cache)\n',
    "    assert p1['mb'] is None and n1 == 1 and m1 == g1 and abs(m1['CCO'] - mean['CCO']) < 1e-12\n",
    '    assert merged_buckets(G, dict(sp, ce_ev=[20.0, 21.0])) is None    # same bucket twice\n',
    '    assert merged_buckets(G, dict(sp, ce_ev=[0.0, 20.0])) == [5.0, 20.0]\n',
    "    assert merged_buckets(G, dict(sp, ce_units='NCE', ce_orig='20,40')) is None\n",
    '    assert merged_buckets(G, dict(sp, ce_ev=[])) is None\n',
    "    assert spec_plan(G, dict(sp, adduct='[M+NH4]+')) is None and spec_plan(G, dict(sp, mode='negative')) is None\n",
    '    pn = spec_plan(G, dict(sp, ce_ev=[]))                              # no CE: SUM_CE jobs, merged == mean\n',
    "    assert pn['mb'] is None and spec_jobs(G, pn, ['CCO']) == {('CCO', '[M+H]+', 'QTOF', c) for c in G.SUM_CE}\n",
    "    return 'glb_driver selftest ok (merged-CE buckets, jobs, merge == SUM_CE path, single-CE identity, coverage)'\n",
    '\n',
    '\n',
    '# ------------------------------------------------------------------------------------------------------ main -----\n',
    'def parse(argv=None):\n',
    '    ap = argparse.ArgumentParser()\n',
    "    ap.add_argument('pkg'); ap.add_argument('inp'); ap.add_argument('out')\n",
    "    ap.add_argument('--device', default='cuda')\n",
    "    ap.add_argument('--budget', type=float, default=0.0)\n",
    "    ap.add_argument('--site', default='/tmp/glb_site')\n",
    "    ap.add_argument('--wheels', default=None)\n",
    "    ap.add_argument('--bs', type=int, default=32)\n",
    "    ap.add_argument('--chunk-jobs', type=int, default=3000)\n",
    "    ap.add_argument('--fake', action='store_true')\n",
    '    return ap.parse_args(argv)\n',
    '\n',
    '\n',
    'def main(argv=None):\n',
    '    a = parse(argv)\n',
    '    if not a.fake:\n',
    "        os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # = gl_runner.main (deterministic torch)\n",
    '    t_start = time.time()\n',
    "    deadline = t_start + a.budget if a.budget > 0 else float('inf')\n",
    "    meta = dict(model='glacier-bench', status='init', fake=a.fake, n_mols=0, n_mols_done=0, n_spec=0, n_spec_covered=0,\n",
    '                n_spec_merged=0, n_jobs=0, errors=[], done=[])\n',
    '    G = None\n',
    '    result = {}\n',
    '\n',
    '    def write(status=None):\n',
    '        if status:\n',
    "            meta['status'] = status\n",
    "        meta['seconds'] = round(time.time() - t_start, 2)\n",
    "        tmp = a.out + '.tmp'\n",
    "        with open(tmp, 'w') as f:\n",
    '            json.dump(result, f, allow_nan=False)\n',
    '        os.replace(tmp, a.out)\n',
    "        with open(a.out + '.meta.json.tmp', 'w') as f:          # atomic too: a kill mid-write never leaves bad JSON\n",
    '            json.dump(meta, f, allow_nan=False)\n',
    "        os.replace(a.out + '.meta.json.tmp', a.out + '.meta.json')\n",
    '\n',
    '    def log(*x):\n',
    "        print(f'[glb {time.time() - t_start:7.1f}s]', *x, file=sys.stderr, flush=True)\n",
    '\n',
    '    try:\n',
    '        mols = json.load(open(a.inp))\n',
    "        meta['n_mols'] = len(mols)\n",
    '        write()\n',
    '        G = load_gl(a.pkg)\n',
    "        meta['gl_runner_sha256'] = sha256(os.path.join(a.pkg, 'gl_runner.py'))\n",
    '        if a.fake:\n',
    "            meta['rdkit'] = 'fake'\n",
    '            G.smiles_ok = fake_smiles_ok(G)\n',
    '            pred = FakePredictor(G)\n',
    "            log('FAKE mode: pseudo-random predictions, no model')\n",
    '        else:\n',
    '            ns = argparse.Namespace(pkg=a.pkg, site=a.site, wheels=a.wheels, any_rdkit=False)\n',
    "            meta['rdkit'] = G.setup_env(ns, log)\n",
    '            import torch\n',
    "            meta['torch'] = torch.__version__\n",
    '            dev = a.device\n',
    "            if dev.startswith('cuda') and not torch.cuda.is_available():\n",
    "                log('cuda unavailable -> cpu'); dev = 'cpu'\n",
    "            meta['device'] = dev\n",
    '            torch.use_deterministic_algorithms(True, warn_only=True)\n',
    '            torch.backends.cudnn.benchmark = False\n',
    '            model = G.load_model(os.path.abspath(a.pkg), dev, log)\n',
    '            pred = G.Predictor(model, dev, a.bs, log)\n',
    '            log(f\'model loaded on {dev} (rdkit {meta["rdkit"]}, torch {torch.__version__}, numba {G.HAVE_NUMBA})\')\n',
    "        meta['t_load'] = round(time.time() - t_start, 2)\n",
    '        okc = {}\n',
    '        chunk, chunk_jobs = [], set()\n',
    '\n',
    '        def flush():\n',
    '            nonlocal chunk, chunk_jobs\n',
    '            if not chunk:\n',
    '                return True\n',
    '            t0 = time.time()\n',
    '            if not pred.predict(chunk_jobs, deadline):\n',
    '                return False\n',
    '            for mid, cands, plans in chunk:\n',
    '                spec = {}\n',
    '                for sid, tag, pl in plans:\n',
    '                    try:\n',
    '                        mean, merged, nb = spec_scores(G, pl, cands, pred.cache)\n',
    '                    except Exception as e:\n',
    "                        meta['errors'].append(f'{mid} {sid}: score {repr(e)[:200]}')\n",
    '                        continue\n',
    "                    spec[sid] = dict(tag=tag, nb=nb, mean=[mean.get(c) for c in result[mid]['cands']],\n",
    "                                     merged=[merged.get(c) for c in result[mid]['cands']])\n",
    "                result[mid]['spec'] = spec\n",
    "                meta['done'].append(mid)\n",
    "            meta['n_mols_done'] = len(meta['done'])\n",
    '            write()\n',
    "            log(f'chunk: {len(chunk)} mols {len(chunk_jobs)} jobs {time.time() - t0:.1f}s '\n",
    '                f\'({meta["n_mols_done"]}/{meta["n_mols"]} mols done, {pred.n_pred} predictions)\')\n',
    '            chunk, chunk_jobs = [], set()\n',
    '            return True\n',
    '\n',
    '        stopped = False\n',
    '        for mol in mols:\n',
    "            mid = str(mol['mid'])\n",
    "            allc = list(dict.fromkeys(mol.get('cands') or []))\n",
    '            result[mid] = dict(cands=allc, spec={})\n',
    '            for c in allc:\n',
    '                if c not in okc:\n',
    '                    okc[c], why = G.smiles_ok(c)\n',
    '                    if not okc[c] and sum(1 for v in okc.values() if not v) <= 5:\n',
    "                        meta['errors'].append(f'smiles rejected ({why}): {str(c)[:100]}')\n",
    '            cands = [c for c in allc if okc[c]]\n',
    '            plans = []\n',
    "            for sp in mol.get('spectra') or []:\n",
    "                meta['n_spec'] += 1\n",
    '                try:\n',
    '                    pl = spec_plan(G, sp)\n',
    '                except Exception as e:\n',
    '                    meta[\'errors\'].append(f\'{mid} {sp.get("sid")}: plan {repr(e)[:200]}\')\n',
    '                    continue\n',
    '                if pl is None:\n',
    '                    continue\n',
    "                meta['n_spec_covered'] += 1\n",
    "                meta['n_spec_merged'] += int(bool(pl['mb']))\n",
    "                plans.append((sp['sid'], sp.get('tag'), pl))\n",
    '            if not cands or not plans:\n',
    "                meta['done'].append(mid)                         # nothing GLACIER could score: done, empty\n",
    '                continue\n',
    '            J = set()\n',
    '            for _, _, pl in plans:\n',
    '                J |= spec_jobs(G, pl, cands)\n',
    "            meta['n_jobs'] += len(J)\n",
    '            chunk.append((mid, cands, plans))\n',
    '            chunk_jobs |= J\n',
    '            if len(chunk_jobs) >= a.chunk_jobs and not flush():\n',
    '                stopped = True\n',
    '                break\n',
    '        if not stopped and not flush():\n',
    '            stopped = True\n',
    '        meta.update(n_pred=pred.n_pred, n_pred_fail=pred.n_fail, n_feat_fail=pred.n_feat_fail, t_pred=round(pred.t_pred, 2),\n',
    '                    pred_per_s=round(pred.n_pred / max(pred.t_pred, 1e-9), 2), bs_final=pred.bs,\n',
    "                    n_bad_smiles=sum(1 for v in okc.values() if not v), n_mols_done=len(meta['done']))\n",
    "        if not a.fake and meta.get('device', '').startswith('cuda'):\n",
    '            import torch\n',
    "            meta['max_mem_gb'] = round(torch.cuda.max_memory_allocated() / 2 ** 30, 2)\n",
    "        write('budget' if stopped else 'ok')\n",
    "        log('status', meta['status'], {k: meta[k] for k in ('n_mols', 'n_mols_done', 'n_spec_covered', 'n_spec_merged',\n",
    "                                                            'n_pred', 'n_pred_fail', 'pred_per_s')})\n",
    '        return 0\n',
    '    except BaseException as e:                                   # never crash the notebook: status error + partial out\n',
    "        meta['errors'].append(traceback.format_exc()[-3000:])\n",
    "        log('ERROR', repr(e)[:300])\n",
    '        try:\n',
    "            write('error')\n",
    '        except Exception:\n',
    '            return 2\n',
    '        return 0\n',
    '\n',
    '\n',
    "if __name__ == '__main__':\n",
    '    sys.exit(main())\n',
])


def _n1_mod(name, path):
    import importlib.util as _iu
    spec = _iu.spec_from_file_location(name, path)
    mod = _iu.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def n1_select(items_all, K):
    """N1 arm A (bench glb_core) on fuse.build_ice_input(..., covered_only=False) items -> (driver items, counts):
    the molecules WITHOUT a real positive [M+H]+ spectrum that have >= 1 [M+NH4]+ / [M-H2O+H]+ proxy, carrying only
    those proxies (bench tags NH4p / WLp, sid '<spectrum index>:<tag>'), in input (= GLACIER priority) order."""
    var, _ = K.build_variants(items_all)
    raw = {str(it['mid']): sum(1 for s in it.get('spectra') or [] if s.get('mode') == 'positive'
                               and s.get('adduct') in K.PROXY_TAG) for it in items_all}
    out = []
    st = dict(molecules=0, with_H=0, blind=0, blind_nh4_wl=0, n1=0, proxies=0, NH4p=0, WLp=0, proxy_empty=0)
    for it in var:
        fl = K.mol_flags(it)
        st['molecules'] += 1
        if fl['has_H']:                  # arm A: a real [M+H]+ exists -> the H1 run above scores this molecule
            st['with_H'] += 1
            continue
        st['blind'] += 1
        tags, _mode = K.arm_tags(N1_ARM, fl)
        sps = [s for s in it['spectra'] if s.get('tag') in tags]
        st['blind_nh4_wl'] += int(raw.get(it['mid'], 0) > 0)
        st['proxy_empty'] += raw.get(it['mid'], 0) - len(sps)
        if not sps or not it['cands']:
            continue
        st['n1'] += 1
        st['proxies'] += len(sps)
        for s in sps:
            st[s['tag']] += 1
        out.append(dict(mid=it['mid'], cands=list(it['cands']), spectra=sps))
    return out, st


def n1_run(items, budget):
    """The bench glb_driver.py in its own process (= gl_fuse.run_gl pattern) -> (out, meta, done). Never raises for a
    driver failure: the driver checkpoints after every chunk, so a budget stop / crash / kill keeps the molecules it
    finished; unreadable output -> nothing done."""
    fin, fout = os.path.join(N1_WORK, 'n1_in.json'), os.path.join(N1_WORK, 'n1_out.json')
    for f in (fout, fout + '.meta.json'):
        if os.path.exists(f):
            os.remove(f)
    with open(fin, 'w') as f:
        json.dump(items, f)
    cmd = [sys.executable, os.path.join(N1_WORK, 'glb_driver.py'), GL_PKG, fin, fout, '--device', N1_DEVICE,
           '--budget', str(float(budget)), '--site', N1_SITE, '--wheels', os.path.join(ICE_PKG, 'wheels')] + list(N1_DRIVER_EXTRA)
    t0 = time.time()
    try:
        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace',
                           timeout=float(budget) + 600)
        tail = (r.stdout or '')[-3000:]
    except subprocess.TimeoutExpired as e:
        tail = f'TIMEOUT after {time.time() - t0:.0f}s ' + (e.stdout.decode(errors='replace') if isinstance(e.stdout, bytes)
                                                            else str(e.stdout or ''))[-2000:]
    print(tail, flush=True)
    meta, out = {}, {}
    try:
        with open(fout + '.meta.json') as f:
            meta = json.load(f)
    except Exception as e:
        print('[N1] driver meta unreadable:', repr(e), flush=True)
    try:
        with open(fout) as f:
            out = json.load(f)
    except Exception as e:
        print('[N1] driver output unreadable:', repr(e), flush=True)
    done = set(meta.get('done') or []) if meta else {m for m, v in out.items() if isinstance(v, dict) and v.get('spec')}
    return out, meta, done


def n1_scores(out, done, items, K, mode=None):
    """Driver output -> {mid: {smiles: score | None}} = bench arm_scores of N1_ARM (mean over the molecule's proxy
    spectra of the per-spectrum similarity in the arm's CE mode, or `mode`); finished molecules only, candidate order
    as sent; non-finite / outside [0, 1] -> None; molecules without any score are left out."""
    tags, amode = K.arm_tags(N1_ARM, dict(has_H=False))
    res = {}
    for it in items:
        mid = str(it['mid'])
        mo = out.get(mid)
        if mid not in done or not isinstance(mo, dict):
            continue
        cands = [str(c) for c in mo.get('cands') or []]
        if cands != [str(c) for c in dict.fromkeys(it['cands'])]:
            print('[N1] candidate order changed -> molecule dropped', mid, flush=True)
            continue
        d = {}
        for c, x in zip(cands, K.arm_scores(mo, tags, mode or amode)):
            d[c] = float(x) if x is not None and math.isfinite(x) and -1e-9 <= x <= 1.0 + 1e-9 else None
        if any(x is not None for x in d.values()):
            res[mid] = d
    return res


def n1_synthetic_te(te_df, mids, K):
    """Commit-only forced exercise input: for every molecule in `mids` with a positive [M+H]+ spectrum a copy
    'n1x:<mid>' WITHOUT any [M+H]+: each positive [M+H]+ row -> an [M+NH4]+ row (precursor + 18.033823 - 1.007276,
    plus an injected peak AT the NH4 precursor that the proxy cut must drop); the first one also -> an [M-H2O+H]+ row
    (precursor - 18.010565, peaks unchanged); original NH4 / water-loss rows are left out; every other row (negative,
    Na, K, 2M, ...) is copied unchanged and must never be proxied.
    -> (DataFrame, {xmid: [(kind, original [M+H]+ precursor, mz list, intensity list) per row]})"""
    rows, ref = [], {}
    want = {str(m) for m in mids}
    for mid, sub in te_df.groupby(te_df.molecule_id.astype(str), sort=False):
        if mid not in want:
            continue
        recs = sub.to_dict('records')
        if not any(r.get('ionization_mode') == 'positive' and r.get('adduct') == '[M+H]+' for r in recs):
            continue
        xm, first, rf = f'n1x:{mid}', True, []
        for r in recs:
            r = dict(r, molecule_id=xm)
            if r.get('ionization_mode') == 'positive' and r.get('adduct') == '[M+H]+':
                prec = float(r['precursor_mz'])
                mz = [float(x) for x in r['ms2_mzs']]
                it = [float(x) for x in r['ms2_normalized_intensities']]
                if first:
                    rows.append(dict(r, adduct='[M-H2O+H]+', precursor_mz=prec - K.H2O, ms2_mzs=np.asarray(mz),
                                     ms2_normalized_intensities=np.asarray(it)))
                    rf.append(('WL', prec, mz, it))
                    first = False
                p2 = prec - K.PROTON + K.NH4_ADD
                rows.append(dict(r, adduct='[M+NH4]+', precursor_mz=p2, ms2_mzs=np.asarray(mz + [p2]),
                                 ms2_normalized_intensities=np.asarray(it + [1.0])))
                rf.append(('NH4', prec, mz, it))
            elif r.get('ionization_mode') == 'positive' and r.get('adduct') in K.PROXY_TAG:
                continue                 # an original NH4 / water-loss row: left out, so every proxy traces to a converted row
            else:
                rows.append(r)
                rf.append(('keep', None, None, None))
        ref[xm] = rf
    return pd.DataFrame(rows), ref


def n1_check_proxies(x_items, ref, K):
    """Commit asserts on the synthetic proxies: relabel, precursor back at the original [M+H]+ m/z (< 1e-6 Da), cut =
    exactly the original peaks below the bench cut (injected NH4 precursor peak gone), one proxy per converted row."""
    n = 0
    for it in x_items:
        rf = ref[it['mid']]
        for s in it['spectra']:
            j, tag = s['sid'].split(':')
            kind, prec0, mz0, it0 = rf[int(j)]
            assert (tag, kind) in (('NH4p', 'NH4'), ('WLp', 'WL')), (it['mid'], s['sid'], kind)
            assert s['adduct'] == '[M+H]+' and s['mode'] == 'positive' and s['n1_src'] == (
                '[M+NH4]+' if kind == 'NH4' else '[M-H2O+H]+'), (it['mid'], s['sid'])
            assert abs(float(s['prec']) - prec0) < 1e-6, (it['mid'], s['sid'], s['prec'], prec0)
            if kind == 'NH4':
                p2 = prec0 - K.PROTON + K.NH4_ADD
                cut = (p2 - K.NH4_ADD) + K.PROTON - K.CUT_DA
            else:
                cut = (prec0 - K.H2O) - K.CUT_DA
            keep = [(m, i) for m, i in zip(mz0, it0) if m < cut]
            assert s['mz'] == [m for m, _ in keep] and s['it'] == [i for _, i in keep], (it['mid'], s['sid'])
            assert max(s['mz']) < float(s['prec']) - K.CUT_DA + 1e-6, (it['mid'], s['sid'])
            n += 1
        assert {s['tag'] for s in it['spectra']} <= {'NH4p', 'WLp'}, it['mid']
    return n


def _n1_spearman(a, b):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    ra, rb = np.argsort(np.argsort(a)), np.argsort(np.argsort(b))
    return float(np.corrcoef(ra, rb)[0, 1]) if ra.std() > 0 and rb.std() > 0 else float('nan')


def n1_ice_finite(ice):
    """Number of finite ICEBERG scores in one molecule's ICE dict (None / NaN / inf / non-numeric = no evidence; the
    per-value test of fuse.rerank and of E006's _finite_coverage)."""
    import math as _m
    n = 0
    for v in (ice.values() if isinstance(ice, dict) else ()):
        try:
            n += v is not None and _m.isfinite(float(v))
        except (TypeError, ValueError):
            pass
    return n


def n1_complement(scores, ice_scores):
    """k2 variant A: drop the N1 GL scores of the molecules whose ICE dict (ice_scores[str(mid)]) holds a finite
    score -> (kept {mid: same score dict}, dropped {mid}). Kept dicts are passed through unchanged."""
    ice_scores = ice_scores if isinstance(ice_scores, dict) else {}
    drop = {m for m in scores if n1_ice_finite(ice_scores.get(str(m))) > 0}
    return {m: v for m, v in scores.items() if m not in drop}, drop


try:
    import math
    _n1_h1_gl = isinstance(GL_SCORES, dict) and any(isinstance(g, dict) and any(v is not None for v in g.values())
                                                  for g in GL_SCORES.values())
    if not _n1_h1_gl:
        raise RuntimeError('H1 GLACIER produced no scores (GL failed / not run) -> N1 not attempted')
    os.makedirs(N1_WORK, exist_ok=True)
    for _n1_fn, _n1_s, _n1_h in (('glb_core.py', N1_CORE_SRC, N1_CORE_SHA256), ('glb_driver.py', N1_DRIVER_SRC, N1_DRIVER_SHA256)):
        assert hashlib.sha256(_n1_s.encode('utf-8')).hexdigest() == _n1_h, ('embedded bench source changed', _n1_fn)
        with open(os.path.join(N1_WORK, _n1_fn), 'w', encoding='utf-8') as _n1_f:
            _n1_f.write(_n1_s)
    N1K = _n1_mod('glb_core', os.path.join(N1_WORK, 'glb_core.py'))
    N1DR = _n1_mod('glb_driver', os.path.join(N1_WORK, 'glb_driver.py'))
    print('[N1]', N1K.selftest(), flush=True)
    _n1_G = N1DR.load_gl(GL_PKG)            # attached gl_runner.py as a module (numpy / numba only; no torch here)
    print('[N1]', N1DR.selftest(_n1_G), flush=True)
    _n1_sg = sha256(os.path.join(GL_PKG, 'gl_runner.py'))
    print(f'[N1] code: glb_core {N1_CORE_SHA256[:12]} glb_driver {N1_DRIVER_SHA256[:12]} (= kdev_glbench_n1) | '
          f'gl_runner.py {_n1_sg[:12]} matches the B1/bench MANIFEST: {_n1_sg == N1_GL_RUNNER_SHA256} | arm {N1_ARM} '
          f'(tags, mode) {N1K.arm_tags(N1_ARM, dict(has_H=False))} | budget {N1_BUDGET:.0f}s', flush=True)
    _n1_items_all = ice_fuse.build_ice_input(te, cands, covered_only=False)
    _n1_items, N1_STATS = n1_select(_n1_items_all, N1K)
    print(f"[N1] molecules: GL/ICE input {N1_STATS['molecules']} | with real [M+H]+ {N1_STATS['with_H']} (H1 run, "
          f"unchanged) | blind {N1_STATS['blind']} (with NH4/WL spectra {N1_STATS['blind_nh4_wl']}) | N1 molecules "
          f"{N1_STATS['n1']} | proxies built {N1_STATS['proxies']} (NH4p {N1_STATS['NH4p']}, WLp {N1_STATS['WLp']}; "
          f"empty after the cut {N1_STATS['proxy_empty']})", flush=True)
    _n1_x_items, _n1_x_ref = [], {}
    if not IS_RERUN:                     # forced commit exercise: synthetic blind copies through the same code
        _n1_tx, _n1_x_ref = n1_synthetic_te(te, list(cands), N1K)
        _n1_x_all = ice_fuse.build_ice_input(_n1_tx, {f'n1x:{m}': c for m, c in cands.items() if f'n1x:{m}' in _n1_x_ref},
                                          covered_only=False)
        _n1_x_items, N1X_STATS = n1_select(_n1_x_all, N1K)
        N1X_STATS['proxy_checked'] = n1_check_proxies(_n1_x_items, _n1_x_ref, N1K)
        assert N1X_STATS['blind'] == N1X_STATS['molecules'] == len(_n1_x_ref) >= 1 and N1X_STATS['with_H'] == 0, N1X_STATS
        assert N1X_STATS['n1'] >= 1 and N1X_STATS['proxy_checked'] == N1X_STATS['proxies'] >= N1X_STATS['n1'], N1X_STATS
        assert N1X_STATS['proxies'] + N1X_STATS['proxy_empty'] == sum(k != 'keep' for v in _n1_x_ref.values() for k, *_ in v)
        # arm-B guard: a molecule WITH a real [M+H]+ plus NH4 / water-loss spectra must NOT become an N1 molecule
        _n1_src = sorted(_n1_x_ref)[0].split(':', 1)[1]
        _n1_gdf = pd.concat([te[te.molecule_id.astype(str) == _n1_src], _n1_tx[_n1_tx.molecule_id == f'n1x:{_n1_src}']],
                            ignore_index=True).assign(molecule_id=f'n1b:{_n1_src}')
        _n1_gi, _n1_gs = n1_select(ice_fuse.build_ice_input(_n1_gdf, {f'n1b:{_n1_src}': {str(m): c for m, c in cands.items()}[_n1_src]},
                                                           covered_only=False), N1K)
        assert _n1_gi == [] and _n1_gs['with_H'] == 1 and _n1_gs['n1'] == 0, ('arm-B guard: [M+H]+ molecule selected', _n1_gs)
        N1X_STATS['armB_guard'] = 'ok'
        N1X_MIDS = {it['mid'] for it in _n1_x_items}
    _n1_run = _n1_items + _n1_x_items
    _n1_exp_merged = sum(bool(N1DR.merged_buckets(_n1_G, s)) for it in _n1_run for s in it['spectra'])
    _n1_out, N1_DRV, _n1_done = n1_run(_n1_run, N1_BUDGET) if _n1_run else ({}, {'status': 'nothing to score'}, set())
    print(f"[N1] driver: status {N1_DRV.get('status')} | molecules sent {len(_n1_run)} (real {len(_n1_items)}, synthetic "
          f"{len(_n1_x_items)}) done {len(_n1_done)} | spectra covered {N1_DRV.get('n_spec_covered')} (merged-CE "
          f"{N1_DRV.get('n_spec_merged')}, expected {_n1_exp_merged}) | predictions {N1_DRV.get('n_pred')} (fail "
          f"{N1_DRV.get('n_pred_fail')}) | {N1_DRV.get('seconds')}s | errors {(N1_DRV.get('errors') or [])[:2]}", flush=True)
    _n1_sc = n1_scores(_n1_out, _n1_done, _n1_run, N1K)
    _n1_real = {m: v for m, v in _n1_sc.items() if not m.startswith('n1x:')}
    _n1_clash = sorted(m for m in _n1_real if any(x is not None for x in (GL_SCORES.get(m) or {}).values()))
    if not IS_RERUN:
        assert not _n1_clash, ('N1 molecule already scored by the H1 run', _n1_clash[:5])
        N1X_GL = {m: v for m, v in _n1_sc.items() if m.startswith('n1x:')}
        assert N1_DRV.get('status') == 'ok' and N1X_MIDS <= _n1_done, (N1_DRV.get('status'), sorted(N1X_MIDS - _n1_done)[:5])
        assert len(N1X_GL) >= 1 and N1_DRV.get('n_spec_merged') == _n1_exp_merged, (len(N1X_GL), N1_DRV.get('n_spec_merged'))
        _n1_nb1 = _n1_nbm = 0
        for _n1_m in N1X_MIDS:              # single-CE spectra: merged-CE mode == mean (production gl_runner) exactly;
            for _n1_sp in (_n1_out[_n1_m].get('spec') or {}).values():     # merged-CE ones: the I14b path acts
                if _n1_sp.get('nb') == 1:
                    assert _n1_sp['mean'] == _n1_sp['merged'], _n1_m
                    _n1_nb1 += 1
                else:
                    _n1_nbm += int(_n1_sp['mean'] != _n1_sp['merged'])
        # the dummy commit molecules carry 20,40,60 eV [M+H]+ spectra -> merged-CE proxies (8 in the local dry run): the
        # per-bucket prediction + merge() must run on them with the real model and change their scores vs mean CE
        assert _n1_exp_merged >= 1 and _n1_nbm >= 1, ('merged-CE path not exercised on the commit', _n1_exp_merged, _n1_nbm)
        _n1_rho = []                        # proxies (= the [M+H]+ peaks, cut) vs the real H1 GL scores of the source
        _n1_mean = n1_scores(_n1_out, _n1_done, _n1_x_items, N1K, mode='mean')
        for _n1_m, _n1_d in _n1_mean.items():
            _n1_h = GL_SCORES.get(_n1_m.split(':', 1)[1]) or {}
            _n1_p = [(v, _n1_h.get(c)) for c, v in _n1_d.items() if v is not None and _n1_h.get(c) is not None]
            if len(_n1_p) >= 4:
                _n1_r = _n1_spearman([a for a, _ in _n1_p], [b for _, b in _n1_p])
                if math.isfinite(_n1_r):
                    _n1_rho.append(_n1_r)
        N1X_STATS.update(gl_scored=len(N1X_GL), single_ce_identical=_n1_nb1, merged_changed=_n1_nbm, rho_n=len(_n1_rho),
                         rho_median=float(np.median(_n1_rho)) if _n1_rho else float('nan'))
        print(f"[N1X] synthetic exercise: molecules {N1X_STATS['molecules']} | N1 molecules {N1X_STATS['n1']} | proxies "
              f"built {N1X_STATS['proxies']} (NH4p {N1X_STATS['NH4p']}, WLp {N1X_STATS['WLp']}; empty "
              f"{N1X_STATS['proxy_empty']}) | proxy checks OK {N1X_STATS['proxy_checked']} | arm-B guard "
              f"{N1X_STATS['armB_guard']} | merged-CE spectra "
              f"{N1_DRV.get('n_spec_merged')} (expected {_n1_exp_merged}; merged != mean CE {_n1_nbm}) | single-CE "
              f"merged==mean {_n1_nb1} | GL-scored "
              f"synthetic molecules {len(N1X_GL)} | proxy (mean CE) vs H1 GL Spearman median "
              f"{N1X_STATS['rho_median']:.3f} over {len(_n1_rho)}", flush=True)
        assert not _n1_rho or N1X_STATS['rho_median'] >= 0.3, ('proxy scores do not track the H1 GL scores', _n1_rho)
    else:
        _n1_real = {m: v for m, v in _n1_real.items() if m not in _n1_clash}
    if N1_COMPLEMENT:                    # k2 variant A: E006's ICE fallback keeps the blind molecules ICEBERG covers
        _n1_n0 = len(_n1_real)
        _n1_real, N1_ICE_DROPPED = n1_complement(_n1_real, ICE_SCORES)
        if not IS_RERUN:                 # forced commit exercise of the filter on THIS commit's real ICE output: a
            #                              synthetic copy 'n1x:<mid>' stands for a blind molecule with <mid>'s ICE dict
            _n1_xi = {m: (ICE_SCORES.get(m.split(':', 1)[1]) if isinstance(ICE_SCORES, dict) else None) for m in N1X_GL}
            _n1_xk, _n1_xd = n1_complement(N1X_GL, _n1_xi)
            assert _n1_xd == {m for m, v in _n1_xi.items() if n1_ice_finite(v) > 0}, _n1_xd
            assert set(_n1_xk) == set(N1X_GL) - _n1_xd and all(_n1_xk[m] is N1X_GL[m] for m in _n1_xk)
            assert _n1_xd, ('complement filter not exercised: no synthetic N1 molecule whose source has a finite ICE '
                            'score', len(N1X_GL), sum(n1_ice_finite(v) for v in _n1_xi.values()))
            _n1_t = n1_complement({'f': {}, 'n': {}, 'x': {}, 'b': {}, 'm': {}, 'e': {}},
                                  {'f': {'p': 0.5, 'q': None}, 'n': {'p': None}, 'x': {'p': float('nan'), 'q': float('inf')},
                                   'b': {'p': 'bad', 'q': [1]}, 'e': {}})
            assert set(_n1_t[0]) == {'n', 'x', 'b', 'm', 'e'} and _n1_t[1] == {'f'}, _n1_t
            assert n1_complement({}, ICE_SCORES) == ({}, set()) and n1_complement({'f': {}}, None) == ({'f': {}}, set())
            N1X_STATS['complement'] = dict(synthetic=len(N1X_GL), dropped=len(_n1_xd), kept=len(_n1_xk))
            print(f"[N1X] complement filter exercise: synthetic N1 molecules with GL {len(N1X_GL)} | source ICE dict has a "
                  f"finite score -> dropped {len(_n1_xd)} | kept {len(_n1_xk)} | unit cases OK", flush=True)
        print(f"[N1] complement (k2 variant A): N1 molecules with GL scores {_n1_n0} | dropped (finite ICE score, E006 ICE "
              f"fallback kept) {len(N1_ICE_DROPPED)} | kept {len(_n1_real)}", flush=True)
    _n1_health = (f"N1 OK: {len(_n1_real)} of {len(_n1_items)} N1 molecules GL-scored "
                  f"({sum(sum(v is not None for v in d.values()) for d in _n1_real.values())} candidates), driver "
                  f"{N1_DRV.get('status')}, clashes dropped {len(_n1_clash)}")
    N1_MIDS, N1_GL_MIDS = {str(it['mid']) for it in _n1_items}, set(_n1_real)
    GL_SCORES.update(_n1_real)           # last statement: N1 scores exist only if everything above succeeded
    N1_HEALTH = _n1_health
except Exception as e:
    for _n1_m in list(N1_GL_MIDS):
        GL_SCORES.pop(_n1_m, None)
    N1_MIDS, N1_GL_MIDS = set(), set()
    N1_HEALTH = f'N1 FAILED: {e!r}'
    print('[N1] FAILED -> H1 behaviour (no N1 GL scores):', repr(e)[:600], flush=True)
    if not IS_RERUN:
        raise
print(f'[N1] GL-scored N1 molecules: {len(N1_GL_MIDS)} of {len(N1_MIDS)} | health: {N1_HEALTH} | '
      f'{time.time()-T0:.0f}s', flush=True)
# Counts distinguish selected proxies from completed, retained forward scores.
N1_CHALLENGER_COVERAGE = dict(
    complement=bool(N1_COMPLEMENT), eligible_molecules=len(N1_MIDS),
    proxy_covered_molecules=len(N1_GL_MIDS),
    proxy_covered_with_finite_ice=sum(n1_ice_finite(ICE_SCORES.get(str(m), {})) > 0
                                       for m in N1_GL_MIDS),
    real_h_molecules_skipped=int(N1_STATS.get('with_H', 0)), health=N1_HEALTH)
print('N1_CHALLENGER_COVERAGE', json.dumps(N1_CHALLENGER_COVERAGE), flush=True)
# ---- /N1A_MCE ---------------------------------------------------------------------------------------------------
def gl_of(m):
    g = GL_SCORES.get(str(m), {}) if isinstance(GL_SCORES, dict) else {}
    return g if isinstance(g, dict) and any(v is not None for v in g.values()) else {}
def gl_rerank(order0, smis, keys, scs, forms, ice, gl, top_n, tag=''):
    # z(scs) + ICE_LAM z(ice) + GL_LAM z(gl) in the same slots; no GL scores / any failure -> order0 (= v4g order)
    if not gl:
        return order0
    try:
        o = gl_fuse.rerank_multi(smis, keys, scs, forms, [ice, gl], [ICE_LAM, GL_LAM], top_n=top_n)
    except Exception as e:
        print('gl rerank failed', tag, repr(e)); return order0
    for k, v in (('molecules', 1), ('changed_vs_ice_top25', int(o[:25] != order0[:25])), ('changed_vs_ice_top1', int(o[:1] != order0[:1]))):
        gl_stats[tag + k] = gl_stats.get(tag + k, 0) + v
    return o
for m in list(BASE):
    smis, keys, lib_max, scs, forms = BASE[m]
    ice = ICE_SCORES.get(str(m), {}) if ICE_SCORES else {}
    gl = gl_of(m)
    if BASE_ICE and ice and any(v is not None for v in ice.values()):
        try:
            order = ice_fuse.rerank(smis, keys, scs, forms, ice, lam=ICE_LAM, top_n=TOPN)
            order = gl_rerank(order, smis, keys, scs, forms, ice, gl, TOPN)
            s2 = [smis[i] for i in order]; k2 = [keys[i] for i in order]
            ice_stats['molecules'] += 1
            ice_stats['changed_top25'] += int(s2[:25] != smis[:25]); ice_stats['changed_top1'] += int(bool(s2) and s2[0] != smis[0])
            smis, keys = s2, k2
        except Exception as e:
            print('rerank failed', m, repr(e))
    BASE[m] = (smis[:25], keys[:25], lib_max)
    p = PC.get(m) if (ICE_PC and isinstance(PC, dict)) else None
    if p and p.get('pc_form') and ice:
        try:                             # re-order the PubChem list by z(f.z) + lam z(ice) inside same-formula groups;
            o = ice_fuse.rerank(p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, lam=ICE_LAM, top_n=25)
            o = gl_rerank(o, p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, gl, 25, 'pc_')
            fz0 = p['pc_fz'][0]          # the gate keeps using the ORIGINAL top f.z (rel), so gate decisions are unchanged
            p['pc'] = [p['pc'][i] for i in o]; p['pc_keys'] = [p['pc_keys'][i] for i in o]
            p['pc_fz'] = [fz0] + [p['pc_fz'][i] for i in o][1:]
            ice_stats['pc_changed'] = ice_stats.get('pc_changed', 0) + int(o[:1] != [0])
        except Exception as e:
            print('pc rerank failed', m, repr(e))
print('ICE rerank stats', ice_stats, f'{time.time()-T0:.0f}s')
print('GL rerank stats', gl_stats, f'{time.time()-T0:.0f}s')


In [ ]:
'''## 8. Gated PubChem merge

PubChem-only candidates enter the main list only when there is no strong library match (`lib_max < 0.9`). When the PubChem fingerprint fit beats the best pool candidate by a wide margin, they take slots 2, 4, 6, 8 and 10. Otherwise they take slots 4, 8, 12, 16 and 20.
'''


In [ ]:
'''gated PubChem merge (v3.6, unchanged) -> submission.csv'''
def merge(base, base_keys, pc, pc_keys, slots, n=25):
    bk = set(base_keys)
    pcs = [s for s, k in zip(pc, pc_keys) if k not in bk]
    out, bi, pj = [], 0, 0
    for pos in range(1, n + 1):
        if pos in slots and pj < len(pcs):
            out.append(pcs[pj]); pj += 1
        elif bi < len(base):
            out.append(base[bi]); bi += 1
        elif pj < len(pcs):
            out.append(pcs[pj]); pj += 1
    return out
'''---- CLAW PROMOTION RULE (b417) --------------------------------------------------'''
'''The shipped gate only INTERLEAVES PubChem-only structures into fixed slots, so the'''
'''main list always keeps rank 1. When the best PubChem proposal is a strong outlier --'''
'''S = z(f.z over the pass-2 set) + 0.25*pop, carried out of the worker -- AND is well'''
'''documented (pop >= 5), let it lead; everything else keeps its usual slot.'''
'''NOTE: S is the Z-SCORED quantity (median ~6). The raw p['pc_fz'][0] (median -1437)'''
'''must never be compared to S_TAU; that would fire on ~71% of molecules.'''
'''EVIDENCE: b416 (promotion, ICEBERG broken) = 0.417 vs base415 (ICE broken, no'''
'''promotion) 0.374 and b415c (ICE working, no promotion) 0.393. The promotion overcame'''
'''losing ICE, so its marginal value is >= +0.024 (est. ~+0.037 using the notebook's own'''
'''ICE +0.013 / GL +0.006 measurements). b417 isolates it on the WORKING-ICE base.'''
S_TAU, POP_TAU = 6.0, 5.0


def promote(base, base_keys, pc, pc_keys, slots, n=25):
    """Confident PubChem proposal leads the usual list; every other entry keeps its slot."""
    usual = merge(base, base_keys, pc, pc_keys, slots, n=n)
    if not pc:
        return usual
    top = pc[0]
    return ([top] + [x for x in usual if x != top])[:n]


'''self-test of both pure functions (this also runs in the cheap smoke pass)'''
_b, _bk = ['A', 'B', 'C'], ['ka', 'kb', 'kc']
_p, _pk = ['X', 'Y'], ['kx', 'ky']
assert merge(_b, _bk, _p, _pk, [2, 4]) == ['A', 'X', 'B', 'Y', 'C'], merge(_b, _bk, _p, _pk, [2, 4])
assert promote(_b, _bk, _p, _pk, [2, 4]) == ['X', 'A', 'B', 'Y', 'C'], promote(_b, _bk, _p, _pk, [2, 4])
assert merge(_b, _bk, _p, _pk, [2, 4])[0] == 'A', 'shipped gate must keep rank 1'
assert merge(_b, _bk, [], [], [2, 4]) == ['A', 'B', 'C']
assert promote(_b, _bk, [], [], [2, 4]) == ['A', 'B', 'C']
assert promote(_b, _bk, _p, _pk, [2, 4])[:1] == ['X'] and len(promote(_b, _bk, _p, _pk, [2, 4])) == 5
print('merge/promote self-test OK')

def candidate_promotion_evidence(p):
    """Promotion evidence of the current PubChem leader, looked up by its retrieval key (sanpier, LB 0.465)."""
    key = p['pc_keys'][0]
    evidence = p.get('promotion_evidence', {}).get(key)
    assert evidence is not None, 'Missing candidate promotion evidence: ' + str(key)
    assert evidence['smiles'] == p['pc'][0], 'PubChem identity/representative mismatch'
    S, pop, raw = (float(evidence[x]) for x in ('S', 'pop', 'raw_fz'))
    assert all(np.isfinite(v) for v in (S, pop, raw)), 'Nonfinite promotion evidence'
    rel = raw - p['best_pool_fz'] if np.isfinite(p['best_pool_fz']) else 1e9
    return S, pop, rel


promotion_changed = 0
rows, stats = [], dict(untouched=0, gentle=0, aggressive=0, no_pc=0, promoted=0)
for mid in te.molecule_id.unique():
    smis, keys, lib_max = BASE.get(mid, ([], [], 0.0))
    p = PC.get(mid)
    if p is None or not p['pc']:
        stats['no_pc'] += 1; final = smis
    elif lib_max >= LIB_TAU:
        stats['untouched'] += 1; final = smis
    else:
        rel = (p['fz_top'] if p.get('fz_top') is not None else p['pc_fz'][0]) - p['best_pool_fz'] if np.isfinite(p['best_pool_fz']) else 1e9
        slots = SLOTS_AGG if rel > REL_TH else SLOTS_GENTLE
        stats['aggressive' if rel > REL_TH else 'gentle'] += 1
        old_S = p.get('S'); old_pop = float(p.get('top_pop') or 0.0)
        old_promote = bool((old_S is not None and float(old_S) > S_TAU and old_pop >= POP_TAU) or rel > 150.0)
        # Promotion uses the evidence of the candidate that leads the list (sanpier, LB 0.465); interleaving slots keep the channel margin.
        S, pop_top, candidate_rel = candidate_promotion_evidence(p)
        new_promote = bool((S > S_TAU and pop_top >= POP_TAU) or candidate_rel > 150.0)
        promotion_changed += old_promote != new_promote
        if new_promote:
            stats['promoted'] += 1
            final = promote(smis, keys, p['pc'], p['pc_keys'], slots)
        else:
            final = merge(smis, keys, p['pc'], p['pc_keys'], slots)
    if not final: final = ['CCO']
    rows.append((mid, ';'.join(final[:25])))
print('merge stats', stats)
print('CANDIDATE PROMOTION: decisions changed', promotion_changed)
sub = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
samp = pd.read_csv(os.path.join(COMP, 'sample_submission.csv'))
sub = samp[['molecule_id']].merge(sub, on='molecule_id', how='left')
sub['smiles'] = sub['smiles'].fillna('CCO')
assert len(sub) == len(samp) and sub.molecule_id.is_unique and sub.smiles.notna().all()
assert sub.smiles.map(lambda s: len(s.split(';'))).max() <= 25
sub.to_csv('submission.csv', index=False)
print('CLAW STAGE HEALTH |', CLAW_ICE_HEALTH, '|', CLAW_GL_HEALTH, flush=True)
try:
    _ice_n = json.load(open('/kaggle/working/ice_work/ice_out.json.meta.json')).get('n_mols_scored', 0)
    _gl_n = json.load(open('/kaggle/working/gl_work/gl_out.json.meta.json')).get('n_mols_scored', 0)
except Exception:
    _ice_n = _gl_n = -1
print(f'CLAW STAGE SCORED | ice={_ice_n} gl={_gl_n}', flush=True)
json.dump(dict(secs=round(time.time() - T0), merge=stats, n_errors=n_err,
               ice_health=CLAW_ICE_HEALTH, gl_health=CLAW_GL_HEALTH,
               ice_n_scored=_ice_n, gl_n_scored=_gl_n,
               families=MAN['families'],
               gpu=torch.cuda.get_device_name(0) if dev == 'cuda' else None), open('run_manifest.json', 'w'), indent=1)
print('done', sub.shape, f'{time.time()-T0:.0f}s')
sub.head()


In [ ]:
'''## 9. Two-engine fusion + final simulator re-rank

The two lists are combined by weighted reciprocal-rank fusion. A candidate's score is $\\frac{1}{3 + r_{\\text{main}}} + \\frac{0.6}{3 + r_{\\text{engine 2}}}$, and the top 40 are kept. ICEBERG and GLACIER then re-rank same-formula groups of the fused list one last time, and the top 25 become the submission.
'''


In [ ]:
ALPHA, KRR = 0.6, 3.0
CELL19_ICE = False       # b421: cell-19 ICE contribution EXPLICITLY off.
'''Reproduces BY DESIGN what b416 got by accident: when ICE errored it still wrote a'''
'''shell dict of all-None values, cell-19's `if ice:` gate passed, and ICE's column then'''
'''contributed nothing (covered count 0) -- leaving GLACIER alone. b420 (ICE real, both'''
'''applied) scored 0.402; b416 (GLACIER only) scored 0.417. This isolates that one'''
'''arithmetic difference on the healthy base.'''
from rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF2
from rdkit import Chem as _Ch2
def _form2(s):
    try:
        return _CMF2(_Ch2.MolFromSmiles(s))
    except Exception:
        return s
def fuse2(v_smis, e_smis, e_keys, n=40):
    sc, smi_of = {}, {}
    for r, s in enumerate(v_smis, 1):
        k = chem.score_key(s) or s
        sc[k] = sc.get(k, 0.0) + 1.0 / (KRR + r); smi_of.setdefault(k, s)
    for r, (s, k) in enumerate(zip(e_smis, e_keys), 1):
        if not k:
            continue
        sc[k] = sc.get(k, 0.0) + ALPHA / (KRR + r); smi_of.setdefault(k, s)
    order = sorted(sc, key=lambda k: -sc[k])[:n]
    return [smi_of[k] for k in order], order, [sc[k] for k in order]

PREFUSION_LISTS = {str(r.molecule_id): str(r.smiles).split(';') for r in pd.read_csv('submission.csv').itertuples(index=False)}
FUSION_OVERFLOW = {}
if ENG:
    v4 = pd.read_csv('submission.csv')
    changed = n_ice = 0; out = []
    for mid, s in zip(v4.molecule_id, v4.smiles):
        vs = [x for x in s.split(';') if x and x != 'CCO']
        e = ENG.get(str(mid))
        if e and e['smiles']:
            fsm, fk, fsc = fuse2(vs, e['smiles'], e['keys'])
            ice = ICE_SCORES.get(str(mid), {}) if isinstance(ICE_SCORES, dict) else {}
            if ice:
                try:
                    o = ice_fuse.rerank(fsm, fk, fsc, [_form2(x) for x in fsm], (ice if CELL19_ICE else {}), lam=ICE_LAM, top_n=len(fsm))
                    _g = gl_of(mid)
                    if _g:
                        o = gl_fuse.rerank_multi(fsm, fk, fsc, [_form2(x) for x in fsm], [(ice if CELL19_ICE else {}), _g], [ICE_LAM, GL_LAM], top_n=len(fsm))
                    n_ice += int(o[:25] != list(range(min(25, len(o)))))
                    fsm = [fsm[i] for i in o]
                except Exception as ex:
                    print('fused ICE rerank failed', mid, repr(ex))
            changed += int(bool(vs) and fsm[:1] != vs[:1])
            FUSION_OVERFLOW[str(mid)] = list(fsm[25:])
            vs = fsm[:25]
        out.append((mid, ';'.join(vs[:25]) if vs else 'CCO'))
    fs = pd.DataFrame(out, columns=['molecule_id', 'smiles'])
    assert len(fs) == len(v4) and fs.molecule_id.is_unique and fs.smiles.notna().all()
    assert fs.smiles.map(lambda s: len(s.split(';'))).max() <= 25
    fs.to_csv('submission.csv', index=False)


print('V1 inference complete; applying separately audited final pass next.')


In [ ]:
'''## Final metric audit

The final pass cannot demote any existing metric-equivalent answer: it retains the first
occurrence of every valid identity in order, then appends new identities up to 25 candidates.
Runtime assertions check this for every active molecule. This guarantee applies to the
same predictions and current metric; it does not predict the score of a fresh inference run.

Outputs: `submission.csv`, `baseline_submission.csv`, `candidate_audit.csv`,
`candidate_audit.json`, and stage health in `run_manifest.json`.
'''


In [ ]:
from functools import lru_cache
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog('rdApp.*')
_tautomer_canonicalizer = rdMolStandardize.TautomerEnumerator()
def canonicalize_smiles_to_ik14(smiles: str) -> str:
    """Canonicalize SMILES to InChIKey14, or None if parsing fails.

    Stereochemistry is removed *before* the tautomer canonicalization, not left to fall out of
    the InChIKey14 truncation afterwards. Truncating to InChIKey14 discards the stereo block, so
    it looks like it should already make stereochemistry free -- but the canonicalization runs
    first, and `TautomerEnumerator` will not move a double bond whose position a neighbouring
    stereocentre constrains. The same skeleton drawn with and without that stereocentre can
    therefore land on two different canonical tautomers, and different tautomers are different
    connectivity, which is the part of the InChIKey14 that survives truncation. Stripping first
    makes the canonical form depend only on the skeleton, which is what the competition promises.

    Both the answer key and the submission go through this function, so the two sides can never
    be treated asymmetrically.
    """
    if not isinstance(smiles, str):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        Chem.RemoveStereochemistry(mol)
        canonical_mol = _tautomer_canonicalizer.Canonicalize(mol)
        inchikey = Chem.MolToInchiKey(canonical_mol)
        return inchikey[:14]
    # RDKit's MolSanitizeException family subclasses ValueError; RuntimeError covers the C++
    # layer and TypeError covers Boost.Python argument errors. One participant's malformed
    # SMILES must never take the whole scoring run down, so a failure here is just a non-match.
    except (ValueError, RuntimeError, TypeError):
        return None

def heavy_atom_composition(smiles: str):
    """Heavy-atom element counts (excluding H, ignoring charge) as a hashable key, or None
    if parsing fails.

    Cheap (no tautomer enumeration) -- used to gate the expensive canonicalization. Anything
    that shares an InChIKey14 with an accepted structure has the same heavy-atom skeleton, so
    a heavy-atom-composition mismatch lets us skip canonicalization with no change to any
    score. We deliberately do NOT gate on full molecular formula: protomers / charge variants
    (e.g. a zwitterion C(=O)[O-]...[N+] vs its neutral-acid protomer C(=O)O...[N+]) share an
    InChIKey14 but differ in H count and charge, so a formula gate would wrongly reject them.
    """
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    if mol is None:
        return None
    counts = {}
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        if sym != 'H':
            counts[sym] = counts.get(sym, 0) + 1
    return tuple(sorted(counts.items()))

@lru_cache(maxsize=150000)
def metric_identity(smiles):
    key = canonicalize_smiles_to_ik14(smiles)
    return (key, heavy_atom_composition(smiles)) if key else None

def safe_extend(original, extra, limit=25):
    """Preserve first valid metric identities; fill only unoccupied ranks."""
    out, seen = [], set()
    for smi in list(original) + list(extra):
        smi = str(smi).strip()
        identity = metric_identity(smi)
        if identity is not None and identity not in seen:
            seen.add(identity)
            out.append(smi)
            if len(out) == limit:
                break
    return out

def first_ranks(smiles):
    ranks = {}
    for rank, smi in enumerate(smiles, 1):
        identity = metric_identity(smi)
        if identity is not None:
            ranks.setdefault(identity, rank)
    return ranks

def assert_monotone(original, revised):
    old, new = first_ranks(original), first_ranks(revised)
    assert all(identity in new and new[identity] <= rank for identity, rank in old.items())


In [ ]:
METRIC_SOURCE_SHA256 = 'c61d163f1bb768313fd0863822952f035f343924f37e3f5e4b5883ccef7e6201'
'''Final pass: no change to any V1 model, fusion weight, promotion gate or simulator.'''
import collections
import hashlib
from pathlib import Path
assert rdkit.__version__ == '2026.03.3'
assert n_err == 0, f'Base inference errors: {n_err}'
_before = pd.read_csv('submission.csv')
_before.to_csv('baseline_submission.csv', index=False)
_active = set(te.molecule_id.astype(str))
assert _active <= set(map(str, ENG)), 'Engine 2 did not cover all active molecules'
_stage_meta = {}
for _stage, _path in [('ice', 'ice_work/ice_out.json.meta.json'), ('glacier', 'gl_work/gl_out.json.meta.json')]:
    if Path(_path).is_file():
        _stage_meta[_stage] = json.loads(Path(_path).read_text())
    else:
        _stage_meta[_stage] = {'status': 'missing'}
assert _stage_meta['ice'].get('status') == 'ok', _stage_meta['ice']
if (te.adduct == '[M+H]+').any():
    assert _stage_meta['glacier'].get('status') == 'ok', _stage_meta['glacier']

_dm_smiles = _dm_mass = _dm_order = None
_dm_paths = sorted(glob.glob('/kaggle/input/**/dreams_np_mass.npy', recursive=True))
_dreams_used = 0
_audit_rows, _out = [], []
_targets = te.groupby('molecule_id').nm.median().to_dict()
for _row in _before.itertuples(index=False):
    _mid = str(_row.molecule_id)
    _original = [s.strip() for s in str(_row.smiles).split(';') if s.strip()]
    if _mid not in _active:  # Visible smoke placeholders; hidden rerun processes every molecule.
        _out.append((_row.molecule_id, _row.smiles))
        continue
    _ranks = first_ranks(_original)
    _clean = safe_extend(_original, [])
    _extras = list(FUSION_OVERFLOW.get(_mid, [])) + list(BASE_TAIL_CANDIDATES.get(_mid, []))
    _engine = ENG.get(_mid, {})
    _extras.extend(_engine.get('smiles', []))
    _pc = PC.get(_row.molecule_id, PC.get(_mid, {}))
    _extras.extend(_pc.get('pc', []))
    _revised = safe_extend(_clean, _extras)
    _after_ranked = len(_revised)
    _target = float(_targets.get(_row.molecule_id, _targets.get(_mid, np.nan)))
    if len(_revised) < 25 and np.isfinite(_target) and _target > 0 and _dm_paths:
        if _dm_mass is None:
            _dd = Path(_dm_paths[0]).parent
            _dm_mass = np.load(_dd / 'dreams_np_mass.npy').astype(np.float64)
            _dm_smiles = np.load(_dd / 'dreams_np_smiles.npy', allow_pickle=True)
            assert len(_dm_mass) == len(_dm_smiles)
            _dm_order = np.argsort(_dm_mass, kind='stable')
            _dm_sorted = _dm_mass[_dm_order]
        _lo, _hi = np.searchsorted(_dm_sorted, [_target * (1 - 10e-6), _target * (1 + 10e-6)])
        _ix = _dm_order[_lo:_hi]
        _ix = _ix[np.argsort(np.abs(_dm_mass[_ix] - _target), kind='stable')]
        _mass_candidates = [s.decode() if isinstance(s, bytes) else str(s) for s in _dm_smiles[_ix]]
        _revised = safe_extend(_revised, _mass_candidates)
        _dreams_used += len(_revised) - _after_ranked
    assert _revised, f'No valid candidates for {_mid}'
    assert len(_revised) <= 25 and len(first_ranks(_revised)) == len(_revised)
    assert_monotone(_original, _revised)
    _new_ranks = first_ranks(_revised)
    _audit_rows.append(dict(molecule_id=_mid, original_count=len(_original),
        original_unique=len(_ranks), invalid_count=sum(metric_identity(s) is None for s in _original),
        duplicate_count=sum(metric_identity(s) is not None for s in _original) - len(_ranks),
        revised_count=len(_revised), ranked_added=_after_ranked - len(_clean),
        dreams_added=len(_revised) - _after_ranked,
        promoted_original=sum(_new_ranks[k] < v for k, v in _ranks.items()),
        changed=_original != _revised))
    _out.append((_row.molecule_id, ';'.join(_revised)))
_final = pd.DataFrame(_out, columns=['molecule_id', 'smiles'])
_expected = pd.read_csv(os.path.join(COMP, 'sample_submission.csv')).molecule_id
assert _final.molecule_id.is_unique and set(_final.molecule_id) == set(_expected)
assert _final.smiles.notna().all() and _final.smiles.str.split(';').map(len).between(1, 25).all()
if IS_RERUN:
    assert len(_active) == len(_final), 'A hidden molecule was left unprocessed'
_final.to_csv('submission.csv', index=False)
_audit_df = pd.DataFrame(_audit_rows)
_audit_df.to_csv('candidate_audit.csv', index=False)
_summary = dict(experiment='V1.1 metric-safe tail', rdkit=rdkit.__version__,
    hidden_rerun=IS_RERUN, active_molecules=len(_active), output_rows=len(_final),
    changed_molecules=int(_audit_df.changed.sum()),
    invalid_removed=int(_audit_df.invalid_count.sum()),
    equivalent_duplicates_removed=int(_audit_df.duplicate_count.sum()),
    existing_candidates_promoted=int(_audit_df.promoted_original.sum()),
    ranked_candidates_added=int(_audit_df.ranked_added.sum()),
    dreams_candidates_added=int(_audit_df.dreams_added.sum()),
    monotone_rank_assertions_passed=True,
    metric_source_sha256=METRIC_SOURCE_SHA256,
    baseline_sha256=hashlib.sha256(Path('baseline_submission.csv').read_bytes()).hexdigest(),
    submission_sha256=hashlib.sha256(Path('submission.csv').read_bytes()).hexdigest(),
    stages=_stage_meta, runtime_seconds=round(time.time()-T0, 1),
    validation_note='Visible test is a smoke check only and overlaps training; no generalization score claimed.')
Path('candidate_audit.json').write_text(json.dumps(_summary, indent=2, default=str))
print(json.dumps({k:v for k,v in _summary.items() if k != 'stages'}, indent=2))
display(_audit_df)


In [ ]:
'''Independent experimental overlay
One vote per official metric identity per engine; compact unique ranks before the 40-candidate fusion shortlist. Preserve the existing PubChem gate and same-formula GLACIER logic. This tests duplicate-vote bias, not a coefficient search. It can change top-1 and reduce the score.'''

def metric_fusion(first, second, identity, alpha=0.6, offset=3.0, limit=40):
    scores, representatives = {}, {}
    collapsed = 0
    for candidates, weight in ((first, 1.0), (second, alpha)):
        seen = set()
        for candidate in candidates:
            key = identity(candidate)
            if key is None:
                continue
            if key in seen:
                collapsed += 1
                continue
            seen.add(key)
            scores[key] = scores.get(key, 0.0) + weight / (offset + len(seen))
            representatives.setdefault(key, candidate)
    ordered = sorted(scores, key=lambda key: -scores[key])[:limit]
    return [representatives[key] for key in ordered], [scores[key] for key in ordered], collapsed

EXPERIMENT = 'metric_identity_fusion'
_reference = pd.read_csv('submission.csv')
_reference.to_csv('submission_reference.csv', index=False)
_rows, _audit = [], []
for _row in _reference.itertuples(index=False):
    _mid = str(_row.molecule_id)
    _original = str(_row.smiles).split(';')
    _revised, _collapsed = list(_original), 0
    _engine = ENG.get(_mid)
    if _mid in PREFUSION_LISTS and _engine and _engine.get('smiles'):
        _first = [s for s in PREFUSION_LISTS[_mid] if s and s != 'CCO']
        _candidates, _scores, _collapsed = metric_fusion(_first, _engine['smiles'], metric_identity,
                                                       alpha=ALPHA, offset=KRR)
        _keys = [chem.score_key(s) or s for s in _candidates]
        _ice = ICE_SCORES.get(_mid, {}) if isinstance(ICE_SCORES, dict) else {}
        _gl = gl_of(_mid)
        if _ice and _gl:
            _order = gl_fuse.rerank_multi(_candidates, _keys, _scores,
                [_form2(s) for s in _candidates], [{}, _gl], [ICE_LAM, GL_LAM], top_n=len(_candidates))
            _candidates = [_candidates[i] for i in _order]
        if _candidates:
            _revised = safe_extend(_candidates[:25], _candidates[25:] + _original)
    _rows.append((_row.molecule_id, ';'.join(_revised)))
    _audit.append(dict(molecule_id=_mid, collapsed_source_votes=_collapsed,
        changed=_revised != _original, changed_top1=_revised[:1] != _original[:1]))

_result = pd.DataFrame(_rows, columns=['molecule_id', 'smiles'])
assert list(_result.molecule_id) == list(_reference.molecule_id)
assert _result.molecule_id.is_unique and _result.smiles.notna().all()
assert _result.smiles.map(lambda value: 1 <= len(value.split(';')) <= 25).all()
_result.to_csv('submission.csv', index=False)
pd.DataFrame(_audit).to_csv('experiment_audit.csv', index=False)
_changed = sum(row['changed'] for row in _audit)
_report = dict(experiment=EXPERIMENT, changed_molecules=_changed,
               changed_top1=sum(row['changed_top1'] for row in _audit),
               reference_sha256=sha256('submission_reference.csv'),
               submission_sha256=sha256('submission.csv'), measured_score=None)
json.dump(_report, open('experiment_audit.json', 'w'), indent=2)
print(_report)
if not _changed:
    print('NO-OP: predictions match the reference; do not spend a second submission on this file.')


In [ ]:
"""Observe the frozen source output and preserve same-run fusion inputs for later work."""


def _e004_postflight(scope):
    import hashlib
    import json
    import time
    from pathlib import Path
    import pandas as pd
    import rdkit

    pre = scope['E004_PREFLIGHT']
    ids = set(pre['input_ids'])
    processed = set(scope['te'].molecule_id.astype(str))
    full = bool(scope['IS_RERUN'])
    assert processed == (ids if full else set(sorted(ids)[:scope['SMOKE_N']]))
    frame = pd.read_csv('submission.csv', dtype=str, keep_default_na=False)
    assert list(frame.columns) == ['molecule_id', 'smiles']
    assert frame.molecule_id.is_unique and set(frame.molecule_id) == ids
    assert frame.smiles.map(lambda s: 1 <= len(s.split(';')) <= 25).all()
    base, eng, pc = scope['BASE'], scope['ENG'], scope['PC']
    assert set(map(str, base)) == processed
    assert processed <= set(map(str, eng))
    assert scope['n_err'] == 0
    if not full:
        assert all(v[0] for v in base.values())
        assert all(eng[mid]['smiles'] and pc.get(mid, {}).get('pc') for mid in processed)
    stages = {}
    for stage in ('ice', 'gl'):
        path = Path(f'{stage}_work/{stage}_out.json.meta.json')
        meta = json.loads(path.read_text()) if path.is_file() else {'status': 'missing'}
        # The frozen source owns full-run failure policy. Extra acceptance checks
        # only gate the visible smoke, where all expected stages are exercised.
        if not full:
            assert meta['status'] == 'ok' and meta['device'].startswith('cuda'), (stage, meta)
            assert meta['rdkit'] == '2025.03.6'
            assert not meta.get('errors') and meta.get('n_pred_fail', 0) == 0
            assert meta.get('n_feat_fail', 0) == 0
            if meta.get('n_mols_covered', 0):
                assert meta['n_mols_scored'] > 0 and meta['n_scored'] > 0
        stages[stage] = meta
    payload = {
        'prefusion': {mid: scope['PREFUSION_LISTS'][mid] for mid in sorted(processed)},
        'engine2': {mid: eng[mid] for mid in sorted(processed)},
        'pubchem': {mid: pc.get(mid, {}) for mid in sorted(processed)},
        'base_tail': {mid: scope['BASE_TAIL_CANDIDATES'][mid] for mid in sorted(processed)},
        'fusion_overflow': {mid: scope['FUSION_OVERFLOW'].get(mid, []) for mid in sorted(processed)},
    }
    Path('e004_candidate_inputs.json').write_text(json.dumps(payload, indent=2, default=str) + '\n')
    hashes = {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in (
        'submission.csv', 'baseline_submission.csv', 'submission_reference.csv',
        'candidate_audit.json', 'experiment_audit.json', 'e004_candidate_inputs.json')}
    report = dict(experiment='E004', source_version_id=356740494, source_version=80,
                  source_code_sha256='eee496a371fce179b4aeda1c5082a58212c724eaf1ca4ba0c943bbe8d7196ef2',
                  **pre, is_full_run=full, processed_ids=sorted(processed),
                  processed_molecules=len(processed), input_molecules=len(ids),
                  output_rows=len(frame), placeholder_fill_rows=len(ids - processed),
                  base_errors=int(scope['n_err']), base_nonempty=sum(bool(v[0]) for v in base.values()),
                  engine2_nonempty=sum(bool(eng[mid]['smiles']) for mid in processed),
                  pubchem_nonempty=sum(bool(pc.get(mid, {}).get('pc')) for mid in processed),
                  main_rdkit=rdkit.__version__, stages=stages, hashes=hashes,
                  elapsed_seconds=round(time.time() - scope['T0'], 2),
                  limitations=['Visible smoke is not independent validation.',
                               'Author 0.436 is not this reproduction score.',
                               'Whole v80 recipe differs from E003 in multiple factors.'])
    Path('e004_verification.json').write_text(json.dumps(report, indent=2) + '\n')
    print('E004_VERIFICATION', json.dumps(report))
    return report


E004_REPORT = _e004_postflight(globals())
del _e004_postflight


In [ ]:
# Fragment score for molecules without forward-model scores (since v46), copied verbatim from
# ahmedberatozer/casmi26-v7c-e006-fraggap version 1; its frag_rescore.py is attributed there to
# Huseyin Emre Aksoy (Apache-2.0).
# v7c: MetFrag-lite fragment evidence for molecules with NO forward-model coverage (no GLACIER, no finite ICEBERG)
FRAG_GAP = {}
try:
    FRAG_SRC = '"""Untrained MetFrag-lite re-scorer: how much MS/MS intensity a candidate\'s <=2-bond-cleavage fragments explain.\n\nIndependent of pv.py (same idea as pv.fragment_masses / explain_score_adduct, new code): enumerate the neutral masses\nof the connected pieces left after removing 1 or 2 bonds of the heavy-atom graph (implicit H stay on their atom),\nremember the fewest cleavages that give each mass, then match every peak against\n`neutral fragment + dh*H + charge carrier` for dh in `h_shifts` and the carriers the precursor adduct allows.\nA peak\'s credit is the weight of its cheapest explanation (1 for 0/1 cleavages, `w2` for 2, `w3` for 3),\nand a spectrum\'s score is the credited share of its (weighted) intensity. Defaults follow the forum ablation on a\n241-structure same-formula isomer panel (+0.065 MRR over the common baseline): w2=0.6, linear intensity,\ntol 0.005 Da, H shifts -2..+3, at most 2 cleavages. `BASELINE_CFG` is the common/pv.py-style setting.\n\n    score_molecule(spectra, cand_smiles, cfg=None, cache=None) -> list[float | None]\n\n`spectra`: list of dicts {mz: ndarray, it: ndarray (base peak = 1), adduct: str, prec: float, mode: +1/-1}.\nReturns one score per candidate (mean over the spectra that have peaks; higher = better), None if the SMILES does\nnot parse. Fragment masses are cached per SMILES in `cache` (a dict; pass one in to reuse it across molecules).\nThe enumeration is numba-compiled when numba is importable (pure-Python fallback otherwise, ~100x slower).\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n# monoisotopic masses (same values as pv.AMU); other elements come from RDKit\'s periodic table\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\nE = 0.00054857990\nPROTON = H - E\nH2O = 2 * H + AMU[\'O\']\nNH4 = AMU[\'N\'] + 4 * H\nFORMATE = AMU[\'C\'] + 2 * H + 2 * AMU[\'O\']\n\n# charge carriers a fragment ion may keep, per precursor adduct (added to the neutral fragment mass)\nCARRIERS = {\n    \'[M+H]+\': (PROTON,), \'[M-H2O+H]+\': (PROTON,), \'[M-2H2O+H]+\': (PROTON,),\n    \'[M+NH4]+\': (PROTON, NH4 - E), \'[M+Na]+\': (PROTON, AMU[\'Na\'] - E), \'[M+K]+\': (PROTON, AMU[\'K\'] - E),\n    \'[M-H]-\': (-PROTON,), \'[M-H2O-H]-\': (-PROTON,), \'[M+CH2O2-H]-\': (-PROTON, FORMATE - PROTON),\n    \'[M+Cl]-\': (-PROTON, AMU[\'Cl\'] + E),\n}\n\nDEFAULT_CFG = dict(\n    w1=1.0,                      # credit for a peak explained by the precursor or a 1-bond cleavage\n    w2=0.6,                      # credit for a peak only explained by a 2-bond cleavage\n    w3=0.3,                      # credit for 3-bond cleavages (only used when max_breaks=3)\n    intensity=\'linear\',          # \'linear\' | \'sqrt\' | a float power applied to `it`\n    mz_power=0.0,                # extra peak weight (mz / prec) ** mz_power (0 = off)\n    tol=0.005,                   # absolute m/z tolerance, Da\n    h_shifts=tuple(range(-2, 4)),  # hydrogen shifts dh applied to neutral fragments (-2..+3)\n    max_breaks=2,                # bond cleavages per fragment (1..3)\n    max_bonds_2=100,             # above this many bonds only 1-bond cleavages are enumerated (cost cap)\n    max_bonds_3=40,              # above this many bonds 3-bond cleavages are skipped\n)\n# the common baseline (pv.py-style): all cleavages equal, sqrt intensity, 0.01 Da, H shifts -2..+2\nBASELINE_CFG = dict(DEFAULT_CFG, w2=1.0, intensity=\'sqrt\', tol=0.01, h_shifts=tuple(range(-2, 3)))\n\n\ndef resolve_cfg(cfg=None):\n    """DEFAULT_CFG updated with `cfg` (unknown keys raise, so typos in variants do not pass silently)."""\n    out = dict(DEFAULT_CFG)\n    for k, v in (cfg or {}).items():\n        if k not in out:\n            raise KeyError(f\'frag_rescore: unknown cfg key {k!r}\')\n        out[k] = v\n    out[\'h_shifts\'] = tuple(int(d) for d in out[\'h_shifts\'])\n    if not 1 <= int(out[\'max_breaks\']) <= 3:\n        raise ValueError(\'frag_rescore: max_breaks must be 1, 2 or 3\')\n    return out\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# fragment enumeration (numba kernel with a pure-Python fallback: same source)\n# ---------------------------------------------------------------------------------------------------------------\ndef _find(parent, a):\n    while parent[a] != a:\n        parent[a] = parent[parent[a]]\n        a = parent[a]\n    return a\n\n\ndef _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, k):\n    """Union the kept bonds; if the graph splits, append each piece\'s mass with cleavage count k."""\n    for a in range(n):\n        parent[a] = a\n        acc[a] = 0.0\n    ncomp = n\n    for e in range(ea.shape[0]):\n        if removed[e]:\n            continue\n        ra = _find(parent, ea[e])\n        rb = _find(parent, eb[e])\n        if ra != rb:\n            parent[ra] = rb\n            ncomp -= 1\n    if ncomp < 2:\n        return cnt\n    for a in range(n):\n        r = _find(parent, a)\n        acc[r] += w[a]\n    for a in range(n):\n        if parent[a] == a:\n            out_m[cnt] = acc[a]\n            out_k[cnt] = k\n            cnt += 1\n    return cnt\n\n\ndef _enumerate(n, w, ea, eb, max_breaks, base_components):\n    """All piece masses after removing 1..max_breaks bonds (k = number removed); row 0.. are the intact pieces."""\n    nb = ea.shape[0]\n    n2 = nb * (nb - 1) // 2 if max_breaks >= 2 else 0\n    n3 = nb * (nb - 1) * (nb - 2) // 6 if max_breaks >= 3 else 0\n    cap = base_components + (base_components + 1) * nb + (base_components + 2) * n2 + (base_components + 3) * n3\n    out_m = np.empty(cap, np.float64)\n    out_k = np.empty(cap, np.int8)\n    parent = np.empty(n, np.int64)\n    acc = np.empty(n, np.float64)\n    removed = np.zeros(nb, np.bool_)\n    cnt = 0\n    for i in range(nb):\n        removed[i] = True\n        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 1)\n        if max_breaks >= 2:\n            for j in range(i + 1, nb):\n                removed[j] = True\n                cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 2)\n                if max_breaks >= 3:\n                    for l in range(j + 1, nb):\n                        removed[l] = True\n                        cnt = _emit_components(n, w, ea, eb, removed, parent, acc, out_m, out_k, cnt, 3)\n                        removed[l] = False\n                removed[j] = False\n        removed[i] = False\n    return out_m[:cnt], out_k[:cnt]\n\n\ntry:  # numba is on the Kaggle image and in requirements.txt; NUMBA_CACHE_DIR is set by nb/01_setup.py\n    from numba import njit\n    _find = njit(cache=True, nogil=True)(_find)\n    _emit_components = njit(cache=True, nogil=True)(_emit_components)\n    _enumerate = njit(cache=True, nogil=True)(_enumerate)\n    HAVE_NUMBA = True\nexcept Exception:  # pragma: no cover - fallback path\n    HAVE_NUMBA = False\n\n\ndef mol_graph(smi):\n    """(atom masses incl. implicit H, bond begin idx, bond end idx, n connected pieces) or None if unparsable."""\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    try:\n        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None\n    except Exception:\n        m = None\n    if m is None or m.GetNumAtoms() == 0:\n        return None\n    pt = Chem.GetPeriodicTable()\n    w = np.empty(m.GetNumAtoms(), np.float64)\n    for a in m.GetAtoms():\n        s = a.GetSymbol()\n        am = AMU.get(s)\n        if am is None:\n            am = pt.GetMostCommonIsotopeMass(s)\n        w[a.GetIdx()] = am + a.GetTotalNumHs() * H\n    ea = np.array([b.GetBeginAtomIdx() for b in m.GetBonds()], np.int64)\n    eb = np.array([b.GetEndAtomIdx() for b in m.GetBonds()], np.int64)\n    return w, ea, eb, len(Chem.GetMolFrags(m))\n\n\ndef fragment_levels(smi, max_breaks=2, max_bonds_2=100, max_bonds_3=40):\n    """Sorted neutral fragment masses grouped by the fewest cleavages that produce them: [k=0, k=1, ..., k=max].\n\n    k=0 holds the intact molecule (sum of its pieces, and each piece if the SMILES has several). None if the\n    SMILES does not parse.\n    """\n    g = mol_graph(smi)\n    if g is None:\n        return None\n    w, ea, eb, npieces = g\n    nb = len(ea)\n    mb = int(max_breaks)\n    if mb >= 3 and nb > max_bonds_3:\n        mb = 2\n    if mb >= 2 and nb > max_bonds_2:\n        mb = 1\n    levels = [np.array([w.sum()])]\n    if npieces > 1:  # salts / mixtures: each intact piece also counts as uncleaved\n        from rdkit import Chem\n        m = Chem.MolFromSmiles(smi)\n        levels = [np.unique(np.concatenate([[w.sum()], [w[list(ix)].sum() for ix in Chem.GetMolFrags(m)]]))]\n    if nb == 0:\n        return levels + [np.zeros(0) for _ in range(int(max_breaks))]\n    fm, fk = _enumerate(len(w), w, ea, eb, mb, npieces)\n    fm = np.concatenate([levels[0], fm])\n    fk = np.concatenate([np.zeros(len(levels[0]), np.int8), fk])\n    key = np.round(fm, 5)\n    order = np.lexsort((fk, key))                  # by mass, then fewest cleavages first\n    key, fm, fk = key[order], fm[order], fk[order]\n    first = np.ones(len(key), bool)\n    first[1:] = key[1:] != key[:-1]\n    fm, fk = fm[first], fk[first]\n    return [np.sort(fm[fk == k]) for k in range(int(max_breaks) + 1)]\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# scoring\n# ---------------------------------------------------------------------------------------------------------------\ndef _carriers(adduct, mode):\n    c = CARRIERS.get(adduct)\n    if c is not None:\n        return c\n    if isinstance(adduct, str) and adduct.endswith(\'-\'):\n        return (-PROTON,)\n    if isinstance(adduct, str) and adduct.endswith(\'+\'):\n        return (PROTON,)\n    return (PROTON,) if (mode or 1) > 0 else (-PROTON,)\n\n\ndef _prep_spectrum(s, c):\n    """Peak weights and the neutral masses each peak would correspond to, for every (carrier, dh) pair."""\n    mz = np.asarray(s.get(\'mz\', ()), np.float64).ravel()\n    it = np.asarray(s.get(\'it\', ()), np.float64).ravel()\n    n = min(len(mz), len(it))\n    mz, it = mz[:n], it[:n]\n    ok = np.isfinite(mz) & np.isfinite(it) & (it > 0)\n    mz, it = mz[ok], it[ok]\n    if len(mz) == 0:\n        return None\n    p = c[\'intensity\']\n    iw = it if p == \'linear\' else np.sqrt(it) if p == \'sqrt\' else it ** float(p)\n    if c[\'mz_power\']:\n        prec = float(s.get(\'prec\') or mz.max())\n        iw = iw * (mz / prec) ** float(c[\'mz_power\'])\n    tot = iw.sum()\n    if not tot > 0:\n        return None\n    shifts = np.array([car + dh * H for car in _carriers(s.get(\'adduct\'), s.get(\'mode\')) for dh in c[\'h_shifts\']])\n    q = (mz[None, :] - shifts[:, None]).ravel()  # neutral mass needed, per (shift, peak)\n    return iw / tot, q, len(shifts), len(mz)\n\n\ndef _score_spectrum(levels, weights, prep, tol):\n    iw, q, ns, npk = prep\n    best = np.zeros(npk)\n    for arr, wt in zip(levels, weights):\n        if len(arr) == 0 or wt <= 0:\n            continue\n        a = np.searchsorted(arr, q - tol, \'left\')\n        b = np.searchsorted(arr, q + tol, \'right\')\n        hit = (b > a).reshape(ns, npk).any(axis=0)\n        best = np.where(hit & (wt > best), wt, best)\n    return float(iw @ best)\n\n\ndef candidate_levels(smi, cfg=None, cache=None):\n    """Cached fragment_levels for one SMILES under `cfg` (only the enumeration keys matter)."""\n    return _levels_cached(smi, resolve_cfg(cfg), cache)\n\n\ndef _levels_cached(smi, c, cache):\n    key = (smi, int(c[\'max_breaks\']), int(c[\'max_bonds_2\']), int(c[\'max_bonds_3\']))\n    if cache is not None and key in cache:\n        return cache[key]\n    try:\n        lv = fragment_levels(smi, c[\'max_breaks\'], c[\'max_bonds_2\'], c[\'max_bonds_3\'])\n    except Exception:\n        lv = None\n    if cache is not None:\n        cache[key] = lv\n    return lv\n\n\ndef score_molecule(spectra, cand_smiles, cfg=None, cache=None):\n    """One score per candidate: mean explained-intensity share over the molecule\'s spectra (None: unparsable)."""\n    c = resolve_cfg(cfg)\n    cache = {} if cache is None else cache\n    weights = [1.0, float(c[\'w1\']), float(c[\'w2\']), float(c[\'w3\'])][: int(c[\'max_breaks\']) + 1]\n    preps = [p for p in (_prep_spectrum(s, c) for s in (spectra or [])) if p is not None]\n    tol = float(c[\'tol\'])\n    out = []\n    for smi in cand_smiles:\n        lv = _levels_cached(smi, c, cache)\n        if lv is None:\n            out.append(None)\n            continue\n        if not preps:\n            out.append(0.0)\n            continue\n        out.append(float(np.mean([_score_spectrum(lv, weights, p, tol) for p in preps])))\n    return out\n\n\ndef warmup():\n    """Compile the numba kernel once (a few seconds) so the first real molecule is not charged for it."""\n    fragment_levels(\'CCOC(=O)c1ccccc1O\')\n    return HAVE_NUMBA\n'
    open('/kaggle/working/frag_rescore.py', 'w').write(FRAG_SRC)
    if '/kaggle/working' not in sys.path:
        sys.path.insert(0, '/kaggle/working')
    import frag_rescore
    frag_rescore.warmup()
    _fcache = {}
    _by = {str(m): g for m, g in te.groupby('molecule_id', sort=False)}
    _t0 = time.time(); _nf = 0
    for _mid, _lst in PREFUSION_LISTS.items():
        _ice = ICE_SCORES.get(_mid, {}) if isinstance(ICE_SCORES, dict) else {}
        if gl_of(_mid) or any(v is not None for v in (_ice or {}).values()):
            continue
        _e = ENG.get(_mid) or {}
        _c = [s for s in dict.fromkeys(list(_lst) + list(_e.get('smiles', []))) if s and s != 'CCO']
        _g = _by.get(_mid)
        if _g is None or len(_c) < 2:
            continue
        _spec = [dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),
                      adduct=r.adduct, prec=float(r.precursor_mz), mode=1 if r.ionization_mode == 'positive' else -1)
                 for r in _g.itertuples()][:16]
        try:
            _sc = frag_rescore.score_molecule(_spec, _c, cache=_fcache)
            FRAG_GAP[_mid] = {s: float(v) for s, v in zip(_c, _sc) if v is not None}
            _nf += 1
        except Exception as _ex:
            print('frag failed', _mid, repr(_ex))
    print('v7c FRAG gap molecules', _nf, 'of', len(PREFUSION_LISTS), f'{time.time()-_t0:.0f}s', flush=True)
except Exception as _e:
    FRAG_GAP = {}
    print('v7c FRAG FAILED -> v7a behaviour:', repr(_e), flush=True)


In [ ]:
def _factorial_entry(scope):
    """Append-only E005–E007 overlay; original v80 functions remain authoritative.

    Embedded function-locally after the complete frozen E004 notebook. This module
    never installs dependencies, changes model scores, or recanonicalizes SMILES.
    """

    import hashlib
    import json
    import math
    from pathlib import Path


    def _finite_coverage(scores, candidates, keys):
        count = 0
        for smi, key in zip(candidates, keys):
            value = scores.get(smi)
            if value is None:
                value = scores.get(key)
            try:
                count += value is not None and math.isfinite(float(value))
            except (TypeError, ValueError):
                pass
        return count


    def rerank_row(scope, mid, original, *, window, ice_fallback, processed):
        """Rebuild final metric fusion from same-run prefusion and engine lists.

        ``original`` is the metric-safe reference BEFORE the frozen final overlay,
        not the E004 final output. The original identity, representative selection,
        same-formula slot rerank and tail fill are reused without modification.
        """
        if window not in (40, 80):
            raise ValueError("Only the prespecified 40/80 candidate windows are supported")
        original = list(original)
        audit = {
            "molecule_id": mid,
            "processed": bool(processed),
            "branch": "placeholder",
            "candidate_count": 0,
            "collapsed_source_votes": 0,
            "ice_finite_count": 0,
            "gl_finite_count": 0,
            "fallback_changed": False,
            "fallback_changed_top1": False,
            "n1_proxy": mid in scope.get("N1_GL_MIDS", set()),
            "joint_ice_proxy_gl": False,
        }
        if not processed:
            return original, audit
        audit["branch"] = "no_engine"
        engine = scope["ENG"].get(mid)
        if mid not in scope["PREFUSION_LISTS"] or not engine or not engine.get("smiles"):
            return original, audit
        first = [s for s in scope["PREFUSION_LISTS"][mid] if s and s != "CCO"]
        candidates, scores, collapsed = scope["metric_fusion"](
            first,
            engine["smiles"],
            scope["metric_identity"],
            alpha=scope["ALPHA"],
            offset=scope["KRR"],
            limit=window,
        )
        keys = [scope["chem"].score_key(s) or s for s in candidates]
        ice_scores = scope["ICE_SCORES"]
        ice = ice_scores.get(mid, {}) if isinstance(ice_scores, dict) else {}
        gl = scope["gl_of"](mid)
        audit.update(
            branch="unrescored",
            candidate_count=len(candidates),
            collapsed_source_votes=collapsed,
            ice_finite_count=_finite_coverage(ice, candidates, keys),
            gl_finite_count=_finite_coverage(gl, candidates, keys),
        )
        before = list(candidates)
        if ice and gl:
            # flexonafft v47: joint ICE + proxy GL only on N1 treatment rows.
            # ice_fallback=False still rebuilds the exact E004 GL-only control.
            order = scope["gl_fuse"].rerank_multi(
                candidates,
                keys,
                scores,
                [scope["_form2"](s) for s in candidates],
                [ice if ice_fallback and mid in scope.get("N1_GL_MIDS", set()) else {}, gl],
                [scope["ICE_LAM"], scope["GL_LAM"]],
                top_n=len(candidates),
            )
            candidates = [candidates[i] for i in order]
            audit["branch"] = "glacier"
            audit["joint_ice_proxy_gl"] = bool(ice_fallback and audit["n1_proxy"])
        elif (ice_fallback and not gl and scope.get("FRAG_GAP", {}).get(mid)
              and not any(v is not None for v in (ice or {}).values())):
            # v7c: no forward-model evidence at all -> MetFrag-lite gap re-rank (lambda 0.5), once
            order = scope["gl_fuse"].rerank_multi(
                candidates, keys, scores, [scope["_form2"](s) for s in candidates],
                [scope["FRAG_GAP"][mid]], [0.5], top_n=len(candidates))
            candidates = [candidates[i] for i in order]
            audit["branch"] = "frag_gap"
        elif ice_fallback and ice and not gl:
            # Public v1 / script 356785434: ICE is used only where gl_of(mid) is empty.
            # Keep its truthy-dict gate: all-None ICE is an observable harmless no-op.
            order = scope["ice_fuse"].rerank(
                candidates,
                keys,
                scores,
                [scope["_form2"](s) for s in candidates],
                ice,
                lam=scope["ICE_LAM"],
                top_n=len(candidates),
            )
            candidates = [candidates[i] for i in order]
            audit["branch"] = "ice_fallback"
        result = (
            scope["safe_extend"](candidates[:25], candidates[25:] + original)
            if candidates
            else original
        )
        if audit["branch"] == "ice_fallback":
            without_ice = (
                scope["safe_extend"](before[:25], before[25:] + original) if before else original
            )
            audit["fallback_changed"] = result != without_ice
            audit["fallback_changed_top1"] = result[:1] != without_ice[:1]
        return result, audit


    def run_overlay(scope, *, experiment, window, ice_fallback, provenance, workdir="."):
        """Save the E004 bytes, validate same-run control, then emit one treatment."""
        import pandas as pd

        root = Path(workdir)
        path = root / "submission.csv"
        control_bytes = path.read_bytes()
        control_sha = hashlib.sha256(control_bytes).hexdigest()
        base_report = json.loads((root / "e004_verification.json").read_text())
        assert base_report["experiment"] == "E004"
        assert base_report["hashes"]["submission.csv"] == control_sha
        # Copy bytes directly; never reserialize or overwrite original E004 audit files.
        (root / "submission_e004_control.csv").write_bytes(control_bytes)
        control = pd.read_csv(path, dtype=str, keep_default_na=False)
        reference = pd.read_csv(root / "submission_reference.csv", dtype=str, keep_default_na=False)
        assert list(reference.columns) == list(control.columns) == ["molecule_id", "smiles"]
        assert list(reference.molecule_id) == list(control.molecule_id)
        assert control.molecule_id.is_unique
        processed = set(base_report["processed_ids"])
        assert processed == set(scope["te"].molecule_id.astype(str))
        controls = dict(zip(control.molecule_id, control.smiles))
        output, audits = [], []
        for row in reference.itertuples(index=False):
            mid, original = str(row.molecule_id), str(row.smiles).split(";")
            active = mid in processed
            # The exact original pre-overlay tail is essential for paired equivalence.
            rebuilt, _ = rerank_row(
                scope, mid, original, window=40, ice_fallback=False, processed=active
            )
            assert ";".join(rebuilt) == controls[mid], f"E004 control reconstruction mismatch: {mid}"
            result, audit = rerank_row(
                scope, mid, original, window=window, ice_fallback=ice_fallback, processed=active
            )
            assert 1 <= len(result) <= 25
            if active:
                identities = [scope["metric_identity"](s) for s in result]
                assert None not in identities and len(identities) == len(set(identities)), mid
            else:
                assert ";".join(result) == controls[mid], f"Placeholder changed: {mid}"
            old = controls[mid].split(";")
            audit.update(changed=result != old, changed_top1=result[:1] != old[:1])
            audits.append(audit)
            output.append((mid, ";".join(result)))
        frame = pd.DataFrame(output, columns=["molecule_id", "smiles"])
        assert list(frame.molecule_id) == list(control.molecule_id)
        frame.to_csv(path, index=False)
        report = {
            "experiment": experiment,
            "window": window,
            "ice_fallback": bool(ice_fallback),
            "source_experiment": "E004",
            "base_report_file": "e004_verification.json",
            "control_file": "submission_e004_control.csv",
            "control_sha256": control_sha,
            "submission_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "processed_ids": sorted(processed),
            "output_rows": len(frame),
            "changed_rows": sum(a["changed"] for a in audits),
            "changed_top1": sum(a["changed_top1"] for a in audits),
            "fallback_rows": sum(a["branch"] == "ice_fallback" for a in audits),
            "fallback_finite_rows": sum(
                a["branch"] == "ice_fallback" and a["ice_finite_count"] > 0 for a in audits
            ),
            "fallback_changed_rows": sum(a["fallback_changed"] for a in audits),
            "fallback_changed_top1": sum(a["fallback_changed_top1"] for a in audits),
            "same_run_control_reconstructed": True,
            "n1_challenger": {
                "complement": bool(scope.get("N1_COMPLEMENT", True)),
                "proxy_covered_molecules": len(scope.get("N1_GL_MIDS", set())),
                "joint_rerank_rows": sum(a["joint_ice_proxy_gl"] for a in audits),
                "joint_finite_ice_rows": sum(a["joint_ice_proxy_gl"] and a["ice_finite_count"] > 0 for a in audits),
                "joint_finite_both_rows": sum(a["joint_ice_proxy_gl"] and a["ice_finite_count"] > 0
                                              and a["gl_finite_count"] > 0 for a in audits),
            },
            "provenance": provenance,
            "limitations": [
                "Visible smoke is not independent validation.",
                "An inactive fallback in visible smoke says nothing about hidden benefit.",
            ],
        }
        pd.DataFrame(audits).to_csv(root / "experiment_audit_factorial.csv", index=False)
        (root / "experiment_audit_factorial.json").write_text(
            json.dumps(dict(report, rows=audits), indent=2, allow_nan=False) + "\n"
        )
        (root / "experiment_verification.json").write_text(
            json.dumps(report, indent=2, allow_nan=False) + "\n"
        )
        print("FACTORIAL_VERIFICATION", json.dumps(report))
        print("N1_CHALLENGER_FINAL", json.dumps(report["n1_challenger"]), flush=True)
        return report

    return run_overlay(scope, **{'experiment': 'E006', 'window': 40, 'ice_fallback': True, 'provenance': {'base_notebook_sha256': '1f0f9b3026a93ddb5ad6fa6993c8ac0a4bfaddc54594eb1dd6dda7a20d8ef0d6', 'base_source_code_sha256': 'eee496a371fce179b4aeda1c5082a58212c724eaf1ca4ba0c943bbe8d7196ef2', 'helper_source_sha256': 'a027a69ddda2c2e14639e3ec5196d5f1eb2ab57cba10ef929723148f31fc8f34', 'source_script_version_id': 356740494, 'ice_policy_source_script_version_id': 356785434, 'ice_policy_source': 'samanyu1808/casmi26-iceberg-where-glacier-is-blind-lb-0-438/1'}})

FACTORIAL_REPORT = _factorial_entry(globals())
del _factorial_entry


In [ ]:
# Rank-2 placement of the second engine's leading structure (since v48), after
# antriksharya/casmi26-lb-0-443-v7c-engine-2-rank-2-insert (MODE 'rank2'): the first structure of the second-engine
# list moves to rank 2 unless it already holds rank 1; keys = E006 metric identity; errors keep E006's file unchanged.
def _rank2_overlay(scope):
    import hashlib, json, os
    from pathlib import Path
    import pandas as pd
    path = Path('submission.csv')
    control = path.read_bytes()
    report = dict(control_sha256=hashlib.sha256(control).hexdigest(), rows=0, inserted=0, status='fallback')
    try:
        identity = scope['metric_identity']
        engine = scope['ENG'] if isinstance(scope.get('ENG'), dict) else {}
        frame = pd.read_csv(path, dtype=str, keep_default_na=False)
        rows = []
        for mid, answer in frame.itertuples(index=False):
            report['rows'] += 1
            toks = [t for t in str(answer).split(';') if t]
            listed = [s for s in ((engine.get(str(mid)) or {}).get('smiles') or []) if s and s != 'CCO']
            top = listed[0] if listed else None
            key = identity(top) if top else None
            if toks and toks[0] != 'CCO' and key is not None:
                keys = [identity(t) for t in toks]
                if key != keys[0]:
                    rest = [t for t, k in zip(toks, keys) if k != key]
                    toks = (rest[:1] + [top] + rest[1:])[:25]
                    report['inserted'] += 1
            rows.append((mid, ';'.join(toks) if toks else answer))
        out = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
        assert out.molecule_id.tolist() == frame.molecule_id.tolist() and out.smiles.notna().all()
        assert out.smiles.map(lambda s: 1 <= len(s.split(';')) <= 25).all()
        for s in out.smiles:
            ids = [identity(x) for x in s.split(';')]
            assert s == 'CCO' or (None not in ids and len(ids) == len(set(ids)))
        Path('submission_e006_control.csv').write_bytes(control)
        out.to_csv('submission_stage.csv', index=False)
        os.replace('submission_stage.csv', path)
        report['status'] = 'ok'
    except Exception as exc:
        path.write_bytes(control)
        report.update(status='fallback', error=f'{type(exc).__name__}: {exc}'[:400], inserted=0)
    report['submission_sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()
    Path('rank2_overlay.json').write_text(json.dumps(report, indent=2))
    print('RANK2_OVERLAY', json.dumps(report), flush=True)
    return report


RANK2_REPORT = _rank2_overlay(globals())
del _rank2_overlay


In [ ]:
V43_DATASET_SOURCES = ['dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets', 'dmitriigluzdov/casmi26-pubchem-popularity-prior', 'dmitriigluzdov/casmi26-natural-product-panel-benchmark', 'dmitriigluzdov/casmi26-natural-product-knowledge-table', 'dmitriigluzdov/casmi26-fold-safe-fpnet']
V43_E006_SHA256 = 'ee55b55c7f5bdb08f1a8154396b18fdb0203fb7178b3a52429e0999e69ad7e53'
# Read the final E006 file; validate it without changing its bytes or candidate order.
def _v43_receipt(scope):
    import hashlib
    import json
    import os
    import time
    from pathlib import Path
    import pandas as pd
    import rdkit

    def digest(path):
        path = Path(path)
        h = hashlib.sha256()
        with path.open('rb') as source:
            for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
                h.update(block)
        return dict(path=str(path), bytes=path.stat().st_size, sha256=h.hexdigest())

    final_path = Path('submission.csv')
    final_before = digest(final_path)
    frame = pd.read_csv(final_path, dtype=str, keep_default_na=False)
    sample = pd.read_csv(Path(scope['COMP']) / 'sample_submission.csv', dtype=str)
    assert list(frame.columns) == ['molecule_id', 'smiles']
    assert frame.molecule_id.tolist() == sample.molecule_id.tolist()
    assert frame.molecule_id.is_unique and frame.smiles.notna().all()
    processed = set(scope['te'].molecule_id.astype(str))
    inputs = set(scope['E004_PREFLIGHT']['input_ids'])
    assert processed == (inputs if scope['IS_RERUN'] else set(sorted(inputs)[:scope['SMOKE_N']]))
    for mid, answer in frame.itertuples(index=False):
        guesses = answer.split(';')
        assert 1 <= len(guesses) <= 25, mid
        identities = [scope['metric_identity'](s) for s in guesses]
        assert None not in identities and len(set(identities)) == len(identities), mid
    assert scope['n_err'] == 0
    assert scope['FACTORIAL_REPORT']['same_run_control_reconstructed']
    assert scope['FACTORIAL_REPORT']['submission_sha256'] == scope['RANK2_REPORT']['control_sha256']
    assert scope['RANK2_REPORT']['submission_sha256'] == final_before['sha256']

    # Hash all files of the mounted public datasets plus the competition files read.
    # The inventory records content hashes, including model arrays and source modules.
    roots = {str(Path(scope['E004_PREFLIGHT']['runtime_input_path']).resolve())}
    input_root = Path('/kaggle/input')
    dataset_roots = set()
    for slug in V43_DATASET_SOURCES:
        owner, name = slug.split('/', 1)
        direct = input_root / name
        if direct.is_dir():
            dataset_roots.add(direct.resolve())
        else:
            dataset_roots.update(p.resolve() for p in (input_root / 'datasets' / owner).glob(name) if p.is_dir())
    mounted = sorted({p.resolve() for root in dataset_roots for p in root.rglob('*') if p.is_file()})
    comp_root = Path(scope['E004_PREFLIGHT']['runtime_input_path'])
    mounted = sorted(set(mounted) | {comp_root.joinpath(name).resolve() for name in
                                    ('train.parquet', 'test.parquet', 'sample_submission.csv')})
    def quick_digest(path, limit=32 << 20, edge=4 << 20):
        path = Path(path)
        size = path.stat().st_size
        if size <= limit:
            return dict(digest(path), full=True)
        h = hashlib.sha256()
        with path.open('rb') as source:
            h.update(source.read(edge))
            source.seek(size - edge)
            h.update(source.read(edge))
        return dict(path=str(path), bytes=size, sha256_head_tail=h.hexdigest(), full=False)

    input_hashes = [quick_digest(path) for path in mounted]   # large files: size + head/tail hash (full reads took > 2 h)
    assert len(dataset_roots) == len(V43_DATASET_SOURCES), (len(dataset_roots), len(V43_DATASET_SOURCES))
    Path('v43_mounted_input_hashes.json').write_text(json.dumps(input_hashes, indent=2) + '\n')
    stages = scope['E004_REPORT']['stages']
    result = dict(
        version=49, method='canqiang_E006_window40_ice_fallback_plus_v7c_fragment_gap_plus_e2_rank2_rel150_e2_bags_chebi_lmsd_pubchem_spectral_bonus_np_bonus_pubchemlite_lotus_refs_depth6_poplam015_fold0_n1a_syn050_pcdelta_extlib_n1challenger_jointicegl_fivefold_pcslot6',
        source_notebook_sha256=V43_E006_SHA256,
        determinism=scope.get('V43_DETERMINISM', {'applied': False}),
        rdkit=rdkit.__version__, output_rows=len(frame), input_molecules=len(inputs),
        processed_molecules=len(processed), processed_ids=sorted(processed),
        scoring_rerun=bool(scope['IS_RERUN']), smoke_n=int(scope['SMOKE_N']),
        placeholder_rows=len(inputs - processed), main_errors=int(scope['n_err']),
        main_lists=len(scope['BASE']), engine2_lists=len(scope['ENG']), pubchem_lists=len(scope['PC']),
        engine2_nonempty=sum(bool(scope['ENG'].get(mid, {}).get('smiles')) for mid in processed),
        pubchem_nonempty=sum(bool(scope['PC'].get(mid, {}).get('pc')) for mid in processed),
        merge=scope['stats'], stages=stages,
        final_overlay=scope['FACTORIAL_REPORT'], rank2_overlay=scope['RANK2_REPORT'],
        pubchem_fpnet_ensemble=dict(models='A+B+mean(fold0..fold4)', raw_rule='z_AB + 0.5 * mean(z_fold0..z_fold4)', measured=True, fold_paths=scope['FOLD_PATHS'], fold_sha256=scope['FOLD_CHECKPOINT_SHA256']),
        mounted_datasets=V43_DATASET_SOURCES, mounted_files=len(input_hashes),
        mounted_input_inventory=digest('v43_mounted_input_hashes.json'),
        train=digest(comp_root / 'train.parquet'), test=digest(comp_root / 'test.parquet'),
        processed_test=digest(Path(scope['COMP']) / 'test.parquet'),
        submission=final_before, source_manifest=digest(Path(scope['V4MOD']) / 'MANIFEST.json'),
        elapsed_seconds=round(time.time() - scope['T0'], 2),
        notebook_elapsed_seconds=round(time.monotonic() - scope['NOTEBOOK_STARTED'], 2),
    )
    assert digest(final_path)['sha256'] == final_before['sha256']
    Path('v43_execution_receipt.json').write_text(json.dumps(result, indent=2, default=str) + '\n')
    print('V43_EXECUTION_RECEIPT', json.dumps({k: result[k] for k in
          ('output_rows', 'processed_molecules', 'placeholder_rows', 'main_errors', 'elapsed_seconds')}))
    return result

V43_RECEIPT = _v43_receipt(globals())
del _v43_receipt


### One molecule's final candidate list

The table shows the first five structures in one molecule's final answer, and Figure 5 draws the first three. Source labels show which search lists contained a structure; one candidate can appear in several lists. These labels describe membership rather than each source's contribution to the final rank.

The molecule is unlabelled, so the ordering illustrates the method rather than measuring accuracy.


In [ ]:
try:
    # Figure 1 reads the final submission file and never reruns the ranking.
    example_ids = list(te.molecule_id.drop_duplicates())
    eligible_ids = [m for m in example_ids if BASE[m][2] < LIB_TAU and PC.get(m, {}).get('pc')]
    example_mid = eligible_ids[0] if eligible_ids else min(example_ids, key=lambda m: BASE[m][2])
    example_query = te[te.molecule_id.eq(example_mid)]
    main_smiles, main_keys, example_lib = BASE[example_mid]
    proposal = PC.get(example_mid, {})
    _ft = proposal.get('fz_top')
    if _ft is None and proposal.get('pc_fz'):
        _ft = proposal['pc_fz'][0]
    _bp = proposal.get('best_pool_fz')
    _delta = (_ft - _bp if _ft is not None and _bp is not None and np.isfinite(_bp) else 1e9)
    if not proposal.get('pc'):
        branch = 'no PubChem proposals'
    elif example_lib >= LIB_TAU:
        branch = 'strong library match: PubChem proposals not inserted'
    else:
        branch = 'PubChem proposals at ranks ' + ', '.join(map(str, globals().get('SLOTS_AGG', [2, 4, 6, 8, 10]) if _delta > REL_TH else globals().get('SLOTS_GENTLE', [4, 8, 12, 16, 20])))
        if (proposal.get('S') is not None and float(proposal['S']) > S_TAU and
            float(proposal.get('top_pop') or 0.0) >= POP_TAU) or _delta > 150.0:
            branch += ', leading proposal moved to rank 1'
    _v43_audits = json.loads(Path('experiment_audit_factorial.json').read_text())['rows']
    _v43_row = next(row for row in _v43_audits if row['molecule_id'] == str(example_mid))
    _final_names = {'glacier': 'GLACIER', 'ice_fallback': 'ICEBERG', 'frag_gap': 'fragment score',
                    'unrescored': 'none', 'no_engine': 'none', 'placeholder': 'none'}
    _stage = _final_names.get(_v43_row['branch'], _v43_row['branch'])
    if _v43_row['branch'] == 'ice_fallback' and not _v43_row.get('ice_finite_count'):
        _stage = 'none (no finite ICEBERG scores)'
    branch += '; formula-level scoring: ' + _stage
    delta_text = f' · Δ {_delta:.0f}' if _delta != 1e9 else ''
    display(Markdown(f'**{example_mid}** · {len(example_query)} spectra · '
        f'neutral mass {np.nanmedian(example_query.nm):.4f} Da · library match {example_lib:.3f} · '
        f'**{branch}**' + delta_text))
    _v43_final = pd.read_csv('submission.csv', dtype=str, keep_default_na=False).set_index('molecule_id')
    final_smiles = _v43_final.at[str(example_mid), 'smiles'].split(';')
    known = {name: {metric_identity(s) for s in candidates} for name, candidates in (
        ('main', BASE_TAIL_CANDIDATES[str(example_mid)]), ('PubChem', proposal.get('pc', [])),
        ('second engine', ENG.get(str(example_mid), {}).get('smiles', [])))}
    example_rows = []
    for rank, smiles in enumerate(final_smiles[:5], 1):
        identity = metric_identity(smiles)
        channels = [name for name, identities in known.items() if identity in identities]
        example_rows.append({'Rank': rank, 'Channel': ' + '.join(channels) or 'list filling',
                             'Connectivity key': identity[0], 'SMILES': smiles})
    display(Markdown('**Worked-example candidates.**'))
    display(pd.DataFrame(example_rows))
    drawn = [Chem.MolFromSmiles(s) for s in final_smiles[:3]]
    if all(m is not None for m in drawn):
        display(Draw.MolsToGridImage(drawn, molsPerRow=3, subImgSize=(360, 280), useSVG=True,
                                   legends=[f'Rank {i}' for i in range(1, len(drawn) + 1)]))
except Exception as exc:
    print('worked example unavailable:', type(exc).__name__, exc)


*Figure 5. The first three structures in the example molecule's final answer order. The accompanying table lists the first five and their source-list membership.*


## 6. Versions, sources and licences

What each change added: the public MRR@25 of every scored version (— where a version was not scored).

| Version | Change | Public MRR@25 |
|---|---|---:|
| 1 | Pipeline of the main notebook (its version 51 method) | 0.450 |
| 3 | Pipeline of main notebook version 52 (public MRR@25 0.451) with the spectral-library bonus of Section 2 | 0.451 |
| 5 | Natural-product bonus in the PubChem channel (Section 2) | 0.451 |
| 7 | PubChemLite candidates in the second engine (Section 5) | 0.448 |
| 8 | LOTUS literature prior in the PubChem channel (Sections 2 and 4) | 0.448 |
| 9 | Second-engine trees at depth 6 instead of 7, as in the public PubChemLite notebooks (Section 5) | 0.455 |
| 10 | PubChem-channel prior weight 0.15 instead of 0.25 (Section 2) | 0.455 |
| 12 | Fold-safe FPNet added to the PubChem channel (Section 3) | 0.455 |
| 13 | GLACIER proxies for molecules without an [M+H]+ spectrum (Section 5) | 0.456 |
| 14 | PubChem synonym counts in the PubChem-channel prior (Section 2) | 0.456 |
| 15 | PubChem supplements in the PubChem channel (Section 2); public external reference spectra (Section 4) | 0.456 |
| 16 | GLACIER proxies kept where ICEBERG scores exist; joint ICEBERG and GLACIER re-ordering of proxy-covered lists (Section 5) | **0.464** |
| 17 | Structures generated from 20 spectral analogues instead of 6, as [Kino Lutor proposed](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/750241) (Section 5) | 0.456 |
| 18 | Five fold-safe FPNet models instead of one in the PubChem channel (Section 3); simpler text | 0.456 |
| 19 | Back to 6 spectral analogues; five fold-safe FPNet models kept | 0.464 |
| 20 | A sixth fixed position (rank 24) for PubChem proposals (Section 5) | 0.464 |
| 26 | Rank-1 PubChem promotion judged on the leading proposal's own scores, as in [Saner Turhaner's notebook](https://www.kaggle.com/code/sanpier/candidate-aware-pubchem-promotion-0-465) (Section 5) | — |

| Dataset | Licence | Terms of the parts |
|---|---|---|
| [CASMI26 Public Models and Candidate Sets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets) | CC BY-NC-SA 4.0 | Dataset-level licence; each copied folder keeps its source licence and notices (`ATTRIBUTION.md`), and competition- and COCONUT-derived files remain non-commercial with attribution. |
| [PubChem Popularity Prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) | Other | PubChem data courtesy of the U.S. National Library of Medicine, reused under the NCBI data-use policy; contributed content keeps its source-specific terms; LOTUS 2026-04-13 membership and reference counts CC BY 4.0, NPAtlas CC BY-NC 4.0. |
| [Natural-Product Panel Benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) | Other | Identifiers, masses and scores derive from the competition training file and follow the competition data terms; no spectra are included. |
| [CASMI26 Fold-Safe FPNet](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-fold-safe-fpnet) | Other | The checkpoints derive from the competition training file and follows the competition data terms (non-commercial, with attribution); the training and evaluation scripts are Apache 2.0. |
| [Natural Product Knowledge Table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) | Other | LOTUS 2026-04-13 (CC BY 4.0) and NPAtlas (CC BY-NC 4.0: non-commercial use, cited wherever the `npatlas_*` columns are used); PubChem counts. |

| Resource | Contribution |
|---|---|
| [CASMI 2026](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra), [metric](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank) | Spectra, submission contract and scoring. |
| [canqiang E006](https://www.kaggle.com/code/canqiang/casmi26-e006-window-40-ice-fallback-true) | Complete inference base of the pipeline. |
| [Ahmed Berat Özer v4n](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4n-inference), [v4m](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4m-inference); [v4b models](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v4b-models), [v3 models](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v3-models), [FPNet full1](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-fpnet-full1), [v2 pool](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-v2-pool), [PubChem tier](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-pubchem-tier) | Main-engine family, its code and rankers, the full1 and FPNet A+B fingerprint weights, the candidate pool and the structure-only PubChem index. |
| [Ahmed Berat Özer v7c](https://www.kaggle.com/code/ahmedberatozer/casmi26-v7c-e006-fraggap) | Fragment score where neither forward model has scores; its `frag_rescore.py` is attributed to [Huseyin Emre Aksoy](https://www.kaggle.com/huseyinemreaksoy) (Apache-2.0). |
| [Prvsiyan](https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline), [megayak](https://www.kaggle.com/code/megayak/casmi26-two-rankers-one-engine), [seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-v4n-engine-fusion-union-lb-0-399) | Analogue propagation, two-ranker scoring and engine fusion. |
| [flexonafft, N1 challenger](https://www.kaggle.com/code/flexonafft/casmi26-closing-the-evidence-gap?scriptVersionId=360486019) | Proxy GLACIER scores kept for molecules with ICEBERG scores, and ICEBERG combined with proxy GLACIER in the final same-formula re-ordering (version 47). |
| [Saner Turhaner, candidate-aware PubChem promotion](https://www.kaggle.com/code/sanpier/candidate-aware-pubchem-promotion-0-465) | Rank-1 PubChem promotion decided on the evidence of the candidate that actually leads the list (version 26). |
| [Ranker features](https://www.kaggle.com/datasets/prvsiyan/casmi26-ranker-features), [FP models](https://www.kaggle.com/datasets/prvsiyan/casmi26-fp-models-v2), [COCONUT candidates](https://www.kaggle.com/datasets/prvsiyan/coconut-casmi26-candidates), [ChEBI/LIPID MAPS](https://www.kaggle.com/datasets/prvsiyan/chebi-lipidmaps-casmi26) (prvsiyan); [simulated ranker rows](https://www.kaggle.com/datasets/megayak/casmi26-simulated-ranker-rows) (megayak) | Features, fingerprint weights, candidate catalogues and fitting rows of the second engine. |
| [wangpenghua v21](https://www.kaggle.com/code/wangpenghua/casmi26-v21-v4i-pop), [v29](https://www.kaggle.com/code/wangpenghua/casmi26-v29-fusion-pop-glmh) | Documentation prior and protonated-ion restriction for GLACIER. |
| [Obstacle V1.1](https://www.kaggle.com/code/obstacledeveloper/casmi26-v1-1-metric-audit-and-safe-tail), [Bob V18](https://www.kaggle.com/code/bobthebot369/enveda-casmi-2026-v18-sovereign-zenith), [flexonafft](https://www.kaggle.com/code/flexonafft/casmi26-public-baseline-adaptation-experiments), [samanyu1808](https://www.kaggle.com/code/samanyu1808/casmi26-iceberg-where-glacier-is-blind-lb-0-438) | Inherited ensemble, proposal promotion, identity audit and candidate filling; metric-identity fusion and ICEBERG fallback adopted by E006. |
| [antriksharya](https://www.kaggle.com/code/antriksharya/casmi26-lb-0-443-v7c-engine-2-rank-2-insert), [seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-e006-pubchem-promo-rel-150-lb-0-446) | Rank-2 placement of the second engine's leading structure; promotion threshold Δ > 150. |
| [ghazarosbarseghyan91](https://www.kaggle.com/code/ghazarosbarseghyan91/enveda-v7c-e006-fraggap-evgen-v14seeds), building on [evgendvorkin, version 14](https://www.kaggle.com/code/evgendvorkin/enveda-casmi-2026) | Deeper trees (depth 7) for the second-engine rankers, used in versions 1–8. |
| [sergiomalv](https://www.kaggle.com/datasets/sergiomalv/casmi-chebi-lmsd-structures) | Standardised ChEBI and LIPID MAPS structures (CC BY 4.0). |
| [seyitkaangunes n1a](https://www.kaggle.com/code/seyitkaangunes/casmi26-e006-n1a-cc-by-chebi-lb-0-452) | GLACIER proxy spectra for molecules without an [M+H]+ spectrum (N1A), embedded with its self-checks. |
| [guhongbin v465](https://www.kaggle.com/code/guhongbin/casmi26-v465-v15-pop015) | PubChem-channel prior weight 0.15. |
| [MassBank](https://massbank.eu/), [MoNA](https://mona.fiehnlab.ucdavis.edu/), [GNPS](https://gnps.ucsd.edu/) | Public MS/MS reference spectra (CC BY 4.0 or CC0 1.0 per record), screened against the training file; in the knowledge table's `external_reference_spectra/`. |
| [guhongbin v447](https://www.kaggle.com/code/guhongbin/casmi26-v447-pubchemlite), [ahmedberatozer v14](https://www.kaggle.com/code/ahmedberatozer/casmi26-v14-pubchemlite) | PubChemLite candidates in the second engine of the E006 line (guhongbin); the block used here (ahmedberatozer). |
| [franciscoangulo](https://www.kaggle.com/datasets/franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints), [cosmicbe](https://www.kaggle.com/datasets/cosmicbe/casmi26-pubchemlite), [PubChemLite](https://doi.org/10.1186/s13321-021-00489-0) | PubChemLite CC0 candidate fingerprints (copied), the candidate set they derive from, and PubChemLite itself (Schymanski et al., 2021). |
| [ICEBERG](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-iceberg), [GLACIER](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-glacier) packages; [ms-pred](https://github.com/coleygroup/ms-pred) (Coley group), [ICEBERG paper](https://doi.org/10.1021/acs.analchem.3c04654) | Forward spectrum prediction: models by the ms-pred authors, packaged for Kaggle by Ahmed Berat Özer. |
| [hengck23 DreaMS resource](https://www.kaggle.com/datasets/hengck23/hengck23-dreams-enveda-casmi26), [DreaMS paper](https://www.nature.com/articles/s41587-025-02663-3) | Mass-based candidate filling; the DreaMS spectral representation by its paper's authors. |
| [Metric RDKit wheel](https://www.kaggle.com/datasets/metric/rdkit-2026-3-3-wheel), [simulator wheels](https://www.kaggle.com/datasets/eveliaveldrine/rdkit-wheels) | Molecular normalisation and the simulator environment (RDKit by its developers). |
| [PubChem](https://pubchem.ncbi.nlm.nih.gov/docs/downloads) (Kim et al., [NAR 2025](https://doi.org/10.1093/nar/gkae1059)), [LOTUS](https://doi.org/10.7554/eLife.70780), [NPAtlas](https://doi.org/10.1093/nar/gkab941), [COCONUT](https://coconut.naturalproducts.net/), [ChEBI](https://www.ebi.ac.uk/chebi/), [LIPID MAPS](https://www.lipidmaps.org/) | Primary chemical and natural-product sources. |
| [MetFrag](https://doi.org/10.1186/s13321-016-0115-9), [NPClassifier](https://doi.org/10.1021/acs.jnatprod.1c00399) | Methods behind the fragment score and the class annotations. |

Scores quoted in the titles of linked notebooks are those notebooks' own public scores.
